# A. Point-in-time portfolio engine

**Foundation for the portfolio results — not article content.** This section builds and locks the
reproducible out-of-sample DRO engine on the clean pipeline. The article figures and tables (added
afterwards, mirroring the reference notebook's structure) consume the audited outputs produced here.

- **panel:** `data/processed/nyse_big_caps_pit_daily.parquet` (built by `01_build_pit_big_small_caps`);
- **signal:** `data/signals/V_uni.parquet` (compounded reference $\rho$, published by `02` → `tests/02`).

The engine is built point-in-time under a strict accounting contract: a formation-and-holding ledger
(§A.2.1), a daily holding engine with delisting and cash (§A.2.2), rebalancing and transaction costs
(§A.2.3), and the audited outputs (§A.2.4). No result is exported until every block passes its
assertions. The B=1000 permutation tests are produced separately on the VPS and loaded later.

In [ ]:
# Setup — clean-base data, DRO engine, plotting charter (matches 03_R_Signal). No window engine here.
import numpy as np, pandas as pd, cvxpy as cp, math, time, os, pickle, hashlib, json
import matplotlib as mpl, matplotlib.pyplot as plt
import polars as pl, scipy.stats as sps
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

# --- paths (clean base, robust to CWD) ---
ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent
DATA  = ROOT / 'data'
PANEL = DATA / 'processed' / 'nyse_big_caps_pit_daily.parquet'   # PIT panel (notebook 01)
SIG   = DATA / 'signals'                                         # validated compounded signals (02)
CRSP  = DATA / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.parquet'

OUT   = ROOT / 'outputs' / 'results' / 'portfolio'; OUT.mkdir(parents=True, exist_ok=True)
# Article deliverables (images/ and tables/) are kept strictly separate from machine-readable
# results (audit/) and from expensive recomputable caches (cache/).
IMG      = OUT / 'images'; IMG.mkdir(parents=True, exist_ok=True)   # .pdf  -> \includegraphics
TAB      = OUT / 'tables'; TAB.mkdir(parents=True, exist_ok=True)   # .tex  -> \input
DATA_OUT = OUT / 'audit';  DATA_OUT.mkdir(parents=True, exist_ok=True)  # .parquet/.json audit trail
# Internal working files live under the project-wide cache root, one folder per domain
# (same pattern as cache/signal/), never under outputs/ which holds article deliverables only.
CACHE    = ROOT / 'cache' / 'portfolio'; CACHE.mkdir(parents=True, exist_ok=True)

W_EST = 36; SOLVER = cp.MOSEK; EPS_FLOOR = 1e-4; EPS_CAP = 2.0
assert 'MOSEK' in cp.installed_solvers(), 'MOSEK solver is required for the DRO engine.'
for p in (PANEL, SIG / 'V_uni.parquet'):
    assert p.exists(), f'missing input: {p}'

# --- academic plotting charter (identical to 03_R_Signal) ---
_NAVY, _RUST, _GREEN, _GREY = '#1f3b5c', '#a23b2e', '#2e6e4e', '#3c3c3c'
PALETTE = [_NAVY, _RUST, _GREEN, '#b5882b', '#5c4b8a']
mpl.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'], 'font.size': 11, 'axes.titlesize': 12,
    'axes.labelsize': 11, 'axes.edgecolor': '#333333', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': '#dddddd', 'grid.linewidth': 0.6, 'legend.frameon': False,
    'legend.fontsize': 10, 'xtick.direction': 'out', 'ytick.direction': 'out',
    'figure.facecolor': 'white', 'axes.facecolor': 'white'})
CRISES = [('Dot-com', '2000-03'), ('GFC', '2008-09'), ('Euro debt', '2011-08'),
          ('COVID-19', '2020-03'), ('Rate shock', '2022-06')]

# --- fixed figure numbering (P_NN is an identity, not a running counter) ---
def save_fig(fig, n, name):
    """Save as R_P_<NN>_<name>.pdf (naming of the reference notebook). `n` is the fixed figure id
    (int -> R_P_01, or str like '08b')."""
    label = f'{n:02d}' if isinstance(n, int) else str(n)
    stem = f'R_P_{label}_{name}'
    target = IMG / f'{stem}.pdf'
    temporary = target.with_suffix('.pdf.tmp')
    fig.savefig(temporary, format='pdf', bbox_inches='tight')
    temporary.replace(target)
    print(f'saved  images/{stem}.pdf')

def load_or_compute(path, compute_fn, kind='parquet'):
    """check -> load -> else compute -> save (atomic). `kind` in {'parquet','pickle'}."""
    path = Path(path)
    if path.exists():
        print(f'loaded cache  {path.relative_to(ROOT)}')
        return pd.read_parquet(path) if kind == 'parquet' else pickle.load(open(path, 'rb'))
    obj = compute_fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    if kind == 'parquet':
        obj.to_parquet(tmp, index=False); tmp.replace(path)
    else:
        pickle.dump(obj, open(tmp, 'wb')); tmp.replace(path)
    print(f'computed+saved {path.relative_to(ROOT)}')
    return obj

# --- DRO portfolio engine (Wasserstein ambiguity, MOSEK) — NO silent fallback ---
def w_dro(R, eps):
    """
    Compute the long-only, fully invested Wasserstein-DRO portfolio.

    For a linear loss and a 2-Wasserstein ambiguity ball of radius
    ``eps``, solve the equivalent second-order cone program

        minimize    -mean(R).T @ w + eps * ||w||_2
        subject to   sum(w) = 1
                     w >= 0.

    Parameters
    ----------
    R : array-like of shape (T, n)
        Finite historical return matrix with T observations and n assets.
    eps : float
        Non-negative 2-Wasserstein ambiguity radius.

    Returns
    -------
    weights : numpy.ndarray or None
        Normalized optimal portfolio weights, or None if the solver fails.
    solver_status : str
        CVXPY solver status.
    """
    R = np.asarray(R, dtype=float)

    if R.ndim != 2 or min(R.shape) == 0:
        raise ValueError("R must be a non-empty two-dimensional return matrix.")
    if not np.isfinite(R).all():
        raise ValueError("R contains non-finite values.")
    if not np.isfinite(eps) or eps < 0:
        raise ValueError("eps must be finite and non-negative.")

    mean_returns = R.mean(axis=0)
    n_assets = R.shape[1]

    w = cp.Variable(n_assets, nonneg=True)

    problem = cp.Problem(
        cp.Minimize(
            -mean_returns @ w
            + float(eps) * cp.norm(w, 2)
        ),
        [cp.sum(w) == 1],
    )
    problem.solve(solver=SOLVER)

    if (
        w.value is None
        or problem.status not in ("optimal", "optimal_inaccurate")
    ):
        return None, problem.status

    weights = np.clip(np.asarray(w.value, dtype=float), 0.0, None)
    total_weight = weights.sum()

    if not np.isfinite(weights).all() or total_weight <= 0:
        return None, "invalid_weights"

    weights /= total_weight
    return weights, problem.status

# --- raw daily returns pivot + monthly returns (estimation uses complete-history members) ---
raw = (pl.read_parquet(PANEL).with_columns(pl.col('DlyCalDt').cast(pl.Date))
       .sort(['PERMNO', 'DlyCalDt']).to_pandas())
raw['date'] = pd.to_datetime(raw['DlyCalDt'])
pivot_d = (raw.drop_duplicates(['date', 'PERMNO'])
           .pivot(index='date', columns='PERMNO', values='DlyRet').sort_index())
pivot_d.columns = pivot_d.columns.astype(int)
# monthly returns for ESTIMATION only (formation members have complete 60-month history -> no NaN here)
pivot_m = (1 + pivot_d).resample('ME').prod(min_count=15) - 1

# --- reference compounded signal (clean base) ---
rho = pd.read_parquet(SIG / 'V_uni.parquet').set_index('date')['rho'].sort_index()
rho.index = pd.to_datetime(rho.index) + pd.offsets.MonthEnd(0)

print(f'panel: {pivot_d.shape[0]:,} days x {pivot_d.shape[1]} PERMNO | rho {len(rho)} pts | output -> {OUT}')

## A.1 Transaction cost schedule

Time-varying per-side transaction cost, reflecting the historical decline of execution frictions
(Frazzini, Israel and Moskowitz, 2018): 50 bps before 2000 (higher-friction environment), 30 bps in
2000--2009 (post-decimalization transition), 20 bps from 2010 (modern low-friction era). Applied to
the two-way turnover: $TC_\tau = c(\tau)\cdot \sum_i |w_{i,\tau} - w_{i,\tau}^{-}|$. Conservative for a
large-cap universe. All performance metrics are reported net of these costs.

In [ ]:
# Time-varying per-side transaction cost (bps), applied to two-way turnover
def cost_bps(date):
    """Per-side cost in bps by market regime (Frazzini et al., 2018). Two-way: TC = cost/1e4 * turnover."""
    y = pd.Timestamp(date).year
    if y <= 1999:   return 50.0   # higher-friction (pre-decimalisation)
    elif y <= 2009: return 30.0   # post-decimalization transition
    else:           return 20.0   # modern low-friction era

for d in ['1997-06-30', '2005-03-31', '2018-09-30']:
    print(f'  {d} -> {cost_bps(d):.0f} bps per-side')

## A.2 Portfolio engine

### A.2.1 Formation and holding ledger

Each decision is recorded explicitly. For every formation month $M$ the universe is the **exact
100-name formation membership** (`is_formation_member == 1`), ordered by `formation_member_rank`. The
estimation window is the 36 monthly returns of those 100 members through the formation close of $M$;
by construction (strict 60-month history) it is complete. The holding month is $M+1$
(`formation_member_active_month`). No information from $M+1$ is used to build the ledger.

In [ ]:
# E1 — build the formation & holding ledger from the panel (point-in-time, no look-ahead)
_mem = (pl.scan_parquet(PANEL).filter(pl.col('is_formation_member') == 1)
        .select([
            pl.col('formation_member_month').alias('formation_month'),
            pl.col('formation_member_active_month').alias('holding_month'),
            'PERMNO',
            pl.col('formation_member_rank').alias('rank'),
            pl.col('DlyCalDt').alias('formation_date'),
            pl.col('formation_member_strict_hist60').alias('strict60'),
        ]).collect(engine='streaming').to_pandas())
for c in ['formation_month', 'holding_month', 'formation_date']:
    _mem[c] = pd.to_datetime(_mem[c])
_mem['PERMNO'] = _mem['PERMNO'].astype(int)

# --- input contracts (hard) ---
_g = _mem.groupby('formation_month')
assert _g['PERMNO'].size().eq(100).all(), 'formation universe is not exactly 100 names every month'
assert _g['PERMNO'].nunique().eq(100).all(), 'duplicate PERMNO within a formation month'
assert _g['rank'].apply(lambda r: sorted(r.tolist()) == list(range(1, 101))).all(), 'ranks not 1..100'
assert _g['formation_date'].nunique().eq(1).all(), 'more than one formation date per month'
assert _mem['strict60'].all(), 'a member fails the strict 60-month history contract'
assert _mem['holding_month'].eq(_mem['formation_month'] + pd.offsets.MonthBegin(1)).all(), 'holding != M+1'

# --- build the ledger: one entry per formation month ---
pm_index = pivot_m.index
ledger = []
for fm, grp in _mem.groupby('formation_month', sort=True):
    grp = grp.sort_values('rank')
    assets = grp['PERMNO'].tolist()
    fdate = pd.Timestamp(grp['formation_date'].iloc[0])
    hmonth = pd.Timestamp(grp['holding_month'].iloc[0])
    fm_end = fm + pd.offsets.MonthEnd(0)                      # month-end key into pivot_m
    start_end = (fm - pd.DateOffset(months=W_EST - 1)) + pd.offsets.MonthEnd(0)
    if start_end < pm_index.min():
        continue                                             # not enough monthly history in the panel
    est = pivot_m.loc[start_end:fm_end, assets]
    if est.shape[0] != W_EST:
        continue                                             # window not fully covered by the monthly grid
    assert est.shape == (W_EST, 100), f'{fm:%Y-%m}: estimation shape {est.shape}'
    assert est.notna().all().all(), f'{fm:%Y-%m}: estimation window has NaN (should be impossible)'
    ledger.append({
        'formation_month': fm, 'formation_date': fdate, 'holding_month': hmonth,
        'assets': assets, 'est': est.values,
    })

# --- ledger audit table (machine-readable) ---
ledger_audit = pd.DataFrame([{
    'formation_month': e['formation_month'], 'formation_date': e['formation_date'],
    'holding_month': e['holding_month'], 'n_assets': len(e['assets']),
    'est_rows': e['est'].shape[0], 'est_cols': e['est'].shape[1],
    'first_permno': e['assets'][0], 'last_permno': e['assets'][-1],
} for e in ledger])
assert ledger_audit['n_assets'].eq(100).all() and ledger_audit['est_rows'].eq(W_EST).all()
assert ledger_audit['holding_month'].is_monotonic_increasing
assert not ledger_audit['formation_month'].duplicated().any()

print(f'E1 ledger: {len(ledger)} formation months '
      f'| {ledger_audit["formation_month"].min():%Y-%m} -> {ledger_audit["formation_month"].max():%Y-%m}')
display(ledger_audit.head(3))
display(ledger_audit.tail(2))

### A.2.2 Daily holding engine

For each held member the holding-period return is the product of its **adjusted** daily returns
`DlyRet` through the holding month:
$$R^{\mathrm{hold}}_{i,M+1}=\prod_{d\le d_i^{*}}\bigl(1+\mathrm{DlyRet}_{i,d}\bigr)-1,$$
where $d_i^{*}$ is the delisting day (or month-end if the security survives). `DlyRet` already embeds
the delisting return ($1+\mathrm{DlyRet}=(1+\mathrm{DlyRetx})(1+\mathrm{DelRet})$), so it is applied
**exactly once** and `DelRet_event` is never added again. After a delisting the security leaves the
book; its terminal proceeds are carried as **zero-return cash** until the next rebalance (handled in
§A.2.3), not redistributed to survivors.

Two documented data anomalies — a single missing `DlyRet` on an active, non-delisting day — are
imputed to zero under a **closed allowlist** (value unchanged that day). Any other null on a held
security-day is a hard failure. The decision ledger contains 373 formation dates, while the realised
ledger contains 372 holding months: the December 2025 decision is retained as an out-of-sample
forecast, not misclassified as 100 security exits in January 2026. A surviving leg must cover the
complete market calendar; only a documented delisting may terminate earlier.

In [ ]:
# E2 — per-(formation_month, asset) holding return from daily data, with delisting and anomalies
ALLOWED_NULL_RETURNS = {(76614, '2015-06-09')}   # documented held-day exception in the common-stock panel

# Separate valid decisions from realised outcomes. The final December-2025 decision has no January-2026 return.
panel_last_month = raw['date'].max().to_period('M')
realised_ledger = [e for e in ledger if pd.Timestamp(e['holding_month']).to_period('M') <= panel_last_month]
forecast_ledger = [e for e in ledger if pd.Timestamp(e['holding_month']).to_period('M') > panel_last_month]
assert len(realised_ledger) == 372 and len(forecast_ledger) == 1, 'unexpected realised/forecast split'

# held (asset, holding-month) pairs from the REALISED ledger -> the only daily returns used by E2
held_pairs = set()
for e in realised_ledger:
    hp = pd.Timestamp(e['holding_month']).to_period('M')
    for permno in e['assets']:
        held_pairs.add((int(permno), hp))

# restrict the daily panel to held asset-days (the anomaly contract is scoped to what is actually used)
hold_daily = raw[['PERMNO', 'date', 'DlyRet', 'PrimaryExch', 'is_delisting_event']].copy()
hold_daily['PERMNO'] = hold_daily['PERMNO'].astype(int)
hold_daily['ym'] = hold_daily['date'].dt.to_period('M')
hold_daily = hold_daily[[(p, m) in held_pairs
                         for p, m in zip(hold_daily['PERMNO'], hold_daily['ym'])]].copy()

# anomaly contracts on HELD days: (1) every held-day null is documented, (2) exactly allowlisted,
# (3) none is a delisting, (4) no null remains after imputation.
_null_mask = hold_daily['DlyRet'].isna()
_null_pairs = set(zip(hold_daily.loc[_null_mask, 'PERMNO'],
                      hold_daily.loc[_null_mask, 'date'].dt.strftime('%Y-%m-%d')))
assert _null_pairs == ALLOWED_NULL_RETURNS, f'held-day null set != allowlist: {_null_pairs ^ ALLOWED_NULL_RETURNS}'
assert int(_null_mask.sum()) == len(ALLOWED_NULL_RETURNS), 'imputation count differs from allowlist'
assert not hold_daily.loc[_null_mask, 'is_delisting_event'].any(), 'an anomaly is flagged as delisting'
hold_daily['return_imputed_zero'] = _null_mask
hold_daily.loc[_null_mask, 'DlyRet'] = 0.0
assert hold_daily['DlyRet'].notna().all(), 'a null remains after documented imputation'

# Monthly compound by held leg. Post-formation exchange migrations remain in the panel.
hold_daily['ym_end'] = hold_daily['ym'].dt.to_timestamp('M')
_mfd = (hold_daily.groupby(['PERMNO', 'ym_end'])
        .agg(hold_factor=('DlyRet', lambda s: float((1.0 + s).prod())),
             n_days=('DlyRet', 'size'),
             last_observed_date=('date', 'max'),
             event_date=('date', lambda s: s[hold_daily.loc[s.index, 'is_delisting_event']].max()),
             delisted=('is_delisting_event', 'any'),
             migrated_outside_formation_set=('PrimaryExch', lambda s: bool((~s.isin(['N', 'A', 'Q'])).any())),
             n_imputed=('return_imputed_zero', 'sum'))
        .reset_index())
market_days = hold_daily.groupby('ym_end')['date'].nunique().rename('n_market_days')
_mfd = _mfd.join(market_days, on='ym_end')
_bad_survivors = _mfd[(~_mfd['delisted']) & (_mfd['n_days'] != _mfd['n_market_days'])]
_bad_delistings = _mfd[_mfd['delisted'] & (_mfd['last_observed_date'] != _mfd['event_date'])]
assert _bad_survivors.empty, f'incomplete non-delisted holding paths:\n{_bad_survivors}'
assert _bad_delistings.empty, f'rows remain after a delisting event:\n{_bad_delistings}'
_mfd_key = _mfd.set_index(['PERMNO', 'ym_end'])

# Assemble exactly 100 realised holding legs per formation month. Missing legs are a hard failure.
holding_rows = []
for e in realised_ledger:
    H = pd.Timestamp(e['holding_month']) + pd.offsets.MonthEnd(0)
    for i, permno in enumerate(e['assets']):
        key = (permno, H)
        assert key in _mfd_key.index, f'missing realised holding leg: {key}'
        r = _mfd_key.loc[key]
        holding_rows.append({
            'formation_month': e['formation_month'], 'holding_month': e['holding_month'],
            'PERMNO': permno, 'rank': i + 1, 'hold_factor': float(r['hold_factor']),
            'r_hold': float(r['hold_factor']) - 1.0, 'n_days': int(r['n_days']),
            'delisted': bool(r['delisted']), 'n_imputed': int(r['n_imputed']),
            'migrated_outside_formation_set': bool(r['migrated_outside_formation_set']),
        })

holding = pd.DataFrame(holding_rows)

# --- E2 contracts ---
assert holding.groupby('formation_month')['PERMNO'].size().eq(100).all(), 'not 100 holding legs/month'
assert holding['r_hold'].notna().all() and np.isfinite(holding['r_hold']).all(), 'non-finite holding return'
assert holding['n_imputed'].sum() == len(ALLOWED_NULL_RETURNS), 'imputation propagation count differs from allowlist'
assert holding.groupby('formation_month')['rank'].apply(lambda x: sorted(x) == list(range(1, 101))).all()
assert holding['hold_factor'].ge(0).all(), 'a holding factor is negative'
print(f'E2: {len(ledger)} decisions | {holding["formation_month"].nunique()} realised holding months '
      f'| forecast-only: {[pd.Timestamp(e["holding_month"]).strftime("%Y-%m") for e in forecast_ledger]}')
_dl = holding[holding['delisted']]
print(f'delisting legs: {len(_dl)} | imputed-zero legs: {int((holding["n_imputed"] > 0).sum())} '
      f'| exchange-migration legs: {int(holding["migrated_outside_formation_set"].sum())}')
display(holding[holding['delisted']].head(3))

### A.2.3 Rebalancing, cash and transaction costs

At each formation close, the static portfolio uses a fixed **operational proxy** for the
ambiguity-radius level, $\varepsilon_{0}=\sqrt{2\log(1/\beta)/W}$ with $\beta=0.10$. This is an
operating point calibrated from the estimation-window size — not a theoretical guarantee: it places
the radius in an interior régime where the weights retain return structure. The dynamic portfolio
multiplies that level by the current $\sqrt{\rho}$ divided by its expanding historical mean. Both
portfolios are solved on the same ordered $36\times100$ estimation matrix, with no solver fallback.

Pre-rebalance weights are the actual drifted weights from the preceding holding month. A delisted
security has zero risky weight after its event; its terminal proceeds enter a zero-return cash sleeve.
Cash is included in the wealth reconciliation but excluded from the traded-security turnover sum.
Redeploying cash is nevertheless captured by the purchases required to reach the new risky targets.
The initial formation starts from cash, hence its risky-asset turnover is one.

In [ ]:
# E3 — solve target weights, then propagate exact pre-rebalance risky and cash weights
BETA_EK = 0.10
EPS_EK = float(np.clip(math.sqrt(2.0 * math.log(1.0 / BETA_EK) / W_EST), EPS_FLOOR, EPS_CAP))

# Signal timing: rho at formation M controls the portfolio held during M+1.
signal_rows = []
for e in ledger:
    signal_date = pd.Timestamp(e['formation_month']) + pd.offsets.MonthEnd(0)
    assert signal_date in rho.index, f'missing rho at formation {signal_date:%Y-%m}'
    signal_rows.append({
        'formation_month': e['formation_month'], 'signal_date': signal_date,
        'rho': float(rho.loc[signal_date]), 'sqrt_rho': math.sqrt(float(rho.loc[signal_date])),
    })
signal_timing = pd.DataFrame(signal_rows).sort_values('formation_month').reset_index(drop=True)
signal_timing['sqrt_rho_expanding_mean'] = signal_timing['sqrt_rho'].expanding(min_periods=1).mean()
signal_timing['rho_modulation'] = signal_timing['sqrt_rho'] / signal_timing['sqrt_rho_expanding_mean']
assert np.isfinite(signal_timing['rho_modulation']).all() and signal_timing['rho_modulation'].gt(0).all()
signal_by_fm = signal_timing.set_index('formation_month')

def _file_signature(path):
    path = Path(path); stat = path.stat()
    payload = f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}'
    return hashlib.sha256(payload.encode()).hexdigest()

engine_contract = {
    'engine': 'portfolio-pit-v2.0.0-direct-w2-socp', 'panel_signature': _file_signature(PANEL),
    'signal_signature': _file_signature(SIG / 'V_uni.parquet'), 'solver': str(SOLVER),
    'W_EST': W_EST, 'beta_ek': BETA_EK, 'eps_floor': EPS_FLOOR, 'eps_cap': EPS_CAP,
    'turnover': 'sum_abs_risky_assets_cash_excluded', 'initial_pretrade_state': '100pct_cash',
    'net_return': 'gross_return_minus_cost_rate_times_turnover',
}
ENGINE_DIGEST = hashlib.sha256(json.dumps(engine_contract, sort_keys=True).encode()).hexdigest()
TARGET_CACHE = CACHE / f'target_weights_{ENGINE_DIGEST[:16]}.parquet'
TARGET_AUDIT_CACHE = CACHE / f'target_solver_audit_{ENGINE_DIGEST[:16]}.parquet'

def _atomic_df(frame, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_parquet(tmp, index=False); tmp.replace(path)

def _compute_target_weights():
    weight_rows, audit_rows = [], []
    for ordinal, e in enumerate(ledger, start=1):
        fm = pd.Timestamp(e['formation_month']); timing = signal_by_fm.loc[fm]
        specifications = {
            'static': EPS_EK,
            'dynamic': float(np.clip(EPS_EK * timing['rho_modulation'], EPS_FLOOR, EPS_CAP)),
        }
        for strategy, epsilon in specifications.items():
            weights, status = w_dro(e['est'], epsilon)
            assert weights is not None, f'{fm:%Y-%m} {strategy}: solver failed ({status})'
            assert len(weights) == 100 and np.isfinite(weights).all()
            assert (weights >= -1e-12).all() and abs(weights.sum() - 1.0) < 1e-10
            for rank, (permno, weight) in enumerate(zip(e['assets'], weights), start=1):
                weight_rows.append({
                    'formation_month': fm, 'formation_date': e['formation_date'],
                    'holding_month': e['holding_month'], 'strategy': strategy,
                    'PERMNO': int(permno), 'rank': rank, 'target_weight': float(weight),
                    'epsilon': float(epsilon), 'rho': float(timing['rho']),
                    'rho_modulation': float(timing['rho_modulation']), 'engine_digest': ENGINE_DIGEST,
                })
            audit_rows.append({
                'formation_month': fm, 'strategy': strategy, 'solver_status': status,
                'epsilon': float(epsilon), 'weight_sum': float(weights.sum()),
                'weight_min': float(weights.min()), 'weight_max': float(weights.max()),
                'weight_l2': float(np.linalg.norm(weights)), 'engine_digest': ENGINE_DIGEST,
            })
        if ordinal % 24 == 0:
            print(f'  target weights {ordinal}/{len(ledger)}')
    return pd.DataFrame(weight_rows), pd.DataFrame(audit_rows)

if TARGET_CACHE.exists() and TARGET_AUDIT_CACHE.exists():
    target_weights = pd.read_parquet(TARGET_CACHE)
    target_solver_audit = pd.read_parquet(TARGET_AUDIT_CACHE)
    cache_ok = (not target_weights.empty and not target_solver_audit.empty
                and target_weights['engine_digest'].eq(ENGINE_DIGEST).all()
                and target_solver_audit['engine_digest'].eq(ENGINE_DIGEST).all())
else:
    cache_ok = False
if not cache_ok:
    target_weights, target_solver_audit = _compute_target_weights()
    _atomic_df(target_weights, TARGET_CACHE); _atomic_df(target_solver_audit, TARGET_AUDIT_CACHE)
    print('E3 target weights computed and cached.')
else:
    print('E3 target weights loaded from signed cache.')

assert len(target_weights) == len(ledger) * 2 * 100
assert target_weights.groupby(['formation_month', 'strategy'])['target_weight'].sum().sub(1.0).abs().max() < 1e-10
assert target_solver_audit['solver_status'].isin(['optimal', 'optimal_inaccurate']).all()

# Exact monthly accounting. Costs do not alter normalised post-trade targets; they reduce net return.
holding_groups = {pd.Timestamp(fm): grp.set_index('PERMNO').sort_index()
                  for fm, grp in holding.groupby('formation_month', sort=True)}
decision_rows, weight_ledger_rows = [], []
for strategy in ['static', 'dynamic']:
    pre_risky, pre_cash = {}, 1.0
    for e in ledger:
        fm = pd.Timestamp(e['formation_month'])
        tw = (target_weights[(target_weights['formation_month'] == fm)
                             & (target_weights['strategy'] == strategy)]
              .sort_values('rank'))
        assert tw['PERMNO'].tolist() == list(map(int, e['assets']))
        target = dict(zip(tw['PERMNO'].astype(int), tw['target_weight'].astype(float)))
        risky_union = sorted(set(pre_risky) | set(target))
        turnover_components = {p: abs(target.get(p, 0.0) - pre_risky.get(p, 0.0)) for p in risky_union}
        turnover = float(sum(turnover_components.values()))
        rate = cost_bps(e['formation_date']) / 1e4
        transaction_cost = rate * turnover
        realised = fm in holding_groups

        if realised:
            h = holding_groups[fm]
            assert set(h.index.astype(int)) == set(target), f'{fm:%Y-%m}: target/holding mismatch'
            terminal_values = {p: target[p] * float(h.loc[p, 'hold_factor']) for p in target}
            gross_factor = float(sum(terminal_values.values()))
            assert gross_factor > 0.0, f'{fm:%Y-%m}: non-positive portfolio wealth'
            end_risky_values = {p: v for p, v in terminal_values.items() if not bool(h.loc[p, 'delisted']) and v > 0.0}
            end_cash_value = float(sum(v for p, v in terminal_values.items() if bool(h.loc[p, 'delisted'])))
            end_risky = {p: v / gross_factor for p, v in end_risky_values.items()}
            end_cash = end_cash_value / gross_factor
            gross_return = gross_factor - 1.0
            weighted_return = float(sum(target[p] * float(h.loc[p, 'r_hold']) for p in target))
            gross_error = gross_return - weighted_return
            assert abs(gross_error) < 1e-12
            assert abs(sum(end_risky.values()) + end_cash - 1.0) < 1e-10
            net_return = gross_return - transaction_cost
            assert net_return > -1.0
            n_delisted = int(h['delisted'].sum())
        else:
            gross_factor = gross_return = net_return = gross_error = np.nan
            end_risky, end_cash, n_delisted = {}, np.nan, 0

        decision_rows.append({
            'formation_month': fm, 'formation_date': e['formation_date'],
            'holding_month': e['holding_month'], 'return_date': pd.Timestamp(e['holding_month']) + pd.offsets.MonthEnd(0),
            'strategy': strategy, 'realised': realised, 'epsilon': float(tw['epsilon'].iloc[0]),
            'rho': float(tw['rho'].iloc[0]), 'rho_modulation': float(tw['rho_modulation'].iloc[0]),
            'pre_risky_sum': float(sum(pre_risky.values())), 'pre_cash_weight': float(pre_cash),
            'target_weight_sum': float(sum(target.values())), 'turnover': turnover,
            'cost_bps': cost_bps(e['formation_date']), 'transaction_cost': transaction_cost,
            'gross_factor': gross_factor, 'gross_return': gross_return, 'net_return': net_return,
            'end_risky_sum': float(sum(end_risky.values())) if realised else np.nan,
            'end_cash_weight': float(end_cash) if realised else np.nan,
            'n_delisted': n_delisted, 'gross_reconciliation_error': gross_error,
            'engine_digest': ENGINE_DIGEST,
        })
        for p in risky_union:
            weight_ledger_rows.append({
                'formation_month': fm, 'strategy': strategy, 'PERMNO': int(p), 'is_cash': False,
                'pre_weight': float(pre_risky.get(p, 0.0)), 'target_weight': float(target.get(p, 0.0)),
                'turnover_component': float(turnover_components[p]),
                'end_weight': float(end_risky.get(p, 0.0)) if realised else np.nan,
                'engine_digest': ENGINE_DIGEST,
            })
        weight_ledger_rows.append({
            'formation_month': fm, 'strategy': strategy, 'PERMNO': -1, 'is_cash': True,
            'pre_weight': float(pre_cash), 'target_weight': 0.0, 'turnover_component': 0.0,
            'end_weight': float(end_cash) if realised else np.nan, 'engine_digest': ENGINE_DIGEST,
        })
        if realised:
            pre_risky, pre_cash = end_risky, float(end_cash)

portfolio_decisions = pd.DataFrame(decision_rows).sort_values(['strategy', 'formation_month']).reset_index(drop=True)
portfolio_weight_ledger = pd.DataFrame(weight_ledger_rows).sort_values(['strategy', 'formation_month', 'is_cash', 'PERMNO']).reset_index(drop=True)
print(f'E3: {len(target_weights):,} target weights | {len(portfolio_decisions)} decisions | EPS_EK={EPS_EK:.6f}')
display(portfolio_decisions.groupby('strategy').agg(decisions=('formation_month', 'size'), realised=('realised', 'sum'), mean_turnover=('turnover', 'mean'), max_cash=('end_cash_weight', 'max')))


### A.2.4 Portfolio outputs and audit trail

Only the 372 realised holding months enter performance. Gross and net wealth are reconstructed from
monthly returns; the December 2025 decision remains in the decision ledger with no fabricated January
2026 outcome. Every exported object carries the engine digest. Atomic replacement is used for all
Parquet files, and a JSON manifest records the complete computational contract.

In [ ]:
# E4 — hard reconciliation, realised performance series and atomic exports
portfolio_monthly = portfolio_decisions[portfolio_decisions['realised']].copy()
assert len(portfolio_monthly) == 2 * 372
assert portfolio_monthly.groupby('strategy')['return_date'].nunique().eq(372).all()
assert portfolio_monthly[['gross_return', 'net_return', 'turnover', 'transaction_cost']].notna().all().all()
assert np.isfinite(portfolio_monthly[['gross_return', 'net_return', 'turnover', 'transaction_cost']]).all().all()
assert portfolio_monthly['transaction_cost'].ge(0).all()
assert (portfolio_monthly['net_return'] <= portfolio_monthly['gross_return'] + 1e-15).all()
assert portfolio_monthly['gross_reconciliation_error'].abs().max() < 1e-12
assert (portfolio_monthly['pre_risky_sum'] + portfolio_monthly['pre_cash_weight'] - 1.0).abs().max() < 1e-10
assert (portfolio_monthly['end_risky_sum'] + portfolio_monthly['end_cash_weight'] - 1.0).abs().max() < 1e-10

portfolio_monthly['gross_wealth'] = portfolio_monthly.groupby('strategy')['gross_return'].transform(lambda x: (1.0 + x).cumprod())
portfolio_monthly['net_wealth'] = portfolio_monthly.groupby('strategy')['net_return'].transform(lambda x: (1.0 + x).cumprod())
assert portfolio_monthly[['gross_wealth', 'net_wealth']].gt(0).all().all()

# Independent reconstruction of target/pre/end sums and risky-asset turnover from the long weight ledger.
weight_reconciliation = (portfolio_weight_ledger.groupby(['formation_month', 'strategy'])
    .agg(pre_sum=('pre_weight', 'sum'), target_sum=('target_weight', 'sum'),
         turnover_rebuilt=('turnover_component', 'sum'), end_sum=('end_weight', 'sum'))
    .reset_index())
decision_audit = portfolio_decisions.merge(weight_reconciliation, on=['formation_month', 'strategy'], how='left', validate='one_to_one')
assert (decision_audit['pre_sum'] - 1.0).abs().max() < 1e-10
assert (decision_audit['target_sum'] - 1.0).abs().max() < 1e-10
assert (decision_audit['turnover_rebuilt'] - decision_audit['turnover']).abs().max() < 1e-12
_realised_audit = decision_audit[decision_audit['realised']]
assert (_realised_audit['end_sum'] - 1.0).abs().max() < 1e-10
assert decision_audit.groupby('strategy')['realised'].sum().eq(372).all()
assert decision_audit.groupby('strategy').size().eq(373).all()

MONTHLY_OUT = DATA_OUT / 'portfolio_monthly.parquet'
DECISIONS_OUT = DATA_OUT / 'portfolio_decisions.parquet'
WEIGHTS_OUT = DATA_OUT / 'portfolio_weight_ledger.parquet'
AUDIT_OUT = DATA_OUT / 'portfolio_engine_audit.parquet'
MANIFEST_OUT = DATA_OUT / 'portfolio_engine_manifest.json'
for frame, path in [(portfolio_monthly, MONTHLY_OUT), (portfolio_decisions, DECISIONS_OUT),
                    (portfolio_weight_ledger, WEIGHTS_OUT), (decision_audit, AUDIT_OUT)]:
    _atomic_df(frame, path)
manifest = dict(engine_contract, engine_digest=ENGINE_DIGEST, decisions_per_strategy=373,
                realised_months_per_strategy=372, first_return='1995-01-31', last_return='2025-12-31',
                outputs={p.name: _file_signature(p) for p in [MONTHLY_OUT, DECISIONS_OUT, WEIGHTS_OUT, AUDIT_OUT]})
_manifest_tmp = MANIFEST_OUT.with_suffix('.json.tmp')
_manifest_tmp.write_text(json.dumps(manifest, indent=2, sort_keys=True)); _manifest_tmp.replace(MANIFEST_OUT)

summary = (portfolio_monthly.groupby('strategy')
    .agg(n=('gross_return', 'size'), date_min=('return_date', 'min'), date_max=('return_date', 'max'),
         gross_mean=('gross_return', 'mean'), net_mean=('net_return', 'mean'),
         mean_turnover=('turnover', 'mean'), total_cost=('transaction_cost', 'sum'),
         max_cash=('end_cash_weight', 'max'), delisting_legs=('n_delisted', 'sum'))
    .reset_index())
print('E4 locked: all accounting assertions passed.')
print('saved:', MONTHLY_OUT, DECISIONS_OUT, WEIGHTS_OUT, AUDIT_OUT, MANIFEST_OUT, sep='\n  ')
display(summary)


# 6.2 Portfolio results — dynamic Wasserstein ambiguity radius

All article results below consume only the locked E1--E4 outputs. Figure identifiers are fixed, tables are written atomically, and every expensive calculation is cached under an identity derived from the audited engine digest.

## 6.2.1 The ambiguity radius in action  [FIG P_01]

The static specification holds the operational radius at $\varepsilon_0$. The dynamic specification scales the same operating point by $\sqrt{\rho_M}$ relative to its expanding historical mean, using only information available at formation $M$. The figure reports the radius attached to each realised holding month $M+1$; shaded bands mark the pre-declared stress episodes used throughout the article.

In [ ]:
# FIG P_01 — audited static and dynamic ambiguity-radius paths
radius_path = (portfolio_decisions[portfolio_decisions['realised']]
    .pivot(index='return_date', columns='strategy', values='epsilon')
    .sort_index()[['static', 'dynamic']])
assert len(radius_path) == 372 and radius_path.notna().all().all()
assert radius_path['static'].nunique() == 1 and abs(radius_path['static'].iloc[0] - EPS_EK) < 1e-12
assert radius_path['dynamic'].between(EPS_FLOOR, EPS_CAP).all()

fig, ax = plt.subplots(figsize=(10.0, 4.4))
for _, peak in CRISES:
    centre = pd.Timestamp(peak)
    ax.axvspan(centre - pd.DateOffset(months=3), centre + pd.DateOffset(months=3),
               color=_GREY, alpha=0.09, lw=0)
ax.plot(radius_path.index, radius_path['static'], color=_NAVY, lw=1.5, label=r'Static $P^{(\mathrm{stat})}$: $\varepsilon_0$')
ax.plot(radius_path.index, radius_path['dynamic'], color=_RUST, lw=1.5,
        label=r'Dynamic $P^{(\rho)}$: $\varepsilon^{(\rho)}_{\tau}$')
ax.set_xlabel(r'Holding month $\tau+1$')
ax.set_ylabel(r'Ambiguity radius $\varepsilon_{\tau}$')
ax.set_title('Static and signal-scaled ambiguity radius')
ax.legend(loc='upper right')
ax.margins(x=0.01)
fig.tight_layout(); save_fig(fig, 1, 'radius_time'); plt.show()
radius_summary = radius_path.agg(['min', 'median', 'mean', 'max']).T
display(radius_summary)


## 6.2.2 From the radius to portfolio weights  [FIG P_02]

To expose the mechanism independently of realised returns, the DRO problem is re-solved on every formation matrix over a pre-declared logarithmic radius grid. The figure reports the cross-time mean of $\lVert w_M(\varepsilon)\rVert_2$, together with its 10th--90th percentile band. Equal weight provides the lower concentration benchmark $1/\sqrt{100}$. The complete sweep is checkpointed after every radius value and is reused only when its engine and grid identities match.

In [ ]:
# FIG P_02 — concentration response to epsilon, checkpointed by radius
from matplotlib.ticker import FixedLocator, FixedFormatter
EPS_SWEEP = np.unique(np.round(np.r_[EPS_FLOOR, np.geomspace(0.005, 1.0, 14)], 6))
sweep_contract = {'engine_digest': ENGINE_DIGEST, 'eps_grid': EPS_SWEEP.tolist(),
                  'statistic': 'cross_time_l2_mean_q10_median_q90', 'n_decisions': len(ledger)}
SWEEP_DIGEST = hashlib.sha256(json.dumps(sweep_contract, sort_keys=True).encode()).hexdigest()
SWEEP_CACHE = CACHE / f'concentration_sweep_{SWEEP_DIGEST[:16]}.parquet'
if SWEEP_CACHE.exists():
    concentration_sweep = pd.read_parquet(SWEEP_CACHE)
    if concentration_sweep.empty or not concentration_sweep['sweep_digest'].eq(SWEEP_DIGEST).all():
        concentration_sweep = pd.DataFrame()
else:
    concentration_sweep = pd.DataFrame()
completed_eps = set(np.round(concentration_sweep.get('epsilon', pd.Series(dtype=float)), 6))

for epsilon in EPS_SWEEP:
    if round(float(epsilon), 6) in completed_eps:
        continue
    concentration, statuses = [], []
    started = time.time()
    for e in ledger:
        weights, status = w_dro(e['est'], float(epsilon))
        assert weights is not None, f'epsilon={epsilon:g}, {e["formation_month"]:%Y-%m}: {status}'
        assert np.isfinite(weights).all() and abs(weights.sum() - 1.0) < 1e-10
        concentration.append(float(np.linalg.norm(weights)))
        statuses.append(status)
    row = pd.DataFrame([{
        'epsilon': float(epsilon), 'concentration_mean': float(np.mean(concentration)),
        'concentration_q10': float(np.quantile(concentration, 0.10)),
        'concentration_median': float(np.median(concentration)),
        'concentration_q90': float(np.quantile(concentration, 0.90)),
        'n_decisions': len(concentration), 'n_optimal': int(sum(s == 'optimal' for s in statuses)),
        'n_optimal_inaccurate': int(sum(s == 'optimal_inaccurate' for s in statuses)),
        'sweep_digest': SWEEP_DIGEST,
    }])
    concentration_sweep = (pd.concat([concentration_sweep, row], ignore_index=True)
                           .drop_duplicates('epsilon', keep='last').sort_values('epsilon'))
    _atomic_df(concentration_sweep, SWEEP_CACHE)
    elapsed = time.time() - started
    print(f'  epsilon={epsilon:g}: mean ||w||2={row.iloc[0]["concentration_mean"]:.4f} | cached | {elapsed:.0f}s')

assert len(concentration_sweep) == len(EPS_SWEEP)
assert concentration_sweep['n_decisions'].eq(len(ledger)).all()
assert (concentration_sweep['n_optimal'] + concentration_sweep['n_optimal_inaccurate']).eq(len(ledger)).all()
equal_weight_concentration = 1.0 / math.sqrt(100)
floor_concentration = float(concentration_sweep.loc[concentration_sweep['epsilon'].eq(EPS_FLOOR), 'concentration_mean'].iloc[0])
plot_sweep = concentration_sweep[concentration_sweep['epsilon'] >= 0.005].copy()
x = plot_sweep['epsilon'].to_numpy(float)
fig, ax = plt.subplots(figsize=(9.0, 4.5))
ax.plot(x, plot_sweep['concentration_mean'], 'o-', color=_NAVY, lw=1.8, ms=5.2,
        label=r'Mean $\Vert w_{\tau}(\varepsilon)\Vert_2$')
ax.axhline(equal_weight_concentration, color=_GREEN, ls='--', lw=1.2, label=r'Equal weight $1/\sqrt{100}$')
ax.set_xscale('log')
ticks = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
ax.xaxis.set_major_locator(FixedLocator(ticks))
ax.xaxis.set_major_formatter(FixedFormatter([f'{v:g}' for v in ticks]))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.set_xlabel(r'Ambiguity radius $\varepsilon$ (log scale)')
ax.set_ylabel(r'Weight concentration $\Vert w\Vert_2$')
ax.set_ylim(
    equal_weight_concentration * 0.75,
    plot_sweep['concentration_mean'].max() * 1.05,
)
ax.set_yticks(np.arange(0.1, 1.0, 0.1))
ax.set_title('The radius drives weights toward equal weight as it grows')
ax.legend(loc='upper right', fontsize=9.0)
fig.tight_layout(); save_fig(fig, 2, 'concentration_vs_eps'); plt.show()
display(concentration_sweep
        .loc[:, ['epsilon', 'concentration_q10', 'concentration_median',
                 'concentration_mean', 'concentration_q90']]
        .set_index('epsilon').round(4))


## 6.2.3 Static versus dynamic portfolios  [FIG P_03 + TABLE P_01]

The core comparison holds the operating level fixed and changes only its timing: $P^{(\mathrm{stat})}$ uses $\varepsilon_0$, whereas $P^{(\rho)}$ uses the point-in-time signal-scaled radius. Results are reported gross and net of the audited historical cost schedule. Statistical inference is paired by month: a Newey--West test evaluates the mean-return difference and a paired stationary bootstrap evaluates the Sharpe-ratio difference. The bootstrap draws are cached under the engine identity. Relative changes in the figure are oriented so that movement to the right always denotes improvement.

In [ ]:
# 6.2.3 — audited return series and performance metrics
gross_wide = portfolio_monthly.pivot(index='return_date', columns='strategy', values='gross_return').sort_index()
net_wide = portfolio_monthly.pivot(index='return_date', columns='strategy', values='net_return').sort_index()
turnover_wide = portfolio_monthly.pivot(index='return_date', columns='strategy', values='turnover').sort_index()
assert gross_wide.shape == net_wide.shape == turnover_wide.shape == (372, 2)
assert gross_wide.index.equals(net_wide.index) and gross_wide.notna().all().all() and net_wide.notna().all().all()
R_stat, R_dyn = gross_wide['static'], gross_wide['dynamic']
R_stat_net, R_dyn_net = net_wide['static'], net_wide['dynamic']
TO_stat, TO_dyn = turnover_wide['static'], turnover_wide['dynamic']

def portfolio_metrics(returns, turnover):
    returns = pd.Series(returns).dropna(); n = len(returns)
    annual_return = float(returns.mean() * 12.0)
    volatility = float(returns.std(ddof=1) * math.sqrt(12.0))
    sharpe = annual_return / volatility if volatility > 0 else np.nan
    downside = float(math.sqrt(np.mean(np.minimum(returns.to_numpy(float), 0.0) ** 2)) * math.sqrt(12.0))
    sortino = annual_return / downside if downside > 0 else np.nan
    wealth = (1.0 + returns).cumprod(); cagr = float(wealth.iloc[-1] ** (12.0 / n) - 1.0)
    drawdown = 1.0 - wealth / wealth.cummax(); max_drawdown = float(drawdown.max())
    dd_date = pd.Timestamp(drawdown.idxmax()).strftime('%Y-%m')
    calmar = cagr / max_drawdown if max_drawdown > 0 else np.nan
    q05 = returns.quantile(0.05); cvar95 = float(-returns[returns <= q05].mean())
    underwater = (drawdown > 1e-12).astype(int); run = max_run = 0
    for flag in underwater:
        run = run + 1 if flag else 0
        max_run = max(max_run, run)
    return {'annual_return': annual_return, 'cagr': cagr, 'volatility': volatility,
            'Sharpe': sharpe, 'Sortino': sortino, 'MaxDD': max_drawdown, 'Calmar': calmar,
            'CVaR95': cvar95, 'DD_duration': int(max_run), 'DD_date': dd_date,
            'turnover': float(pd.Series(turnover).iloc[1:].mean())}

metric_sets = {
    ('static', 'gross'): portfolio_metrics(R_stat, TO_stat),
    ('dynamic', 'gross'): portfolio_metrics(R_dyn, TO_dyn),
    ('static', 'net'): portfolio_metrics(R_stat_net, TO_stat),
    ('dynamic', 'net'): portfolio_metrics(R_dyn_net, TO_dyn),
}
M_stat, M_dyn = metric_sets[('static', 'gross')], metric_sets[('dynamic', 'gross')]
M_stat_net, M_dyn_net = metric_sets[('static', 'net')], metric_sets[('dynamic', 'net')]

def paired_hac_mean_test(dynamic, static, lags=12):
    diff = np.asarray(dynamic, float) - np.asarray(static, float); n = len(diff)
    centred = diff - diff.mean(); long_run_variance = float(centred @ centred / n)
    for lag in range(1, min(lags, n - 1) + 1):
        gamma = float(centred[lag:] @ centred[:-lag] / n)
        long_run_variance += 2.0 * (1.0 - lag / (lags + 1.0)) * gamma
    standard_error = math.sqrt(max(long_run_variance, 0.0) / n)
    statistic = float(diff.mean() / standard_error) if standard_error > 0 else np.nan
    pvalue = float(2.0 * (1.0 - sps.norm.cdf(abs(statistic)))) if np.isfinite(statistic) else np.nan
    return {'mean_difference': float(diff.mean()), 't_stat': statistic, 'p_value': pvalue, 'lags': lags}

HAC_LAGS = 12
hac_gross = paired_hac_mean_test(R_dyn, R_stat, HAC_LAGS)
hac_net = paired_hac_mean_test(R_dyn_net, R_stat_net, HAC_LAGS)

# Paired stationary bootstrap for the Sharpe-ratio difference.
BOOTSTRAP_DRAWS, BOOTSTRAP_RESTART, BOOTSTRAP_SEED = 10_000, 0.10, 20250302
bootstrap_contract = {'engine_digest': ENGINE_DIGEST, 'draws': BOOTSTRAP_DRAWS,
    'restart_probability': BOOTSTRAP_RESTART, 'seed': BOOTSTRAP_SEED,
    'statistic': 'paired_annualised_sharpe_difference'}
BOOTSTRAP_DIGEST = hashlib.sha256(json.dumps(bootstrap_contract, sort_keys=True).encode()).hexdigest()
BOOTSTRAP_CACHE = CACHE / f'pair_sharpe_bootstrap_{BOOTSTRAP_DIGEST[:16]}.parquet'
if BOOTSTRAP_CACHE.exists():
    bootstrap_draws = pd.read_parquet(BOOTSTRAP_CACHE)
    bootstrap_ok = (len(bootstrap_draws) == BOOTSTRAP_DRAWS
                    and bootstrap_draws['bootstrap_digest'].eq(BOOTSTRAP_DIGEST).all())
else:
    bootstrap_ok = False
if not bootstrap_ok:
    rng = np.random.default_rng(BOOTSTRAP_SEED); n = len(R_stat)
    arrays = {'static_gross': R_stat.to_numpy(float), 'dynamic_gross': R_dyn.to_numpy(float),
              'static_net': R_stat_net.to_numpy(float), 'dynamic_net': R_dyn_net.to_numpy(float)}
    def _annualised_sharpe(values):
        sd = np.std(values, ddof=1)
        return float(np.mean(values) / sd * math.sqrt(12.0)) if sd > 0 else np.nan
    rows = []
    for draw in range(BOOTSTRAP_DRAWS):
        index = np.empty(n, dtype=int); index[0] = rng.integers(n)
        for t in range(1, n):
            index[t] = rng.integers(n) if rng.random() < BOOTSTRAP_RESTART else (index[t - 1] + 1) % n
        rows.append({'draw': draw,
            'delta_sharpe_gross': _annualised_sharpe(arrays['dynamic_gross'][index]) - _annualised_sharpe(arrays['static_gross'][index]),
            'delta_sharpe_net': _annualised_sharpe(arrays['dynamic_net'][index]) - _annualised_sharpe(arrays['static_net'][index]),
            'bootstrap_digest': BOOTSTRAP_DIGEST})
    bootstrap_draws = pd.DataFrame(rows); _atomic_df(bootstrap_draws, BOOTSTRAP_CACHE)
    print('Paired stationary-bootstrap draws computed and cached.')
else:
    print('Paired stationary-bootstrap draws loaded from signed cache.')

def bootstrap_inference(column, observed):
    draws = bootstrap_draws[column].to_numpy(float)
    lower, upper = np.quantile(draws, [0.025, 0.975])
    left = (np.sum(draws <= 0.0) + 1.0) / (len(draws) + 1.0)
    right = (np.sum(draws >= 0.0) + 1.0) / (len(draws) + 1.0)
    return {'estimate': observed, 'ci_low': float(lower), 'ci_high': float(upper),
            'p_value': float(min(1.0, 2.0 * min(left, right)))}

delta_sharpe_gross = M_dyn['Sharpe'] - M_stat['Sharpe']
delta_sharpe_net = M_dyn_net['Sharpe'] - M_stat_net['Sharpe']
bootstrap_gross = bootstrap_inference('delta_sharpe_gross', delta_sharpe_gross)
bootstrap_net = bootstrap_inference('delta_sharpe_net', delta_sharpe_net)
metrics_machine = pd.DataFrame([{'strategy': strategy, 'basis': basis, **values}
                                for (strategy, basis), values in metric_sets.items()])
inference_machine = pd.DataFrame([
    {'basis': 'gross', 'delta_sharpe': delta_sharpe_gross, **{f'hac_{k}': v for k, v in hac_gross.items()},
     **{f'bootstrap_{k}': v for k, v in bootstrap_gross.items()}},
    {'basis': 'net', 'delta_sharpe': delta_sharpe_net, **{f'hac_{k}': v for k, v in hac_net.items()},
     **{f'bootstrap_{k}': v for k, v in bootstrap_net.items()}},
])
_atomic_df(metrics_machine, DATA_OUT / 'P_01_static_dynamic_metrics.parquet')
_atomic_df(inference_machine, DATA_OUT / 'P_01_static_dynamic_inference.parquet')
print(f'DeltaSharpe gross={delta_sharpe_gross:+.4f}, net={delta_sharpe_net:+.4f}')
print(f'HAC(12) gross p={hac_gross["p_value"]:.4f}, net p={hac_net["p_value"]:.4f}')
print(f'Bootstrap gross p={bootstrap_gross["p_value"]:.4f}, net p={bootstrap_net["p_value"]:.4f}')


In [ ]:
# FIG P_03 — relative performance change, net of costs, oriented so right always means improvement
figure_metrics = ['annual_return', 'Sharpe', 'MaxDD', 'volatility', 'CVaR95', 'turnover']
better_when_lower = {'MaxDD', 'volatility', 'CVaR95', 'turnover'}
metric_labels = {'annual_return': r'$\mu$', 'volatility': r'$\sigma$',
                 'CVaR95': r'CVaR$_{95}$', 'turnover': 'Turnover'}

S, D = metric_sets[('static', 'net')], metric_sets[('dynamic', 'net')]
relative = []
for metric in figure_metrics:
    change = 100.0 * (D[metric] - S[metric]) / abs(S[metric])
    relative.append(-change if metric in better_when_lower else change)
relative = np.asarray(relative)

fig, ax = plt.subplots(figsize=(7.6, 4.2))
y = np.arange(len(figure_metrics))[::-1]
colours = [_GREEN if value >= 0 else _RUST for value in relative]
ax.barh(y, relative, color=colours, alpha=0.85, height=0.6, zorder=3)
ax.axvline(0.0, color=_GREY, lw=1.0, zorder=2)
for yi, value in zip(y, relative):
    ax.text(value + (0.03 if value >= 0 else -0.03) * np.abs(relative).max(), yi,
            f'{value:+.2f}%', va='center', ha='left' if value >= 0 else 'right',
            fontsize=9, color=_GREY)
ax.set_yticks(y)
ax.set_yticklabels([metric_labels.get(metric, metric) for metric in figure_metrics])
pad = 0.25 * np.abs(relative).max()
ax.set_xlim(-np.abs(relative).max() - pad, np.abs(relative).max() + pad)
ax.set_xlabel('Relative change of dynamic vs static (%),  right = improvement')
ax.set_title(r'Timing effect $P^{(\rho)}$ vs $P^{(\mathrm{stat})}$, net of costs')
ax.grid(axis='y', alpha=0)
fig.tight_layout(); save_fig(fig, 3, 'relative_gain'); plt.show()

# TABLE P_01 — static vs dynamic, net of transaction costs
def _f(value, digits):
    return f'{value:.{digits}f}'

tex_lines = [r'\begin{tabular}{lcc}', r'\toprule',
             r' & $P^{(\mathrm{stat})}$ & $P^{(\rho)}$ \\', r'\midrule']
tex_lines.append(rf"Return $\mu$ (\%) & {_f(S['annual_return']*100, 2)} & {_f(D['annual_return']*100, 2)} \\")
tex_lines.append(rf"Volatility $\sigma$ (\%) & {_f(S['volatility']*100, 2)} & {_f(D['volatility']*100, 2)} \\")
tex_lines.append(rf"Sharpe & {_f(S['Sharpe'], 3)} & {_f(D['Sharpe'], 3)} \\")
tex_lines.append(rf"Max drawdown & {_f(S['MaxDD'], 3)} & {_f(D['MaxDD'], 3)} \\")
tex_lines.append(rf"\quad date & {S['DD_date']} & {D['DD_date']} \\")
tex_lines.append(rf"\quad duration (months) & {S['DD_duration']} & {D['DD_duration']} \\")
tex_lines.append(rf"CVaR$_{{95}}$ & {_f(S['CVaR95'], 3)} & {_f(D['CVaR95'], 3)} \\")
tex_lines.append(rf"Turnover (two-way) & {_f(S['turnover'], 4)} & {_f(D['turnover'], 4)} \\")
tex_lines.append(r'\midrule')
tex_lines.append(rf"$\Delta\mathrm{{SR}}$ & \multicolumn{{2}}{{c}}{{${delta_sharpe_net:+.4f}$}} \\")
tex_lines.append(rf"Paired HAC mean test & \multicolumn{{2}}{{c}}{{$t={hac_net['t_stat']:.2f},\,p={hac_net['p_value']:.3f}$}} \\")
tex_lines.append(rf"Stationary bootstrap $\Delta\mathrm{{SR}}$ & \multicolumn{{2}}{{c}}{{$[{bootstrap_net['ci_low']:+.4f},\,{bootstrap_net['ci_high']:+.4f}],\,p={bootstrap_net['p_value']:.3f}$}} \\")
tex_lines.extend([r'\bottomrule', r'\end{tabular}'])
pair_table_tex = '\n'.join(tex_lines) + '\n'

PAIR_TABLE = TAB / 'P_01_static_dynamic_metrics.tex'
_pair_tmp = PAIR_TABLE.with_suffix('.tex.tmp')
_pair_tmp.write_text(pair_table_tex); _pair_tmp.replace(PAIR_TABLE)

# --- clean console recap (net only; the LaTeX file is written, never printed) ---
_W = 58
print('=' * _W)
print('  Static vs dynamic — net of transaction costs')
print('=' * _W)
print(f'  {"metric":<24}{"static":>12}{"dynamic":>12}')
print('-' * _W)
for label, key, scale, digits in [
        ('Return (%)', 'annual_return', 100.0, 2), ('Volatility (%)', 'volatility', 100.0, 2),
        ('Sharpe', 'Sharpe', 1.0, 3), ('Max drawdown', 'MaxDD', 1.0, 3),
        ('CVaR 95', 'CVaR95', 1.0, 3), ('Turnover (two-way)', 'turnover', 1.0, 4)]:
    print(f'  {label:<24}{S[key]*scale:>12.{digits}f}{D[key]*scale:>12.{digits}f}')
print(f'  {"Drawdown trough":<24}{S["DD_date"]:>12}{D["DD_date"]:>12}')
print(f'  {"Drawdown length (m)":<24}{S["DD_duration"]:>12d}{D["DD_duration"]:>12d}')
print('-' * _W)
print(f'  ΔSharpe (net)          {delta_sharpe_net:+.4f}')
print(f'  HAC({HAC_LAGS}) mean test      t={hac_net["t_stat"]:+.2f}, p={hac_net["p_value"]:.3f}')
print(f'  Bootstrap ΔSharpe      [{bootstrap_net["ci_low"]:+.4f}, {bootstrap_net["ci_high"]:+.4f}], '
      f'p={bootstrap_net["p_value"]:.3f}')
print('=' * _W)
print(f'saved  tables/{PAIR_TABLE.name}')

## 6.2.4 Validity of the timing: placebo tests  [FIG P_04 + TABLE P_02]

The dynamic portfolio differs from the static one only through the chronology of $\sqrt{\rho}$. The
placebo therefore destroys that chronology and leaves everything else untouched: same estimation
windows, same universe, same holding factors, same costs, same accounting. Three designs are run,
each with $B$ draws:

- **simple** — full permutation of the modulation path (destroys all timing, preserves the marginal
  distribution of the modulation values);
- **block6** / **block12** — permutation of contiguous six- and twelve-month blocks, which preserves
  local persistence and only destroys the medium-term alignment with the market.

Each draw has a deterministic seed derived from `(base seed, method, draw)`, so the result does not
depend on the order or the number of resumptions. The real path is recomputed with the *same*
parameterised solver used for the draws, so the reference Sharpe and the null distribution share
identical numerics; the reported $p$-value is the Davison--Hinkley corrected upper tail
$(\#\{{\text{null}\ge\text{real}\}}+1)/(B+1)$. No solver failure is replaced by equal weight.

In [ ]:
# MANUAL EXECUTION GATE — external VPS timing experiment
# Keep True during the local pre-VPS run. Set False only after the matching
# signed standalone result has been copied into cache/portfolio.
STOP_BEFORE_TIMING_TESTS = False

if STOP_BEFORE_TIMING_TESTS:
    raise RuntimeError(
        'INTENTIONAL STOP BEFORE THE 1,000 TIMING DRAWS — run the independent '
        'standalone experiment on the VPS, import its signed artifacts, then '
        'set STOP_BEFORE_TIMING_TESTS = False and resume.'
    )


In [ ]:
# 6.2.4 — shared signed panel for all three timing-inference levels
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.portfolio_timing_tests import (
    inference_tables as _timing_inference_tables,
    load_or_compute as _timing_load_or_compute,
    observed_statistics as _timing_observed_statistics,
)

TIMING_DIGEST_SHORT = '88e42643b59e'
TIMING_MANIFEST = CACHE / f'timing_test_manifest_{TIMING_DIGEST_SHORT}.json'
TIMING_INPUTS = CACHE / f'timing_test_inputs_{TIMING_DIGEST_SHORT}.npz'
TIMING_CACHE = CACHE / f'timing_tests_B1000_{TIMING_DIGEST_SHORT}.parquet'
for _path in (TIMING_MANIFEST, TIMING_INPUTS):
    assert _path.exists(), f'missing signed timing-test input: {_path}'

timing_manifest, timing_arrays, timing_draws = _timing_load_or_compute(
    TIMING_MANIFEST, TIMING_INPUTS, TIMING_CACHE,
    threads=1, checkpoint_every=5, progress_every=5)
assert timing_manifest['experiment_digest'].startswith(TIMING_DIGEST_SHORT)
assert timing_manifest['target_cell'] == 'S_exdc'
assert timing_manifest['method_blocks'] == {'block12': 12, 'block6': 6, 'simple': 1}
assert timing_manifest['cells4'] == ['L_all', 'L_exdc', 'S_all', 'S_exdc']
assert len(timing_draws) == timing_manifest['draws'] == 1000
timing_placebo, timing_inference = _timing_inference_tables(
    timing_draws, timing_manifest, timing_arrays)
_timing_observed, _timing_real = _timing_observed_statistics(timing_manifest, timing_arrays)

_atomic_df(timing_placebo, DATA_OUT / 'P_02_placebo_inference.parquet')
_atomic_df(timing_inference, DATA_OUT / 'P_07_P_08_timing_inference.parquet')

# Compatibility objects consumed by the reference-composition Figure P_04 cell.
PLACEBO_B = timing_manifest['draws']
PLACEBO_METHODS = timing_manifest['method_blocks']
PLACEBO_DIGEST = timing_manifest['experiment_digest']
PLACEBO_CACHE = TIMING_CACHE
real_path = _timing_real['L']['dynamic']
_placebo_frames = []
for _method in ('simple', 'block6', 'block12'):
    _frame = timing_draws[[
        'draw', f'seed_{_method}', f'L_{_method}_sharpe_gross',
        f'L_{_method}_sharpe_net', f'L_{_method}_mean_turnover']].copy()
    _frame.columns = ['draw', 'seed', 'sharpe_gross', 'sharpe_net', 'mean_turnover']
    _frame.insert(0, 'method', _method)
    _frame['real_sharpe_gross'] = real_path['sharpe_gross']
    _frame['real_sharpe_net'] = real_path['sharpe_net']
    _frame['placebo_digest'] = PLACEBO_DIGEST
    _placebo_frames.append(_frame)
placebo_draws = pd.concat(_placebo_frames, ignore_index=True)
assert len(placebo_draws) == 3 * PLACEBO_B
assert not placebo_draws.duplicated(['method', 'draw']).any()

print(f'loaded and validated timing panel: {len(timing_draws):,} joint draws')
print(f'experiment digest: {PLACEBO_DIGEST}')
print('saved  audit/P_02_placebo_inference.parquet')
print('saved  audit/P_07_P_08_timing_inference.parquet')


In [ ]:
# FIG P_04 + TABLE P_02 — placebo tests: reads the cached draws only, no optimisation here
sh_real = float(real_path['sharpe_net'])
_null = {m: placebo_draws.loc[placebo_draws['method'].eq(m)].sort_values('draw')['sharpe_net'].to_numpy()
         for m in ('simple', 'block6', 'block12')}
sh_perm, sh_b6, sh_b12 = _null['simple'], _null['block6'], _null['block12']
assert len(sh_perm) == len(sh_b6) == len(sh_b12) == PLACEBO_B

# Davison-Hinkley corrected upper tail, identical to the reference notebook
pval = lambda dist: ((dist >= sh_real).sum() + 1) / (len(dist) + 1)
p_perm, p_b6, p_b12 = pval(sh_perm), pval(sh_b6), pval(sh_b12)

# two-panel figure (neutral scientific title)
def _legend_observed_first(ax):
    handles, labels = ax.get_legend_handles_labels()
    i_obs = next(i for i, label in enumerate(labels) if label.startswith('Observed'))
    order = [i_obs] + [i for i in range(len(labels)) if i != i_obs]
    ax.legend([handles[i] for i in order], [labels[i] for i in order],
              loc='upper left', fontsize=8, handlelength=1.6,
              labelspacing=0.35, borderpad=0.4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.hist(sh_perm, bins=18, color=_GREY, alpha=0.6, label=f'Simple permutation (B={len(sh_perm)})')
ax.axvline(sh_real, color=_RUST, lw=2.2, label=f'Observed (SR={sh_real:.3f})')
ax.axvline(sh_perm.mean(), color=_NAVY, lw=1.2, ls='--', label=f'Placebo mean ({sh_perm.mean():.3f})')
ax.set_xlabel('Sharpe ratio'); ax.set_ylabel('Frequency')
ax.set_title('Simple permutation'); _legend_observed_first(ax)
ax = axes[1]
ax.hist(sh_b6, bins=18, color=_NAVY, alpha=0.45, label='Block 6m')
ax.hist(sh_b12, bins=18, color=_GREEN, alpha=0.45, label='Block 12m')
ax.axvline(sh_real, color=_RUST, lw=2.2, label=f'Observed (SR={sh_real:.3f})')
ax.set_xlabel('Sharpe ratio'); ax.set_ylabel('Frequency')
ax.set_title('Block permutation'); _legend_observed_first(ax)
fig.suptitle('Timing placebo: observed temporal ordering vs permuted signal paths', y=1.0)
fig.tight_layout(); save_fig(fig, 4, 'placebo_timing'); plt.show()

# TABLE P_02 — same structure as the reference notebook
def fr(x): return f'{x:.3f}'
tex = (r"\begin{tabular}{lccc}""\n"r"\toprule""\n"
       r"Placebo & Preserves & Real percentile & $p$-value \\""\n"r"\midrule""\n"
       f"Simple permutation & marginals & {(sh_perm < sh_real).mean()*100:.1f} & {fr(p_perm)} \\\\\n"
       f"Block 6 months & marginals, short persistence & {(sh_b6 < sh_real).mean()*100:.1f} & {fr(p_b6)} \\\\\n"
       f"Block 12 months & marginals, long persistence & {(sh_b12 < sh_real).mean()*100:.1f} & {fr(p_b12)} \\\\\n"
       r"\bottomrule""\n"r"\end{tabular}""\n")
_p02 = TAB / 'P_02_placebo.tex'
_p02_tmp = _p02.with_suffix('.tex.tmp'); _p02_tmp.write_text(tex); _p02_tmp.replace(_p02)

# machine-readable summary of the placebo panel
placebo_panel = pd.DataFrame([
    {'placebo': 'simple', 'preserves': 'marginals', 'B': len(sh_perm),
     'real_percentile': float((sh_perm < sh_real).mean()*100), 'null_mean': float(sh_perm.mean()),
     'null_q95': float(np.quantile(sh_perm, 0.95)), 'p_value': float(p_perm)},
    {'placebo': 'block6', 'preserves': 'marginals, short persistence', 'B': len(sh_b6),
     'real_percentile': float((sh_b6 < sh_real).mean()*100), 'null_mean': float(sh_b6.mean()),
     'null_q95': float(np.quantile(sh_b6, 0.95)), 'p_value': float(p_b6)},
    {'placebo': 'block12', 'preserves': 'marginals, long persistence', 'B': len(sh_b12),
     'real_percentile': float((sh_b12 < sh_real).mean()*100), 'null_mean': float(sh_b12.mean()),
     'null_q95': float(np.quantile(sh_b12, 0.95)), 'p_value': float(p_b12)},
]).assign(real_sharpe_net=sh_real, engine_digest=ENGINE_DIGEST, placebo_digest=PLACEBO_DIGEST)
_atomic_df(placebo_panel, DATA_OUT / 'placebo_panel_summary.parquet')

_W = 62
print('=' * _W)
print('  Placebo — observed temporal ordering vs permuted signal paths (net)')
print('=' * _W)
print(f'  real Sharpe (net) = {sh_real:.4f}')
print('-' * _W)
print(f'  {"placebo":<10}{"B":>6}{"percentile":>12}{"null mean":>11}{"p-value":>10}')
for row in placebo_panel.itertuples():
    print(f'  {row.placebo:<10}{row.B:>6}{row.real_percentile:>11.1f}%{row.null_mean:>11.4f}{row.p_value:>10.4f}')
print('=' * _W)
print(f'saved  tables/{_p02.name}')
print(f'saved  audit/{(DATA_OUT / "placebo_panel_summary.parquet").name}')

## 6.2.5 Conditional attribution: when does timing pay?  [FIG P_05 + TABLE P_03]

The baseline attribution is conducted **net of transaction costs**, because the signal changes both portfolio weights and the trading required to implement them. Gross results are retained in the machine-readable audit as a friction-free control. Formation months are classified ex post into high- and low-dispersion regimes using the full-sample median of the point-in-time $\sqrt{\rho_M}$ signal; this split is descriptive and is never used to construct the portfolio.

Two complementary quantities are reported. Conditional $\Delta\mathrm{SR}$ describes risk-adjusted performance within each regime but is not additive. The annualised mean-return contribution is additive by construction: each regime's conditional return spread is weighted by its sample frequency, and the two contributions reconcile exactly to the full-sample mean spread. The five pre-declared crisis windows cover $\pm12$ months around their dated peaks. Crisis estimates are descriptive small-sample diagnostics, not separate causal or significance claims.

In [ ]:
# FIG P_05 + TABLE P_03 — conditional attribution, net primary and gross audit
signal_for_attribution = (portfolio_monthly.loc[portfolio_monthly['strategy'].eq('dynamic')]
                          .set_index('return_date')[['formation_month', 'rho', 'rho_modulation']]
                          .sort_index())
signal_for_attribution.index = pd.to_datetime(signal_for_attribution.index)
assert signal_for_attribution.index.equals(net_wide.index)
signal_for_attribution['sqrt_rho'] = np.sqrt(signal_for_attribution['rho'].astype(float))
rho_threshold = float(signal_for_attribution['sqrt_rho'].median())
CRISIS_HALF_WINDOW = 12
attribution_contract = {
    'attribution_version': 'conditional-attribution-v1.0.0', 'engine_digest': ENGINE_DIGEST,
    'primary_basis': 'net', 'gross_control': True, 'regime_variable': 'formation_sqrt_rho',
    'threshold_rule': 'full_sample_median_descriptive', 'crisis_half_window_months': CRISIS_HALF_WINDOW,
    'crises': CRISES,
}
ATTRIBUTION_DIGEST = hashlib.sha256(json.dumps(attribution_contract, sort_keys=True).encode()).hexdigest()
high_regime = signal_for_attribution['sqrt_rho'] >= rho_threshold
low_regime = ~high_regime
assert int(high_regime.sum() + low_regime.sum()) == len(signal_for_attribution) == 372

return_pairs = {
    'gross': (R_stat, R_dyn),
    'net': (R_stat_net, R_dyn_net),
}

def _conditional_sharpe(values):
    values = pd.Series(values).dropna()
    standard_deviation = values.std(ddof=1)
    return float(values.mean() / standard_deviation * math.sqrt(12.0)) if len(values) > 1 and standard_deviation > 0 else np.nan

def _attribution_record(name, mask, basis, scope):
    static, dynamic = return_pairs[basis]
    mask = pd.Series(mask, index=static.index).astype(bool)
    static_sub, dynamic_sub = static[mask], dynamic[mask]
    n = int(mask.sum())
    mean_spread_monthly = float((dynamic_sub - static_sub).mean())
    return {
        'scope': scope, 'name': name, 'basis': basis, 'n': n,
        'sharpe_static': _conditional_sharpe(static_sub),
        'sharpe_dynamic': _conditional_sharpe(dynamic_sub),
        'delta_sharpe': _conditional_sharpe(dynamic_sub) - _conditional_sharpe(static_sub),
        'mean_static_ann': float(static_sub.mean() * 12.0),
        'mean_dynamic_ann': float(dynamic_sub.mean() * 12.0),
        'delta_mean_ann': mean_spread_monthly * 12.0,
        'sample_weight': n / len(static),
        'mean_contribution_ann': mean_spread_monthly * 12.0 * n / len(static),
    }

regime_masks = {
    'High dispersion': high_regime,
    'Low dispersion': low_regime,
    'All months': pd.Series(True, index=net_wide.index),
}
regime_rows = [_attribution_record(name, mask, basis, 'regime')
               for basis in ['gross', 'net'] for name, mask in regime_masks.items()]
regime_attribution = pd.DataFrame(regime_rows)
for basis in ['gross', 'net']:
    subset = regime_attribution[(regime_attribution['basis'] == basis)
                                & regime_attribution['name'].isin(['High dispersion', 'Low dispersion'])]
    full_delta = float(regime_attribution[(regime_attribution['basis'] == basis)
                                          & (regime_attribution['name'] == 'All months')]['delta_mean_ann'].iloc[0])
    assert abs(subset['mean_contribution_ann'].sum() - full_delta) < 1e-15

# Pre-declared +/-12-month crisis windows, aligned to realised return months.
crisis_rows = []
for basis in ['gross', 'net']:
    for crisis_name, peak in CRISES:
        peak_date = pd.Timestamp(peak) + pd.offsets.MonthEnd(0)
        start = peak_date - pd.DateOffset(months=CRISIS_HALF_WINDOW)
        end = peak_date + pd.DateOffset(months=CRISIS_HALF_WINDOW)
        mask = pd.Series((net_wide.index >= start) & (net_wide.index <= end), index=net_wide.index)
        record = _attribution_record(crisis_name, mask, basis, 'crisis')
        record.update({'peak_date': peak_date, 'window_start': start, 'window_end': end})
        crisis_rows.append(record)
crisis_attribution = pd.DataFrame(crisis_rows)
assert crisis_attribution['n'].eq(2 * CRISIS_HALF_WINDOW + 1).all()
conditional_attribution = pd.concat([regime_attribution, crisis_attribution], ignore_index=True, sort=False)
conditional_attribution['rho_threshold'] = rho_threshold
conditional_attribution['engine_digest'] = ENGINE_DIGEST
conditional_attribution['attribution_digest'] = ATTRIBUTION_DIGEST
ATTRIBUTION_PARQUET = DATA_OUT / 'P_03_conditional_attribution.parquet'
_atomic_df(conditional_attribution, ATTRIBUTION_PARQUET)

# Article figure: net-of-cost conditional DeltaSharpe.
net_regimes = (regime_attribution[(regime_attribution['basis'] == 'net')
                                  & regime_attribution['name'].isin(['High dispersion', 'Low dispersion'])]
               .set_index('name').loc[['High dispersion', 'Low dispersion']])
net_crises = (crisis_attribution[crisis_attribution['basis'] == 'net']
              .set_index('name').loc[[name for name, _ in CRISES]])
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))
ax = axes[0]
regime_values = net_regimes['delta_sharpe'].to_numpy(float)
bars = ax.bar([r'High $\rho$' + '\n(dispersion)', r'Low $\rho$' + '\n(dispersion)'],
              regime_values, color=[_GREEN if value >= 0 else _RUST for value in regime_values],
              width=0.52, zorder=3)
ax.axhline(0.0, color=_GREY, lw=0.8)
for bar, value in zip(bars, regime_values):
    ax.text(bar.get_x() + bar.get_width()/2, value, f'{value:+.4f}', ha='center',
            va='bottom' if value >= 0 else 'top', fontsize=9.5, color=_GREY)
ax.set_ylabel(r'Net $\Delta\mathrm{SR}$, dynamic minus static')
ax.set_title('By formation-time dispersion regime')
ax.margins(y=0.22)

ax = axes[1]
crisis_values = net_crises['delta_sharpe'].to_numpy(float)
crisis_names = net_crises.index.tolist()
positions = np.arange(len(crisis_names))[::-1]
bars = ax.barh(positions, crisis_values,
               color=[_GREEN if value >= 0 else _RUST for value in crisis_values], height=0.62, zorder=3)
ax.axvline(0.0, color=_GREY, lw=0.8)
ax.set_yticks(positions); ax.set_yticklabels(crisis_names)
for bar, value in zip(bars, crisis_values):
    offset = 0.02 * max(1e-4, np.abs(crisis_values).max())
    ax.text(value + (offset if value >= 0 else -offset), bar.get_y() + bar.get_height()/2,
            f'{value:+.3f}', va='center', ha='left' if value >= 0 else 'right', fontsize=8.8, color=_GREY)
span = max(1e-4, np.abs(crisis_values).max())
ax.set_xlim(-1.25 * span, 1.25 * span)
ax.set_xlabel(r'Net $\Delta\mathrm{SR}$ over $\pm12$ months')
ax.set_title('Around pre-declared stress peaks')
fig.suptitle('Conditional attribution of the timing effect, net of costs', y=1.0)
fig.tight_layout(); save_fig(fig, 5, 'conditional_attribution'); plt.show()

# TABLE P_03 — net primary; gross controls remain in the Parquet audit.
net_regime_table = (regime_attribution[regime_attribution['basis'] == 'net']
                    .set_index('name').loc[['High dispersion', 'Low dispersion', 'All months']])
tex_lines = [r'\begin{tabular}{lrrrrr}', r'\toprule',
             r'Regime & $n$ & Sharpe $P^{(\mathrm{stat})}$ & Sharpe $P^{(\rho)}$ & $\Delta\mathrm{SR}$ & Mean contribution (pp) \\',
             r'\midrule']
for name, label in [('High dispersion', r'High $\rho$'), ('Low dispersion', r'Low $\rho$'), ('All months', 'All')]:
    row = net_regime_table.loc[name]
    contribution = row['mean_contribution_ann'] * 100.0 if name != 'All months' else row['delta_mean_ann'] * 100.0
    tex_lines.append(rf'{label} & {int(row["n"])} & {row["sharpe_static"]:.3f} & {row["sharpe_dynamic"]:.3f} & {row["delta_sharpe"]:+.4f} & {contribution:+.3f} \\')
tex_lines += [r'\midrule', r'\multicolumn{6}{l}{Panel B: pre-declared $\pm12$-month stress windows} \\',
              r'Window & $n$ & Sharpe $P^{(\mathrm{stat})}$ & Sharpe $P^{(\rho)}$ & $\Delta\mathrm{SR}$ & $\Delta\mu$ (pp, ann.) \\',
              r'\midrule']
for crisis_name, _ in CRISES:
    row = net_crises.loc[crisis_name]
    tex_lines.append(rf'{crisis_name} & {int(row["n"])} & {row["sharpe_static"]:.3f} & {row["sharpe_dynamic"]:.3f} & {row["delta_sharpe"]:+.4f} & {row["delta_mean_ann"]*100:+.3f} \\')
tex_lines.extend([r'\bottomrule', r'\end{tabular}'])
attribution_tex = '\n'.join(tex_lines) + '\n'
ATTRIBUTION_TABLE = TAB / 'P_03_conditional_attribution.tex'
_attr_tmp = ATTRIBUTION_TABLE.with_suffix('.tex.tmp')
_attr_tmp.write_text(attribution_tex); _attr_tmp.replace(ATTRIBUTION_TABLE)

print('=' * 72)
print('  Conditional attribution — net of transaction costs')
print('=' * 72)
print(net_regime_table[['n', 'sharpe_static', 'sharpe_dynamic', 'delta_sharpe',
                        'delta_mean_ann', 'mean_contribution_ann']].round(5).to_string())
print('\nCrisis windows:')
print(net_crises[['n', 'sharpe_static', 'sharpe_dynamic', 'delta_sharpe', 'delta_mean_ann']].round(5).to_string())
print('=' * 72)
print(f'saved  tables/{ATTRIBUTION_TABLE.name}')
print(f'saved  audit/{ATTRIBUTION_PARQUET.name}')


## 6.2.6 The portfolio landscape within the DRO family  [FIG P_06 + TABLE P_04]

The static and signal-timed portfolios are placed between two limits of the same linear-loss allocation family. The **empirical-floor** portfolio uses the numerical lower bound $\varepsilon_{floor}=10^{-4}$; it is an operational approximation to the empirical optimiser, not an exact zero-radius solution. The **equal-weight** portfolio is the diversified limiting anchor. All four strategies use the same point-in-time memberships, realised holding factors, delisting cash convention, weight drift, turnover definition, and historical transaction-cost schedule.

Net-of-cost performance is the primary comparison; gross performance is shown to make cost erosion visible. The figure uses a single linear Sharpe axis. Covariance-based portfolios are deliberately excluded because they solve a different objective and do not belong to this continuum.

In [ ]:
# FIG P_06 + TABLE P_04 — four portfolios under one audited accounting engine
LANDSCAPE_VERSION = 'dro-landscape-v1.0.0'
landscape_contract = {
    'version': LANDSCAPE_VERSION, 'engine_digest': ENGINE_DIGEST,
    'strategies': ['empirical_floor', 'static', 'dynamic', 'equal_weight'],
    'empirical_epsilon': EPS_FLOOR, 'equal_weight_assets': 100,
    'accounting': 'cash_drift_costs_v1', 'primary_basis': 'net',
}
LANDSCAPE_DIGEST = hashlib.sha256(json.dumps(landscape_contract, sort_keys=True).encode()).hexdigest()
EMPIRICAL_TARGET_CACHE = CACHE / f'empirical_floor_targets_{LANDSCAPE_DIGEST[:16]}.parquet'

# Resumable target-weight calculation for the only new optimised anchor.
if EMPIRICAL_TARGET_CACHE.exists():
    empirical_targets = pd.read_parquet(EMPIRICAL_TARGET_CACHE)
    if empirical_targets.empty or not empirical_targets['landscape_digest'].eq(LANDSCAPE_DIGEST).all():
        empirical_targets = pd.DataFrame()
else:
    empirical_targets = pd.DataFrame()
valid_cached_groups = (empirical_targets.groupby('formation_month')['PERMNO'].size()
                       if len(empirical_targets) else pd.Series(dtype=int))
completed_formations = set(pd.to_datetime(valid_cached_groups[valid_cached_groups == 100].index))
new_target_rows = []
for ordinal, entry in enumerate(realised_ledger, start=1):
    formation_month = pd.Timestamp(entry['formation_month'])
    if formation_month in completed_formations:
        continue
    weights, status = w_dro(entry['est'], EPS_FLOOR)
    assert weights is not None, f'{formation_month:%Y-%m}: empirical-floor solver failure ({status})'
    assert len(weights) == 100 and np.isfinite(weights).all() and abs(weights.sum() - 1.0) < 1e-10
    for rank, (permno, weight) in enumerate(zip(entry['assets'], weights), start=1):
        new_target_rows.append({
            'formation_month': formation_month, 'PERMNO': int(permno), 'rank': rank,
            'target_weight': float(weight), 'epsilon': EPS_FLOOR, 'solver_status': status,
            'landscape_digest': LANDSCAPE_DIGEST,
        })
    if ordinal % 12 == 0 or ordinal == len(realised_ledger):
        empirical_targets = (pd.concat([empirical_targets, pd.DataFrame(new_target_rows)], ignore_index=True)
                             .drop_duplicates(['formation_month', 'PERMNO'], keep='last')
                             .sort_values(['formation_month', 'rank']).reset_index(drop=True))
        _atomic_df(empirical_targets, EMPIRICAL_TARGET_CACHE); new_target_rows = []
        print(f'  empirical-floor targets: {empirical_targets["formation_month"].nunique()}/{len(realised_ledger)} cached')
assert len(empirical_targets) == len(realised_ledger) * 100
assert empirical_targets.groupby('formation_month')['PERMNO'].nunique().eq(100).all()
assert empirical_targets.groupby('formation_month')['target_weight'].sum().sub(1.0).abs().max() < 1e-10
assert empirical_targets['solver_status'].isin(['optimal', 'optimal_inaccurate']).all()

# Target sources for the four strategies. Static/dynamic are loaded from E3, EW is deterministic.
realised_month_set = {pd.Timestamp(entry['formation_month']) for entry in realised_ledger}
static_targets = target_weights[(target_weights['strategy'] == 'static')
                                & target_weights['formation_month'].isin(realised_month_set)]
dynamic_targets = target_weights[(target_weights['strategy'] == 'dynamic')
                                 & target_weights['formation_month'].isin(realised_month_set)]
equal_weight_targets = pd.DataFrame([
    {'formation_month': pd.Timestamp(entry['formation_month']), 'PERMNO': int(permno),
     'rank': rank, 'target_weight': 1.0 / 100.0}
    for entry in realised_ledger for rank, permno in enumerate(entry['assets'], start=1)
])
target_sources = {
    'empirical_floor': empirical_targets, 'static': static_targets,
    'dynamic': dynamic_targets, 'equal_weight': equal_weight_targets,
}

def _run_landscape_strategy(strategy, targets):
    groups = {pd.Timestamp(fm): group.sort_values('rank')
              for fm, group in targets.groupby('formation_month', sort=True)}
    previous_risky, rows = {}, []
    for entry in realised_ledger:
        formation_month = pd.Timestamp(entry['formation_month'])
        target_frame = groups[formation_month]
        assert target_frame['PERMNO'].astype(int).tolist() == list(map(int, entry['assets']))
        target = dict(zip(target_frame['PERMNO'].astype(int), target_frame['target_weight'].astype(float)))
        assert abs(sum(target.values()) - 1.0) < 1e-10
        risky_union = set(previous_risky) | set(target)
        turnover = float(sum(abs(target.get(p, 0.0) - previous_risky.get(p, 0.0)) for p in risky_union))
        holding_frame = holding_groups[formation_month]
        terminal_values = {p: target[p] * float(holding_frame.loc[p, 'hold_factor']) for p in target}
        gross_factor = float(sum(terminal_values.values()))
        assert gross_factor > 0.0
        previous_risky = {p: value / gross_factor for p, value in terminal_values.items()
                          if not bool(holding_frame.loc[p, 'delisted']) and value > 0.0}
        end_cash = float(sum(value for p, value in terminal_values.items()
                             if bool(holding_frame.loc[p, 'delisted'])) / gross_factor)
        assert abs(sum(previous_risky.values()) + end_cash - 1.0) < 1e-10
        gross_return = gross_factor - 1.0
        transaction_cost = cost_bps(entry['formation_date']) / 1e4 * turnover
        rows.append({
            'strategy': strategy, 'formation_month': formation_month,
            'return_date': pd.Timestamp(entry['holding_month']) + pd.offsets.MonthEnd(0),
            'gross_return': gross_return, 'net_return': gross_return - transaction_cost,
            'turnover': turnover, 'transaction_cost': transaction_cost,
            'end_cash_weight': end_cash, 'n_delisted': int(holding_frame['delisted'].sum()),
            'landscape_digest': LANDSCAPE_DIGEST,
        })
    return pd.DataFrame(rows)

landscape_monthly = pd.concat([_run_landscape_strategy(strategy, targets)
                               for strategy, targets in target_sources.items()], ignore_index=True)
assert len(landscape_monthly) == 4 * 372
assert not landscape_monthly.duplicated(['strategy', 'return_date']).any()
assert np.isfinite(landscape_monthly[['gross_return', 'net_return', 'turnover']]).all().all()

# Hard non-regression guard: the generic accounting must reproduce E4 for static and dynamic.
e4_compare = (landscape_monthly[landscape_monthly['strategy'].isin(['static', 'dynamic'])]
              .merge(portfolio_monthly[['strategy', 'return_date', 'gross_return', 'net_return', 'turnover']],
                     on=['strategy', 'return_date'], suffixes=('_landscape', '_e4'), validate='one_to_one'))
for column in ['gross_return', 'net_return', 'turnover']:
    error = (e4_compare[f'{column}_landscape'] - e4_compare[f'{column}_e4']).abs().max()
    assert error < 1e-12, f'{column}: landscape/E4 mismatch {error}'

landscape_metric_rows = []
for strategy, group in landscape_monthly.groupby('strategy', sort=False):
    group = group.sort_values('return_date')
    for basis in ['gross', 'net']:
        values = portfolio_metrics(group[f'{basis}_return'], group['turnover'])
        landscape_metric_rows.append({'strategy': strategy, 'basis': basis, **values,
                                      'landscape_digest': LANDSCAPE_DIGEST})
landscape_metrics = pd.DataFrame(landscape_metric_rows)
LANDSCAPE_MONTHLY_OUT = DATA_OUT / 'P_04_landscape_monthly.parquet'
LANDSCAPE_METRICS_OUT = DATA_OUT / 'P_04_landscape_metrics.parquet'
_atomic_df(landscape_monthly, LANDSCAPE_MONTHLY_OUT)
_atomic_df(landscape_metrics, LANDSCAPE_METRICS_OUT)

strategy_order = ['empirical_floor', 'static', 'dynamic', 'equal_weight']
strategy_labels = {
    'empirical_floor': r'$P^{(\mathrm{emp})}$ ($\varepsilon_{\min}$)',
    'static': r'$P^{(\mathrm{stat})}$', 'dynamic': r'$P^{(\rho)}$', 'equal_weight': r'$P^{(\mathrm{EW})}$',
}
metric_index = landscape_metrics.set_index(['strategy', 'basis'])
gross_sharpes = np.array([metric_index.loc[(strategy, 'gross'), 'Sharpe'] for strategy in strategy_order])
net_sharpes = np.array([metric_index.loc[(strategy, 'net'), 'Sharpe'] for strategy in strategy_order])
# Appearance follows the reference notebook: horizontal bars on a single continuous axis with a
# piecewise scale (low range compressed, high range dilated) and a white gross->net erosion segment.
# The break points are recalibrated to this sample's Sharpe range (the reference hard-coded 0.10/0.65/0.70).
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

items = [(strategy_labels['empirical_floor'], gross_sharpes[0], _GREY),
         (strategy_labels['static'],          gross_sharpes[1], _NAVY),
         (strategy_labels['dynamic'],         gross_sharpes[2], _RUST),
         (strategy_labels['equal_weight'],    gross_sharpes[3], _GREEN)]
net_sh = list(net_sharpes)
for (name, gross_value, _), net_value in zip(items, net_sh):
    print(f'  {name:28s} Sharpe={gross_value:.4f}  Sharpe_net={net_value:.4f}')

# piecewise scale: [LO, B] compressed onto FRAC of the axis, [B, HI] dilated onto the rest
B, LO, HI = 0.70, 0.38, 0.82
FRAC = 0.45
def fwd(x):
    x = np.asarray(x, float)
    return np.where(x <= B, (x - LO) / (B - LO) * FRAC, FRAC + (x - B) / (HI - B) * (1 - FRAC))
def inv(u):
    u = np.asarray(u, float)
    return np.where(u <= FRAC, LO + u / FRAC * (B - LO), B + (u - FRAC) / (1 - FRAC) * (HI - B))
assert LO < np.r_[gross_sharpes, net_sharpes].min() and np.r_[gross_sharpes, net_sharpes].max() < HI

fig, ax = plt.subplots(figsize=(9.5, 3.4))
y = np.arange(len(items))[::-1]
base = LO
for (name, val, col), yi, vnet in zip(items, y, net_sh):
    ax.barh(yi, val - base, left=base, color=col, height=0.6, zorder=3, alpha=0.85)
    # erosion segment (gross -> net) + net marker
    ax.plot([vnet, val], [yi, yi], color='white', lw=1.6, zorder=4, solid_capstyle='round',
            path_effects=[pe.Stroke(linewidth=2.6, foreground=_GREY), pe.Normal()])
    ax.plot(vnet, yi, marker='|', color='white', ms=12, mew=2.0, zorder=5,
            path_effects=[pe.Stroke(linewidth=3.4, foreground=_GREY), pe.Normal()])
    ax.annotate(f'{val:.3f}', (val, yi), xytext=(6, 0), textcoords='offset points',
                va='center', fontsize=9, color=_GREY)
ax.set_xscale('function', functions=(fwd, inv))
ax.set_xlim(LO, HI)
ax.set_xticks([0.40, 0.50, 0.60, 0.70, 0.72, 0.74, 0.76, 0.78, 0.80])
ax.axvline(B, color='#cccccc', lw=0.8, ls=':', zorder=1)   # subtle marker of scale change
ax.set_yticks(y); ax.set_yticklabels([n for n, _, _ in items])
ax.set_xlabel('Sharpe ratio')
ax.set_title('Sharpe ratio across the DRO continuum')
_neth = Line2D([0], [0], color='white', marker='|', ms=10, mew=2.0, lw=1.6,
               path_effects=[pe.Stroke(linewidth=3.0, foreground=_GREY), pe.Normal()],
               label=r'gross $\rightarrow$ net')
ax.legend(handles=[_neth], loc='upper right', fontsize=8.5, frameon=True,
          framealpha=0.9, edgecolor='#dddddd')
ax.grid(axis='y', alpha=0)
fig.tight_layout(); save_fig(fig, 6, 'landscape'); plt.show()

# TABLE P_04 — net primary, with gross Sharpe retained as a friction-free reference.
tex_lines = [r'\begin{tabular}{lrrrr}', r'\toprule',
             r' & $P^{(\mathrm{emp})}$ & $P^{(\mathrm{stat})}$ & $P^{(\rho)}$ & $P^{(\mathrm{EW})}$ \\', r'\midrule']
def _landscape_row(label, basis, key, scale=1.0, digits=3):
    values = [metric_index.loc[(strategy, basis), key] * scale for strategy in strategy_order]
    return label + ' & ' + ' & '.join(f'{value:.{digits}f}' for value in values) + r' \\'
tex_lines.append(_landscape_row(r'Annual return, net (\%)', 'net', 'annual_return', 100.0, 2))
tex_lines.append(_landscape_row(r'Volatility, net (\%)', 'net', 'volatility', 100.0, 2))
tex_lines.append(_landscape_row('Sharpe, gross', 'gross', 'Sharpe'))
tex_lines.append(_landscape_row('Sharpe, net', 'net', 'Sharpe'))
tex_lines.append(_landscape_row('Maximum drawdown, net', 'net', 'MaxDD'))
tex_lines.append(_landscape_row(r'CVaR$_{95}$, net', 'net', 'CVaR95'))
tex_lines.append(_landscape_row('Recurring two-way turnover', 'net', 'turnover', 1.0, 4))
tex_lines.extend([r'\bottomrule', r'\end{tabular}'])
landscape_tex = '\n'.join(tex_lines) + '\n'
LANDSCAPE_TABLE = TAB / 'P_04_landscape.tex'
_landscape_tmp = LANDSCAPE_TABLE.with_suffix('.tex.tmp')
_landscape_tmp.write_text(landscape_tex); _landscape_tmp.replace(LANDSCAPE_TABLE)

print('=' * 76)
print('  Portfolio landscape — audited net performance')
print('=' * 76)
print(landscape_metrics[landscape_metrics['basis'] == 'net']
      .set_index('strategy')[['annual_return', 'volatility', 'Sharpe', 'MaxDD', 'CVaR95', 'turnover']]
      .loc[strategy_order].round(4).to_string())
print('=' * 76)
print(f'saved  tables/{LANDSCAPE_TABLE.name}')
print(f'saved  audit/{LANDSCAPE_METRICS_OUT.name}')
print(f'saved  audit/{LANDSCAPE_MONTHLY_OUT.name}')


## 6.2.7 Degeneracy of performance-based calibration [FIG P_07 + TABLE P_05]

This diagnostic examines how the ambiguity radius behaves when it is selected exclusively by maximum historical Sharpe. For every radius on the fixed logarithmic grid $\mathcal{E}_{CV}=[0.05,10]$, a genuine monthly walk-forward candidate strategy is constructed. At formation month $M$, its weights use only the latest 36 monthly returns and are then applied to the realised $M+1$ holding period. This produces one strictly out-of-sample candidate return per radius and per month.

The first 36 realised candidate returns, from January 1995 through December 1997, form the explicit initialisation sample. During that interval the displayed and implemented radius is the pre-declared median of the grid and must not be interpreted as cross-validated. Beginning with the January 1998 holding month, the selector chooses the radius with the highest gross Sharpe over all candidate returns realised strictly before the current holding month. The comparison window is expansive: it contains 36 observations at the first selection, 37 one month later, and subsequently grows by one observation every month. Exact ties are resolved in favour of the smallest radius.

After selection, the chosen radius is applied to the current 36-month formation matrix and held during the following month. Gross candidate returns are used for calibration so that the criterion is independent of the turnover path created only after radii have been selected; the implemented selected path is nevertheless evaluated both gross and net using the exact accounting engine above. The purpose is not to claim an optimally tuned competing strategy, but to observe the behaviour of the radius under a maximum-Sharpe CV rule. Persistent selection of the ceiling, together with convergence of its weights toward equal weight, indicates an empirically degenerate calibration rather than an informative interior radius.

In [ ]:
# FIG P_07 + TABLE P_05 — monthly expanding CV over genuine walk-forward candidate paths
CV_VERSION = 'expanding-walk-forward-sharpe-v3.0.0'
CV_GRID = np.round(np.geomspace(0.05, 10.0, 16), 6)
CV_INIT_MONTHS = 36
CV_INIT_EPSILON = float(np.median(CV_GRID))
CV_CEILING = float(CV_GRID[-1])
CV_ALL_RADII = np.unique(np.append(CV_GRID, CV_INIT_EPSILON))
assert len(CV_GRID) == 16 and len(CV_ALL_RADII) in (16, 17)
assert len(realised_ledger) == 372

cv_contract = {
    'cv_version': CV_VERSION, 'engine_digest': ENGINE_DIGEST,
    'candidate_weight_window_months': W_EST, 'grid': CV_GRID.tolist(),
    'initialisation_months': CV_INIT_MONTHS,
    'initialisation_epsilon': CV_INIT_EPSILON,
    'candidate_returns': 'monthly_gross_walk_forward_M_plus_1',
    'selection_score': 'annualised_gross_sharpe_all_prior_candidate_returns',
    'selection_window': 'expanding_from_1995_01',
    'tie_break': 'smallest_epsilon',
    'implemented_path_accounting': 'exact_E4_gross_and_net',
}
CV_DIGEST = hashlib.sha256(json.dumps(cv_contract, sort_keys=True).encode()).hexdigest()
CV_TARGET_CACHE = CACHE / f'cv_expanding_targets_{CV_DIGEST[:16]}.parquet'

# Radius cache key. np.round and the built-in round disagree on exact halves
# (np.round(0.7181635, 6) = 0.718164 while round(0.7181635, 6) = 0.718163), which would make the
# interpolated initialisation radius unrecognisable and silently recomputed at every run.
# A fixed-precision string is unambiguous and used on both sides of the membership test.
EPS_KEY_DECIMALS = 12
def _eps_key(value):
    return f'{float(value):.{EPS_KEY_DECIMALS}f}'
target_columns = {
    'epsilon', 'formation_month', 'PERMNO', 'rank',
    'target_weight', 'solver_status', 'cv_digest',
}

# One signed, resumable target path for each grid radius and for the pre-declared initialisation radius.
if CV_TARGET_CACHE.exists():
    cv_candidate_targets = pd.read_parquet(CV_TARGET_CACHE)
    assert target_columns.issubset(cv_candidate_targets.columns), 'incompatible expanding-CV cache schema'
    assert cv_candidate_targets['cv_digest'].eq(CV_DIGEST).all(), 'stale expanding-CV target cache'
else:
    cv_candidate_targets = pd.DataFrame(columns=sorted(target_columns))

expected_rows_per_radius = len(realised_ledger) * 100
if len(cv_candidate_targets):
    cached_counts = cv_candidate_targets.groupby('epsilon').size()
    incomplete_radii = cached_counts[cached_counts != expected_rows_per_radius].index.to_numpy(float)
    duplicate_radii = cv_candidate_targets.loc[
        cv_candidate_targets.duplicated(['epsilon', 'formation_month', 'PERMNO'], keep=False),
        'epsilon'].unique()
    invalid_radii = set(map(float, incomplete_radii)) | set(map(float, duplicate_radii))
    if invalid_radii:
        cv_candidate_targets = cv_candidate_targets[
            ~cv_candidate_targets['epsilon'].isin(invalid_radii)].copy()
        _atomic_df(cv_candidate_targets, CV_TARGET_CACHE)
completed_radii = {_eps_key(value) for value in cv_candidate_targets['epsilon'].unique()} \
    if len(cv_candidate_targets) else set()

for epsilon in CV_ALL_RADII:
    epsilon = float(epsilon)
    if _eps_key(epsilon) in completed_radii:
        continue
    radius_rows = []
    for entry in realised_ledger:
        weights, status = w_dro(np.asarray(entry['est'], dtype=float), epsilon)
        assert weights is not None, (entry['formation_month'], epsilon, status)
        assert status in ('optimal', 'optimal_inaccurate')
        assert len(weights) == 100 and np.isfinite(weights).all()
        assert (weights >= -1e-12).all() and abs(float(weights.sum()) - 1.0) < 1e-10
        radius_rows.extend({
            'epsilon': epsilon, 'formation_month': pd.Timestamp(entry['formation_month']),
            'PERMNO': int(permno), 'rank': rank, 'target_weight': float(weight),
            'solver_status': status, 'cv_digest': CV_DIGEST,
        } for rank, (permno, weight) in enumerate(zip(entry['assets'], weights), start=1))
    radius_frame = pd.DataFrame(radius_rows)
    assert len(radius_frame) == expected_rows_per_radius
    cv_candidate_targets = pd.concat(
        [cv_candidate_targets, radius_frame], ignore_index=True)
    cv_candidate_targets = cv_candidate_targets.sort_values(
        ['epsilon', 'formation_month', 'rank']).reset_index(drop=True)
    _atomic_df(cv_candidate_targets, CV_TARGET_CACHE)
    completed_radii.add(_eps_key(epsilon))
    print(f'  expanding-CV target checkpoint: epsilon={epsilon:g} '
          f'| {len(completed_radii)}/{len(CV_ALL_RADII)} radii')

expected_formation_months = {pd.Timestamp(entry['formation_month']) for entry in realised_ledger}
assert len(cv_candidate_targets) == len(CV_ALL_RADII) * expected_rows_per_radius
assert set(pd.to_datetime(cv_candidate_targets['formation_month'].unique())) == expected_formation_months
assert not cv_candidate_targets.duplicated(['epsilon', 'formation_month', 'PERMNO']).any()
assert cv_candidate_targets['solver_status'].isin(['optimal', 'optimal_inaccurate']).all()
assert cv_candidate_targets.groupby(
    ['epsilon', 'formation_month'])['target_weight'].sum().sub(1.0).abs().lt(1e-10).all()
assert cv_candidate_targets.groupby(['epsilon', 'formation_month']).size().eq(100).all()

# Realised M+1 gross return of every candidate radius; the current return is never used to select itself.
grid_targets = cv_candidate_targets[
    cv_candidate_targets['epsilon'].isin(CV_GRID)].copy()
candidate_legs = grid_targets.merge(
    holding[['formation_month', 'holding_month', 'PERMNO', 'hold_factor']],
    on=['formation_month', 'PERMNO'], how='left', validate='many_to_one')
assert candidate_legs[['holding_month', 'hold_factor']].notna().all().all()
candidate_legs['weighted_factor'] = (
    candidate_legs['target_weight'] * candidate_legs['hold_factor'])
candidate_monthly = (candidate_legs.groupby(
    ['epsilon', 'formation_month', 'holding_month'], as_index=False)
    .agg(gross_factor=('weighted_factor', 'sum'), n_assets=('PERMNO', 'size')))
candidate_monthly['return_date'] = (
    pd.to_datetime(candidate_monthly['holding_month']) + pd.offsets.MonthEnd(0))
candidate_monthly['gross_return'] = candidate_monthly['gross_factor'] - 1.0
candidate_monthly['cv_digest'] = CV_DIGEST
assert candidate_monthly['n_assets'].eq(100).all()
assert len(candidate_monthly) == len(CV_GRID) * len(realised_ledger)
assert np.isfinite(candidate_monthly['gross_return']).all()

candidate_matrix = (candidate_monthly.pivot(
    index='return_date', columns='epsilon', values='gross_return')
    .sort_index().reindex(columns=CV_GRID))
assert candidate_matrix.shape == (len(realised_ledger), len(CV_GRID))
assert np.isfinite(candidate_matrix.to_numpy(float)).all()
assert candidate_matrix.index.min() == pd.Timestamp('1995-01-31')

# Monthly expanding selector. For row t, [:t] excludes the current M+1 candidate return by construction.
selection_rows, score_rows = [], []
for t, return_date in enumerate(candidate_matrix.index):
    entry = realised_ledger[t]
    expected_return_date = (
        pd.Timestamp(entry['holding_month']) + pd.offsets.MonthEnd(0))
    assert return_date == expected_return_date
    if t < CV_INIT_MONTHS:
        selected_epsilon = CV_INIT_EPSILON
        selected_score = np.nan
        selection_source = 'predeclared_initialisation'
    else:
        prior = candidate_matrix.iloc[:t]
        assert len(prior) == t and prior.index.max() < return_date
        prior_volatility = prior.std(axis=0, ddof=1)
        scores = (np.sqrt(12.0) * prior.mean(axis=0)
                  .div(prior_volatility.where(prior_volatility > 0.0)))
        assert np.isfinite(scores.to_numpy(float)).all()
        # Columns are ascending, hence np.argmax implements the smallest-radius exact-tie rule.
        best_position = int(np.argmax(scores.to_numpy(float)))
        selected_epsilon = float(CV_GRID[best_position])
        selected_score = float(scores.iloc[best_position])
        selection_source = 'expanding_max_sharpe'
        score_rows.extend({
            'formation_month': pd.Timestamp(entry['formation_month']),
            'return_date': return_date, 'epsilon': float(epsilon),
            'prior_observations': t, 'prior_sharpe': float(scores.loc[epsilon]),
            'cv_digest': CV_DIGEST,
        } for epsilon in CV_GRID)
    selection_rows.append({
        'formation_month': pd.Timestamp(entry['formation_month']),
        'return_date': return_date, 'selected_epsilon': selected_epsilon,
        'selected_prior_sharpe': selected_score, 'prior_observations': t,
        'is_initialisation': t < CV_INIT_MONTHS,
        'selection_source': selection_source, 'cv_digest': CV_DIGEST,
    })

cv_selection = pd.DataFrame(selection_rows)
cv_scores = pd.DataFrame(score_rows)
assert len(cv_selection) == len(realised_ledger) == 372
assert int(cv_selection['is_initialisation'].sum()) == CV_INIT_MONTHS
assert cv_selection.loc[cv_selection['is_initialisation'], 'selected_epsilon'].eq(
    CV_INIT_EPSILON).all()
assert cv_selection.loc[~cv_selection['is_initialisation'], 'selected_epsilon'].isin(
    CV_GRID).all()
assert cv_selection['return_date'].is_monotonic_increasing
assert not cv_selection['formation_month'].duplicated().any()
assert len(cv_scores) == (len(realised_ledger) - CV_INIT_MONTHS) * len(CV_GRID)
first_cv_row = cv_selection.loc[~cv_selection['is_initialisation']].iloc[0]
assert first_cv_row['return_date'] == pd.Timestamp('1998-01-31')
assert int(first_cv_row['prior_observations']) == CV_INIT_MONTHS

# Retrieve the already-solved target corresponding to the selected radius at each formation.
cv_selected_targets = (cv_selection[['formation_month', 'selected_epsilon']]
    .merge(cv_candidate_targets,
           left_on=['formation_month', 'selected_epsilon'],
           right_on=['formation_month', 'epsilon'],
           how='left', validate='one_to_many')
    [['formation_month', 'PERMNO', 'rank', 'target_weight']]
    .sort_values(['formation_month', 'rank']).reset_index(drop=True))
assert len(cv_selected_targets) == len(realised_ledger) * 100
assert cv_selected_targets.groupby('formation_month').size().eq(100).all()
assert cv_selected_targets.groupby(
    'formation_month')['target_weight'].sum().sub(1.0).abs().lt(1e-10).all()

# Exact selected-path accounting over the full displayed period, including the disclosed initialisation.
cv_portfolio_monthly = _run_landscape_strategy(
    'cv_expanding', cv_selected_targets)
cv_portfolio_monthly = cv_portfolio_monthly.merge(
    cv_selection[['formation_month', 'selected_epsilon', 'selected_prior_sharpe',
                  'prior_observations', 'is_initialisation', 'selection_source']],
    on='formation_month', how='left', validate='one_to_one')
cv_portfolio_monthly['cv_digest'] = CV_DIGEST
assert len(cv_portfolio_monthly) == len(realised_ledger)
assert cv_portfolio_monthly['return_date'].min() == pd.Timestamp('1995-01-31')
assert np.isfinite(
    cv_portfolio_monthly[['gross_return', 'net_return', 'turnover']]).all().all()

# Degeneracy statistics exclude the 36 pre-declared initialisation months.
post_selection = cv_selection.loc[~cv_selection['is_initialisation']].copy()
fraction_ceiling = float(
    np.isclose(post_selection['selected_epsilon'], CV_CEILING).mean())
fraction_top_region = float(
    (post_selection['selected_epsilon'] >= 7.0).mean())
ceiling_weights = cv_candidate_targets[
    np.isclose(cv_candidate_targets['epsilon'], CV_CEILING)].copy()
ceiling_concentration = ceiling_weights.groupby(
    'formation_month')['target_weight'].apply(
        lambda weights: float(np.linalg.norm(weights.to_numpy(float))))
ceiling_l2_to_ew = ceiling_weights.groupby(
    'formation_month')['target_weight'].apply(
        lambda weights: float(np.linalg.norm(weights.to_numpy(float) - 0.01)))
assert len(ceiling_concentration) == len(realised_ledger)
assert (ceiling_concentration >= 0.1 - 1e-10).all()

# Net performance is matched on the genuine selection period; full-path metrics remain an audit output.
post_dates = set(post_selection['return_date'])
cv_post = (cv_portfolio_monthly[
    cv_portfolio_monthly['return_date'].isin(post_dates)]
    .sort_values('return_date').set_index('return_date'))
ew_post = (landscape_monthly[
    (landscape_monthly['strategy'] == 'equal_weight')
    & landscape_monthly['return_date'].isin(post_dates)]
    .sort_values('return_date').set_index('return_date'))
assert cv_post.index.equals(ew_post.index)
assert len(cv_post) == len(realised_ledger) - CV_INIT_MONTHS
cv_post_net_metrics = portfolio_metrics(cv_post['net_return'], cv_post['turnover'])
ew_post_net_metrics = portfolio_metrics(ew_post['net_return'], ew_post['turnover'])
cv_full_net_metrics = portfolio_metrics(
    cv_portfolio_monthly.set_index('return_date')['net_return'],
    cv_portfolio_monthly.set_index('return_date')['turnover'])

summary_rows = [
    {'statistic': 'CV grid lower bound', 'value': float(CV_GRID[0]), 'unit': 'epsilon'},
    {'statistic': 'CV grid upper bound', 'value': CV_CEILING, 'unit': 'epsilon'},
    {'statistic': 'Initialisation months', 'value': CV_INIT_MONTHS, 'unit': 'months'},
    {'statistic': 'Initialisation epsilon', 'value': CV_INIT_EPSILON, 'unit': 'epsilon'},
    {'statistic': 'First genuine CV holding month',
     'value': float(pd.Timestamp('1998-01-31').toordinal()), 'unit': 'date_ordinal'},
    {'statistic': 'Median selected epsilon after initialisation',
     'value': float(post_selection['selected_epsilon'].median()), 'unit': 'epsilon'},
    {'statistic': 'Fraction selected at ceiling after initialisation',
     'value': fraction_ceiling, 'unit': 'share'},
    {'statistic': 'Fraction selected in top region after initialisation',
     'value': fraction_top_region, 'unit': 'share'},
    {'statistic': 'Mean concentration at epsilon = 10',
     'value': float(ceiling_concentration.mean()), 'unit': 'l2_norm'},
    {'statistic': 'Equal-weight concentration', 'value': 0.1, 'unit': 'l2_norm'},
    {'statistic': 'Mean L2 distance from equal weight at epsilon = 10',
     'value': float(ceiling_l2_to_ew.mean()), 'unit': 'l2_distance'},
    {'statistic': 'Operational interior radius',
     'value': float(EPS_EK), 'unit': 'epsilon'},
    {'statistic': 'Expanding-CV net Sharpe after initialisation',
     'value': float(cv_post_net_metrics['Sharpe']), 'unit': 'ratio'},
    {'statistic': 'Equal-weight net Sharpe on matched period',
     'value': float(ew_post_net_metrics['Sharpe']), 'unit': 'ratio'},
    {'statistic': 'Implemented path net Sharpe including initialisation',
     'value': float(cv_full_net_metrics['Sharpe']), 'unit': 'ratio'},
]
degeneracy_summary = pd.DataFrame(summary_rows).assign(cv_digest=CV_DIGEST)

CV_SELECTION_OUT = DATA_OUT / 'P_05_cv_selection_path.parquet'
CV_SCORES_OUT = DATA_OUT / 'P_05_cv_expanding_scores.parquet'
CV_CANDIDATE_OUT = DATA_OUT / 'P_05_cv_candidate_monthly.parquet'
CV_MONTHLY_OUT = DATA_OUT / 'P_05_cv_portfolio_monthly.parquet'
CV_SUMMARY_OUT = DATA_OUT / 'P_05_degeneracy_summary.parquet'
_atomic_df(cv_selection, CV_SELECTION_OUT)
_atomic_df(cv_scores, CV_SCORES_OUT)
_atomic_df(candidate_monthly, CV_CANDIDATE_OUT)
_atomic_df(cv_portfolio_monthly, CV_MONTHLY_OUT)
_atomic_df(degeneracy_summary, CV_SUMMARY_OUT)

# FIG P_07 — reference composition retained; only the clean candidate data and fixed figure id change.
from matplotlib.ticker import FixedLocator, FixedFormatter
fig, ax = plt.subplots(figsize=(10, 4.0))
for _, peak in CRISES:
    peak_date = pd.Period(peak, freq='M').to_timestamp('M')
    ax.axvspan(peak_date - pd.DateOffset(months=3),
               peak_date + pd.DateOffset(months=3),
               color=_GREY, alpha=0.10, lw=0)
ax.plot(cv_selection['return_date'], cv_selection['selected_epsilon'],
        color=_RUST, lw=1.4,
        label=r'$\varepsilon_{\mathrm{CV},\tau}$ (expanding max-Sharpe, grid $[0.05,10]$)')
ax.axhline(CV_CEILING, color=_GREY, lw=0.9, ls=':',
           label='grid ceiling (10)')
ax.axhline(EPS_EK, color=_NAVY, lw=1.2, ls='--',
           label=rf'operational interior radius $\varepsilon_0={EPS_EK:.2f}$')
ax.set_yscale('log')
_y_ticks = [0.1, 0.3, 1.0, 3.0, 10.0]
ax.yaxis.set_major_locator(FixedLocator(_y_ticks))
ax.yaxis.set_major_formatter(FixedFormatter([f'{tick:g}' for tick in _y_ticks]))
ax.yaxis.set_minor_locator(FixedLocator([]))
ax.set_ylim(0.04, 12.0)
ax.set_xlim(cv_selection['return_date'].min(), cv_selection['return_date'].max())
ax.set_xlabel('Date')
ax.set_ylabel(r'Selected radius $\varepsilon$ (log scale)')
ax.set_title(r'Cross-validated radius $\varepsilon_{\mathrm{CV},\tau}$ over the sample')
ax.legend(loc='center right', fontsize=9)
ax.margins(x=0.01)
fig.tight_layout(); save_fig(fig, 7, 'degeneracy'); plt.show()

# TABLE P_05 — selection behaviour is reported after, not during, initialisation.
tex_lines = [r'\begin{tabular}{lr}', r'\toprule', r'Diagnostic & Value \\', r'\midrule',
    rf'Initialization length (months) & {CV_INIT_MONTHS} \\',
    rf'Initialization radius (grid median) & {CV_INIT_EPSILON:.3f} \\',
    r'First genuine CV holding month & 1998--01 \\',
    rf'Median selected $\varepsilon_{{\mathrm{{CV}}}}$ & {post_selection["selected_epsilon"].median():.3f} \\',
    rf'Fraction at grid ceiling $\varepsilon=10$ (\%) & {100*fraction_ceiling:.1f} \\',
    rf'Fraction with $\varepsilon\geq 7$ (\%) & {100*fraction_top_region:.1f} \\',
    rf'Mean $\lVert w(10)\rVert_2$ & {ceiling_concentration.mean():.4f} \\',
    r'Equal-weight $\lVert w^{(\mathrm{EW})}\rVert_2$ & 0.1000 \\',
    rf'Mean $\lVert w(10)-w^{{(\mathrm{{EW}})}}\rVert_2$ & {ceiling_l2_to_ew.mean():.4f} \\',
    rf'Operational interior radius $\varepsilon_0$ & {EPS_EK:.4f} \\',
    rf'Expanding-CV Sharpe, net (1998--2025) & {cv_post_net_metrics["Sharpe"]:.3f} \\',
    rf'Equal-weight Sharpe, net (1998--2025) & {ew_post_net_metrics["Sharpe"]:.3f} \\',
    r'\bottomrule', r'\end{tabular}']
degeneracy_tex = '\n'.join(tex_lines) + '\n'
DEGENERACY_TABLE = TAB / 'P_05_degeneracy.tex'
_degeneracy_tmp = DEGENERACY_TABLE.with_suffix('.tex.tmp')
_degeneracy_tmp.write_text(degeneracy_tex)
_degeneracy_tmp.replace(DEGENERACY_TABLE)

print('=' * 76)
print('  Performance-based calibration — monthly expanding max-Sharpe audit')
print('=' * 76)
print(f'candidate holdings={candidate_matrix.index.min():%Y-%m}..{candidate_matrix.index.max():%Y-%m} '
      f'| rolling weight window={W_EST} months | grid={CV_GRID[0]:g}..{CV_GRID[-1]:g}')
print(f'initialisation={CV_INIT_MONTHS} months at epsilon={CV_INIT_EPSILON:.4f} '
      f'| first genuine CV holding={first_cv_row["return_date"]:%Y-%m}')
print(f'median selected epsilon={post_selection["selected_epsilon"].median():.4f} | '
      f'ceiling={100*fraction_ceiling:.1f}% | epsilon>=7={100*fraction_top_region:.1f}%')
print(f'mean ||w(10)||2={ceiling_concentration.mean():.6f} | '
      f'mean distance to EW={ceiling_l2_to_ew.mean():.6f}')
print(f'net Sharpe 1998--2025: expanding CV={cv_post_net_metrics["Sharpe"]:.4f} | '
      f'equal weight={ew_post_net_metrics["Sharpe"]:.4f}')
print('=' * 76)
print(f'saved  tables/{DEGENERACY_TABLE.name}')
print(f'saved  audit/{CV_SELECTION_OUT.name}')
print(f'saved  audit/{CV_SCORES_OUT.name}')
print(f'saved  audit/{CV_CANDIDATE_OUT.name}')
print(f'saved  audit/{CV_MONTHLY_OUT.name}')
print(f'saved  audit/{CV_SUMMARY_OUT.name}')


## 6.2.8 The value of timing across capitalization universes  [FIGS P_08--P_09 + TABLES P_06--P_08]

To assess whether the timing result generalizes across capitalization segments, we repeat the static-versus-dynamic comparison on two independently formed point-in-time universes: the broad-exchange big-cap universe selected relative to the NYSE 90th-percentile size breakpoint, and the NYSE-only small--mid-cap universe between the NYSE 20th and 50th size percentiles. The dispersion signal is recomputed within each universe; neither memberships nor signals are transferred across universes. This exercise evaluates cross-universe stability without presuming that either segment must produce the larger full-sample gain.

Both strategies are evaluated with the same audited portfolio engine, 36-month formation window, one-month holding convention, delisting treatment, weight drift, and historical transaction-cost schedule used throughout this section. Net-of-cost $\Delta\mathrm{SR}$ is the primary statistic and its gross counterpart is retained as an audit. Results are reported for the full evaluation sample and for the sample excluding January 1999 through December 2001. The exclusion is a pre-declared sensitivity analysis for concentration of the result around the dot-com episode, not an alternative strategy-selection window.

Inference proceeds hierarchically from one signed panel of $B=1{,}000$ simple monthly permutations. Within each replication, the same monthly rearrangement is applied to the two universe-specific timing paths, both dynamic portfolios are fully recomputed, and all segment statistics are evaluated jointly. **Figure P_08 and Table P_06** report the four observed cells and their unadjusted marginal permutation $p$-values. **Figure P_09 and Table P_07** then use the maximum across these four cells to control family-wise selection, with the small--mid-cap ex-dot-com cell retained as an illustrative reporting target. The full-sample large-cap cell remains the main specification. **Table P_08** extends the same joint permutation experiment to the broader 12-cell diagnostic family. The four-cell family provides the restricted cross-universe and sample-period control, whereas the 12-cell family is a more conservative exploratory extension.

The simple monthly permutation is distinct from the six- and twelve-month block placebos in Section 6.2.4. Those block designs test persistence robustness of the large-cap timing chronology; they do not replace the monthly resampling law used for the cross-universe max-$T$ families. Every reported statistic is reconstructed from the signed draw-level panel, and no observed value or $p$-value is hard-coded.


In [ ]:
# FIG P_08 + TABLE P_06 — observed four-cell comparison only; no permutation is run here.
SMALL_PANEL = DATA / 'processed' / 'nyse_small_caps_p20_p50_pit_daily.parquet'
SMALL_SIGNAL = SIG / 'V_small_uni.parquet'
SMALL_ALLOWED_NULL_RETURNS = {(25312, '1997-09-22')}
SMALL_ALLOWED_NULL_RETURNS |= {(63079, f'2005-04-{day:02d}') for day in [8, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29]}
for _path in (SMALL_PANEL, SMALL_SIGNAL, MONTHLY_OUT):
    assert _path.exists(), f'missing input: {_path}'

_p08_contract = {
    'engine': 'portfolio-pit-observed-universe-v1.0.0',
    'panel_signature': _file_signature(SMALL_PANEL),
    'signal_signature': _file_signature(SMALL_SIGNAL),
    'W_EST': W_EST, 'beta_ek': BETA_EK, 'eps_floor': EPS_FLOOR, 'eps_cap': EPS_CAP,
    'solver': str(SOLVER), 'cost_schedule': 'frazzini-time-varying-per-side-v1',
    'turnover': 'sum_abs_risky_assets_cash_excluded',
    'initial_pretrade_state': '100pct_cash',
    'null_return_policy': sorted([list(x) for x in SMALL_ALLOWED_NULL_RETURNS]),
}
P08_DIGEST = hashlib.sha256(json.dumps(_p08_contract, sort_keys=True).encode()).hexdigest()
P08_TARGET_CACHE = CACHE / f'universe_small_targets_{P08_DIGEST[:16]}.parquet'
P08_SOLVER_CACHE = CACHE / f'universe_small_solver_audit_{P08_DIGEST[:16]}.parquet'
P08_MONTHLY_OUT = DATA_OUT / 'P_06_universe_monthly.parquet'
P08_OBSERVED_OUT = DATA_OUT / 'P_06_universe_observed.parquet'

def _prepare_small_universe():
    raw_u = (pl.read_parquet(SMALL_PANEL).with_columns(pl.col('DlyCalDt').cast(pl.Date))
             .sort(['PERMNO', 'DlyCalDt']).to_pandas())
    raw_u['date'] = pd.to_datetime(raw_u['DlyCalDt'])
    raw_u['PERMNO'] = raw_u['PERMNO'].astype(int)
    pivot_d_u = (raw_u.drop_duplicates(['date', 'PERMNO'])
                 .pivot(index='date', columns='PERMNO', values='DlyRet').sort_index())
    pivot_d_u.columns = pivot_d_u.columns.astype(int)
    pivot_m_u = (1.0 + pivot_d_u).resample('ME').prod(min_count=15) - 1.0

    members = (raw_u.loc[raw_u['is_formation_member'].eq(1), [
        'formation_member_month', 'formation_member_active_month', 'PERMNO',
        'formation_member_rank', 'DlyCalDt', 'formation_member_strict_hist60']]
        .rename(columns={'formation_member_month': 'formation_month',
                         'formation_member_active_month': 'holding_month',
                         'formation_member_rank': 'rank',
                         'DlyCalDt': 'formation_date',
                         'formation_member_strict_hist60': 'strict60'}))
    for c in ['formation_month', 'holding_month', 'formation_date']:
        members[c] = pd.to_datetime(members[c])
    members = members.sort_values(['formation_month', 'rank']).reset_index(drop=True)
    grouped = members.groupby('formation_month')
    assert grouped.size().eq(100).all()
    assert grouped['PERMNO'].nunique().eq(100).all()
    assert grouped['rank'].apply(lambda x: sorted(x.tolist()) == list(range(1, 101))).all()
    assert grouped['formation_date'].nunique().eq(1).all() and members['strict60'].all()
    assert members['holding_month'].eq(members['formation_month'] + pd.offsets.MonthBegin(1)).all()

    ledger_u = []
    for fm, grp in grouped:
        assets = grp['PERMNO'].astype(int).tolist()
        fm_end = pd.Timestamp(fm) + pd.offsets.MonthEnd(0)
        start_end = (pd.Timestamp(fm) - pd.DateOffset(months=W_EST - 1)) + pd.offsets.MonthEnd(0)
        if start_end < pivot_m_u.index.min():
            continue
        est = pivot_m_u.loc[start_end:fm_end, assets]
        assert est.shape == (W_EST, 100), f'{fm:%Y-%m}: small estimation shape={est.shape}'
        assert est.notna().all().all(), f'{fm:%Y-%m}: NaN in small estimation matrix'
        ledger_u.append({'formation_month': pd.Timestamp(fm),
                         'formation_date': pd.Timestamp(grp['formation_date'].iloc[0]),
                         'holding_month': pd.Timestamp(grp['holding_month'].iloc[0]),
                         'assets': assets, 'est': est.to_numpy(float)})
    assert len(ledger_u) == 373

    last_month = raw_u['date'].max().to_period('M')
    realised_u = [e for e in ledger_u if e['holding_month'].to_period('M') <= last_month]
    forecast_u = [e for e in ledger_u if e['holding_month'].to_period('M') > last_month]
    assert len(realised_u) == 372 and len(forecast_u) == 1
    held_pairs = {(int(p), e['holding_month'].to_period('M')) for e in realised_u for p in e['assets']}
    hold_u = raw_u[['PERMNO', 'date', 'DlyRet', 'PrimaryExch', 'is_delisting_event']].copy()
    hold_u['ym'] = hold_u['date'].dt.to_period('M')
    hold_u = hold_u[[(p, m) in held_pairs for p, m in zip(hold_u['PERMNO'], hold_u['ym'])]].copy()
    null_mask = hold_u['DlyRet'].isna()
    null_pairs = set(zip(hold_u.loc[null_mask, 'PERMNO'],
                         hold_u.loc[null_mask, 'date'].dt.strftime('%Y-%m-%d')))
    assert null_pairs == SMALL_ALLOWED_NULL_RETURNS, f'small held-day null set mismatch: {null_pairs}'
    assert int(null_mask.sum()) == len(SMALL_ALLOWED_NULL_RETURNS)
    assert not hold_u.loc[null_mask, 'is_delisting_event'].any()
    hold_u['return_imputed_zero'] = null_mask
    hold_u.loc[null_mask, 'DlyRet'] = 0.0
    hold_u['ym_end'] = hold_u['ym'].dt.to_timestamp('M')
    monthly_factor = (hold_u.groupby(['PERMNO', 'ym_end'])
        .agg(hold_factor=('DlyRet', lambda x: float((1.0 + x).prod())),
             n_days=('DlyRet', 'size'), last_observed_date=('date', 'max'),
             event_date=('date', lambda x: x[hold_u.loc[x.index, 'is_delisting_event']].max()),
             delisted=('is_delisting_event', 'any'), n_imputed=('return_imputed_zero', 'sum'))
        .reset_index())
    market_days = hold_u.groupby('ym_end')['date'].nunique().rename('n_market_days')
    monthly_factor = monthly_factor.join(market_days, on='ym_end')
    assert monthly_factor[(~monthly_factor['delisted']) &
                          (monthly_factor['n_days'] != monthly_factor['n_market_days'])].empty
    assert monthly_factor[monthly_factor['delisted'] &
                          (monthly_factor['last_observed_date'] != monthly_factor['event_date'])].empty
    factor_key = monthly_factor.set_index(['PERMNO', 'ym_end'])
    holding_rows = []
    for e in realised_u:
        month_end = e['holding_month'] + pd.offsets.MonthEnd(0)
        for rank, permno in enumerate(e['assets'], start=1):
            assert (permno, month_end) in factor_key.index
            row = factor_key.loc[(permno, month_end)]
            holding_rows.append({'formation_month': e['formation_month'], 'PERMNO': permno,
                                 'rank': rank, 'hold_factor': float(row['hold_factor']),
                                 'delisted': bool(row['delisted']), 'n_imputed': int(row['n_imputed'])})
    holding_u = pd.DataFrame(holding_rows)
    assert holding_u.groupby('formation_month').size().eq(100).all()
    assert holding_u['n_imputed'].sum() == len(SMALL_ALLOWED_NULL_RETURNS)
    assert holding_u['hold_factor'].ge(0).all()

    rho_u = pd.read_parquet(SMALL_SIGNAL).set_index('date')['rho'].sort_index()
    rho_u.index = pd.to_datetime(rho_u.index) + pd.offsets.MonthEnd(0)
    signal_u = []
    for e in ledger_u:
        signal_date = e['formation_month'] + pd.offsets.MonthEnd(0)
        assert signal_date in rho_u.index
        signal_u.append({'formation_month': e['formation_month'], 'rho': float(rho_u.loc[signal_date]),
                         'sqrt_rho': math.sqrt(float(rho_u.loc[signal_date]))})
    signal_u = pd.DataFrame(signal_u).sort_values('formation_month')
    signal_u['sqrt_rho_expanding_mean'] = signal_u['sqrt_rho'].expanding().mean()
    signal_u['rho_modulation'] = signal_u['sqrt_rho'] / signal_u['sqrt_rho_expanding_mean']
    assert np.isfinite(signal_u['rho_modulation']).all() and signal_u['rho_modulation'].gt(0).all()
    return ledger_u, holding_u, signal_u.set_index('formation_month')

small_ledger, small_holding, small_signal_by_fm = _prepare_small_universe()

def _compute_small_targets():
    weight_rows, solver_rows = [], []
    for ordinal, e in enumerate(small_ledger, start=1):
        fm = e['formation_month']; timing = small_signal_by_fm.loc[fm]
        epsilons = {'static': EPS_EK,
                    'dynamic': float(np.clip(EPS_EK * timing['rho_modulation'], EPS_FLOOR, EPS_CAP))}
        for strategy, epsilon in epsilons.items():
            weights, status = w_dro(e['est'], epsilon)
            assert weights is not None, f'{fm:%Y-%m} small {strategy}: solver failed ({status})'
            assert len(weights) == 100 and np.isfinite(weights).all()
            for rank, (permno, weight) in enumerate(zip(e['assets'], weights), start=1):
                weight_rows.append({'formation_month': fm, 'strategy': strategy, 'PERMNO': int(permno),
                                    'rank': rank, 'target_weight': float(weight), 'epsilon': float(epsilon),
                                    'rho_modulation': float(timing['rho_modulation']),
                                    'engine_digest': P08_DIGEST})
            solver_rows.append({'formation_month': fm, 'strategy': strategy, 'solver_status': status,
                                'epsilon': float(epsilon), 'weight_sum': float(weights.sum()),
                                'weight_l2': float(np.linalg.norm(weights)), 'engine_digest': P08_DIGEST})
        if ordinal % 24 == 0:
            print(f'  small-universe targets {ordinal}/{len(small_ledger)}')
    return pd.DataFrame(weight_rows), pd.DataFrame(solver_rows)

if P08_TARGET_CACHE.exists() and P08_SOLVER_CACHE.exists():
    small_targets = pd.read_parquet(P08_TARGET_CACHE)
    small_solver_audit = pd.read_parquet(P08_SOLVER_CACHE)
    _small_cache_ok = (not small_targets.empty and not small_solver_audit.empty
                       and small_targets['engine_digest'].eq(P08_DIGEST).all()
                       and small_solver_audit['engine_digest'].eq(P08_DIGEST).all())
else:
    _small_cache_ok = False
if not _small_cache_ok:
    small_targets, small_solver_audit = _compute_small_targets()
    _atomic_df(small_targets, P08_TARGET_CACHE); _atomic_df(small_solver_audit, P08_SOLVER_CACHE)
    print('small-universe targets computed and saved to signed cache')
else:
    print('small-universe targets loaded from signed cache')
assert len(small_targets) == len(small_ledger) * 2 * 100
assert small_solver_audit['solver_status'].isin(['optimal', 'optimal_inaccurate']).all()

def _account_small_observed():
    holding_groups = {pd.Timestamp(fm): grp.set_index('PERMNO')
                      for fm, grp in small_holding.groupby('formation_month', sort=True)}
    rows = []
    for strategy in ['static', 'dynamic']:
        pre_risky = {}
        for e in small_ledger:
            fm = e['formation_month']
            if fm not in holding_groups:
                continue
            tw = small_targets[(small_targets['formation_month'] == fm) &
                               (small_targets['strategy'] == strategy)].sort_values('rank')
            assert tw['PERMNO'].astype(int).tolist() == e['assets']
            target = dict(zip(tw['PERMNO'].astype(int), tw['target_weight'].astype(float)))
            turnover = float(sum(abs(target.get(p, 0.0) - pre_risky.get(p, 0.0))
                                 for p in set(target) | set(pre_risky)))
            h = holding_groups[fm]
            assert set(h.index.astype(int)) == set(target)
            terminal = {p: target[p] * float(h.loc[p, 'hold_factor']) for p in target}
            gross_factor = float(sum(terminal.values()))
            assert gross_factor > 0.0
            pre_risky = {p: value / gross_factor for p, value in terminal.items()
                         if not bool(h.loc[p, 'delisted']) and value > 0.0}
            end_cash = float(sum(value for p, value in terminal.items()
                                 if bool(h.loc[p, 'delisted'])) / gross_factor)
            assert abs(sum(pre_risky.values()) + end_cash - 1.0) < 1e-10
            gross_return = gross_factor - 1.0
            transaction_cost = cost_bps(e['formation_date']) / 1e4 * turnover
            rows.append({'universe': 'small_caps_p20_p50', 'formation_month': fm,
                         'return_date': e['holding_month'] + pd.offsets.MonthEnd(0),
                         'strategy': strategy, 'gross_return': gross_return,
                         'net_return': gross_return - transaction_cost, 'turnover': turnover,
                         'transaction_cost': transaction_cost, 'end_cash_weight': end_cash,
                         'n_delisted': int(h['delisted'].sum()), 'engine_digest': P08_DIGEST})
    out = pd.DataFrame(rows).sort_values(['strategy', 'return_date']).reset_index(drop=True)
    assert len(out) == 2 * 372
    assert out.groupby('strategy')['return_date'].nunique().eq(372).all()
    assert np.isfinite(out[['gross_return', 'net_return', 'turnover']]).all().all()
    return out

small_monthly = _account_small_observed()
big_monthly = pd.read_parquet(MONTHLY_OUT).copy()
big_monthly['universe'] = 'big_caps'
big_monthly = big_monthly[['universe', 'formation_month', 'return_date', 'strategy',
                           'gross_return', 'net_return', 'turnover', 'transaction_cost',
                           'end_cash_weight', 'n_delisted', 'engine_digest']]
universe_monthly = pd.concat([big_monthly, small_monthly], ignore_index=True)
assert universe_monthly.groupby(['universe', 'strategy']).size().eq(372).all()
assert universe_monthly.groupby('universe')['return_date'].apply(tuple).nunique() == 1
_atomic_df(universe_monthly, P08_MONTHLY_OUT)

def _p08_sharpe(values):
    values = pd.Series(values).dropna()
    assert len(values) > 1 and values.std(ddof=1) > 0
    return float(values.mean() / values.std(ddof=1) * np.sqrt(12.0))

observed_rows = []
for universe, frame in universe_monthly.groupby('universe', sort=False):
    frame = frame.copy(); frame['return_date'] = pd.to_datetime(frame['return_date'])
    for sample, mask in {
        'full': np.ones(len(frame), dtype=bool),
        'ex_dotcom': ~frame['return_date'].between('1999-01-01', '2001-12-31'),
    }.items():
        sub = frame.loc[mask]
        def _recurring_turnover(strategy):
            values = (sub.loc[sub['strategy'].eq(strategy)]
                      .sort_values('return_date')['turnover'].reset_index(drop=True))
            assert len(values) >= 2 and np.isclose(values.iloc[0], 1.0, rtol=0.0, atol=1e-12)
            return float(values.iloc[1:].mean())
        for basis, return_col in [('net', 'net_return'), ('gross', 'gross_return')]:
            static = sub.loc[sub['strategy'].eq('static'), return_col]
            dynamic = sub.loc[sub['strategy'].eq('dynamic'), return_col]
            assert len(static) == len(dynamic)
            static_sharpe, dynamic_sharpe = _p08_sharpe(static), _p08_sharpe(dynamic)
            observed_rows.append({'universe': universe, 'sample': sample, 'basis': basis,
                                  'n_months': len(static), 'static_sharpe': static_sharpe,
                                  'dynamic_sharpe': dynamic_sharpe,
                                  'delta_sharpe': dynamic_sharpe - static_sharpe,
                                  'static_turnover': _recurring_turnover('static'),
                                  'dynamic_turnover': _recurring_turnover('dynamic')})
universe_observed = pd.DataFrame(observed_rows)
assert len(universe_observed) == 8 and universe_observed['delta_sharpe'].notna().all()
# Self-contained inference guard: this cell may be run after a kernel restart.
if 'timing_inference' not in globals():
    import sys
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.portfolio_timing_tests import (
        inference_tables as _timing_inference_tables,
        load_or_compute as _timing_load_or_compute,
    )
    TIMING_DIGEST_SHORT = '88e42643b59e'
    TIMING_MANIFEST = CACHE / f'timing_test_manifest_{TIMING_DIGEST_SHORT}.json'
    TIMING_INPUTS = CACHE / f'timing_test_inputs_{TIMING_DIGEST_SHORT}.npz'
    TIMING_CACHE = CACHE / f'timing_tests_B1000_{TIMING_DIGEST_SHORT}.parquet'
    timing_manifest, timing_arrays, timing_draws = _timing_load_or_compute(
        TIMING_MANIFEST, TIMING_INPUTS, TIMING_CACHE,
        threads=1, checkpoint_every=5, progress_every=5)
    timing_placebo, timing_inference = _timing_inference_tables(
        timing_draws, timing_manifest, timing_arrays)

_p08_cell = {('big_caps', 'full'): 'L_all', ('big_caps', 'ex_dotcom'): 'L_exdc',
             ('small_caps_p20_p50', 'full'): 'S_all',
             ('small_caps_p20_p50', 'ex_dotcom'): 'S_exdc'}
universe_observed['cell'] = [
    _p08_cell[(u, sample)] for u, sample in zip(universe_observed['universe'], universe_observed['sample'])]
_p08_inference = timing_inference[timing_inference['cell'].isin(timing_manifest['cells4'])][
    ['basis', 'cell', 'p_marginal', 'marginal_exceedances', 'experiment_digest']]
universe_observed = universe_observed.merge(_p08_inference, on=['basis', 'cell'], how='left', validate='one_to_one')
assert universe_observed['p_marginal'].notna().all()
_atomic_df(universe_observed, P08_OBSERVED_OUT)

# FIG P_08 — primary net-of-cost statistic; the gross audit remains in the parquet and table.
net_obs = universe_observed[universe_observed['basis'].eq('net')].set_index(['universe', 'sample'])
figure_values = {
    'big_caps': [net_obs.loc[('big_caps', 'full'), 'delta_sharpe'],
                 net_obs.loc[('big_caps', 'ex_dotcom'), 'delta_sharpe']],
    'small_caps_p20_p50': [net_obs.loc[('small_caps_p20_p50', 'full'), 'delta_sharpe'],
                             net_obs.loc[('small_caps_p20_p50', 'ex_dotcom'), 'delta_sharpe']],
}
fig, ax = plt.subplots(figsize=(8.4, 4.4))
x = np.arange(2); width = 0.36
bars_big = ax.bar(x - width / 2, figure_values['big_caps'], width, color=_NAVY,
                  label='Big caps (top 100 above NYSE p90)')
bars_small = ax.bar(x + width / 2, figure_values['small_caps_p20_p50'], width, color=_RUST,
                    label='NYSE small–mid caps (p20–p50)')
ax.axhline(0.0, color=_GREY, lw=0.8)
for bars in (bars_big, bars_small):
    for bar in bars:
        value = bar.get_height()
        ax.annotate(f'{value:+.4f}', (bar.get_x() + bar.get_width() / 2, value),
                    xytext=(0, 4 if value >= 0 else -4), textcoords='offset points',
                    ha='center', va='bottom' if value >= 0 else 'top', fontsize=9, color=_GREY)
all_values = np.asarray(figure_values['big_caps'] + figure_values['small_caps_p20_p50'], float)
padding = max(0.001, 0.18 * np.max(np.abs(all_values)))
ax.set_ylim(min(0.0, float(all_values.min())) - (padding if all_values.min() < 0 else 0.0),
            max(0.0, float(all_values.max())) + padding)
ax.set_xticks(x, ['Full sample\n(1995–2025)', 'Excluding dot-com\n(ex 1999–2001)'])
ax.set_ylabel(r'Net $\Delta\mathrm{SR}$ (dynamic $-$ static)')
ax.set_title('Timing gain by capitalization universe')
ax.legend(loc='best', fontsize=9.2)
fig.tight_layout(); save_fig(fig, 8, 'universe_gain'); plt.show()

# TABLE P_06 — observed estimates only; multiplicity-adjusted inference is reported next.
def _p08_value(universe, sample, basis, column='delta_sharpe'):
    row = universe_observed[(universe_observed['universe'] == universe) &
                            (universe_observed['sample'] == sample) &
                            (universe_observed['basis'] == basis)]
    assert len(row) == 1
    return float(row.iloc[0][column])

def _p08_table_row(label, sample, basis, column, number_format):
    values = [_p08_value(universe, sample, basis, column)
              for universe in ('big_caps', 'small_caps_p20_p50')]
    return f'{label} & {format(values[0], number_format)} & {format(values[1], number_format)} ' + r'\\'

_table_rows = [
    r'\multicolumn{3}{l}{\textit{Net timing inference}} \\',
    _p08_table_row(r'Full sample $\Delta\mathrm{SR}$', 'full', 'net', 'delta_sharpe', '+.4f'),
    _p08_table_row(r'Full sample $p$-value', 'full', 'net', 'p_marginal', '.3f'),
    _p08_table_row(r'Ex dot-com $\Delta\mathrm{SR}$', 'ex_dotcom', 'net', 'delta_sharpe', '+.4f'),
    _p08_table_row(r'Ex dot-com $p$-value', 'ex_dotcom', 'net', 'p_marginal', '.3f'),
    r'\addlinespace[4pt]',
    r'\multicolumn{3}{l}{\textit{Gross audit}} \\',
    _p08_table_row(r'Full sample $\Delta\mathrm{SR}$', 'full', 'gross', 'delta_sharpe', '+.4f'),
    _p08_table_row(r'Ex dot-com $\Delta\mathrm{SR}$', 'ex_dotcom', 'gross', 'delta_sharpe', '+.4f'),
    r'\addlinespace[4pt]',
    r'\multicolumn{3}{l}{\textit{Recurring turnover}} \\',
    _p08_table_row('Static', 'full', 'net', 'static_turnover', '.4f'),
    _p08_table_row('Dynamic', 'full', 'net', 'dynamic_turnover', '.4f'),
]
p08_tex = '\n'.join([
    r'\begin{tabular}{lcc}', r'\toprule',
    r'Metric & \shortstack{Big caps\\(top 100 above NYSE $P90$)} & \shortstack{NYSE small--mid caps\\($P20$--$P50$)} \\',
    r'\midrule', *_table_rows, r'\bottomrule', r'\end{tabular}']) + '\n'
P08_TABLE = TAB / 'P_06_universe.tex'
_p08_table_tmp = P08_TABLE.with_suffix('.tex.tmp')
_p08_table_tmp.write_text(p08_tex); _p08_table_tmp.replace(P08_TABLE)

print('=' * 76)
print('  Observed timing gain across capitalization universes — no permutations')
print('=' * 76)
display(universe_observed.sort_values(['basis', 'universe', 'sample']))
print(f'saved  images/R_P_08_universe_gain.pdf')
print(f'saved  tables/{P08_TABLE.name}')
print(f'saved  audit/{P08_MONTHLY_OUT.name}')
print(f'saved  audit/{P08_OBSERVED_OUT.name}')


### 6.2.8.1 Dependence-preserving max-$T$ inference  [FIG P_09 + TABLES P_07--P_08]

At each of the $B=1{,}000$ replications, a single simple monthly permutation is applied synchronously to the big- and small--mid-cap timing modulations. The two dynamic portfolios are then re-solved over all 372 formation months with unchanged point-in-time estimation windows, realised holding factors, delisting cash, risky-weight drift, and transaction costs. The common rearrangement preserves cross-universe dependence, while the full and ex-dot-com statistics remain dependent evaluations of the same simulated paths.

The primary family contains the four cells in Figure P_08. Their four placebo $\Delta\mathrm{SR}$ statistics are computed jointly and their maximum defines the max-$T_4$ reference distribution. Figure P_09 compares the pre-declared `S_exdc` statistic with this best-of-family null; Table P_07 reports the four marginal and adjusted results. Table P_08 retains the same target and the same replications but expands the within-draw maximum to 12 cells by adding six non-overlapping universe--decade cells and two large-cap dispersion-state cells. Consequently, the extended adjusted $p$-value must be no smaller than its four-cell counterpart.

The draw panel is loaded only after hard checks of its experiment digest, draw coverage, three seed sequences, finite statistics, and reconstructed max-$T_4$ and max-$T_{12}$ columns. If unavailable, the shared engine in Section 6.2.4 recomputes and checkpoints all three inferential levels. All upper-tail $p$-values use the finite-sample correction $(r+1)/(B+1)$.


In [ ]:
# FIG P_09 + TABLES P_07--P_08 — monthly joint family inference from the signed panel.
if 'timing_draws' not in globals():
    import sys
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.portfolio_timing_tests import (
        inference_tables as _timing_inference_tables,
        load_or_compute as _timing_load_or_compute,
    )
    TIMING_DIGEST_SHORT = '88e42643b59e'
    TIMING_MANIFEST = CACHE / f'timing_test_manifest_{TIMING_DIGEST_SHORT}.json'
    TIMING_INPUTS = CACHE / f'timing_test_inputs_{TIMING_DIGEST_SHORT}.npz'
    TIMING_CACHE = CACHE / f'timing_tests_B1000_{TIMING_DIGEST_SHORT}.parquet'
    timing_manifest, timing_arrays, timing_draws = _timing_load_or_compute(
        TIMING_MANIFEST, TIMING_INPUTS, TIMING_CACHE,
        threads=1, checkpoint_every=5, progress_every=5)
    timing_placebo, timing_inference = _timing_inference_tables(
        timing_draws, timing_manifest, timing_arrays)

assert timing_manifest['target_cell'] == 'S_exdc'
assert timing_manifest['method_blocks']['simple'] == 1
assert len(timing_draws) == timing_manifest['draws'] == 1000
_maxt4 = timing_inference[timing_inference['cell'].isin(timing_manifest['cells4'])].copy()
_maxt12 = timing_inference.copy()
_target_net = _maxt12[( _maxt12['basis'] == 'net') & _maxt12['is_target']]
assert len(_target_net) == 1
_target_net = _target_net.iloc[0]
assert _target_net['p_maxT12'] >= _target_net['p_maxT4']

MAXT_DRAWS_OUT = DATA_OUT / 'P_06_timing_tests_draws.parquet'
MAXT_4_OUT = DATA_OUT / 'P_07_maxt4_inference.parquet'
MAXT_12_OUT = DATA_OUT / 'P_08_maxt12_inference.parquet'
MAXT_SUMMARY_OUT = DATA_OUT / 'P_06_timing_tests_summary.json'
_atomic_df(timing_draws, MAXT_DRAWS_OUT)
_atomic_df(_maxt4, MAXT_4_OUT); _atomic_df(_maxt12, MAXT_12_OUT)
_summary = {
    'schema_version': 'portfolio-timing-tests-notebook-v1.0.0',
    'experiment_digest': timing_manifest['experiment_digest'],
    'draws': len(timing_draws), 'target_cell': timing_manifest['target_cell'],
    'target_observed_delta_sharpe': float(_target_net['observed_delta_sharpe']),
    'target_p_marginal': float(_target_net['p_marginal']),
    'target_p_maxT4': float(_target_net['p_maxT4']),
    'target_p_maxT12': float(_target_net['p_maxT12']),
}
_tmp = MAXT_SUMMARY_OUT.with_suffix('.json.tmp')
_tmp.write_text(json.dumps(_summary, indent=2, sort_keys=True) + '\n'); _tmp.replace(MAXT_SUMMARY_OUT)

# FIG P_09 — reference composition: one max-T4 null and the illustrative S_exdc target.
_maxdist4 = timing_draws['net_maxT4'].to_numpy()
_obs_target = float(_target_net['observed_delta_sharpe'])
_p_target4 = float(_target_net['p_maxT4'])
fig, ax = plt.subplots(figsize=(10.6, 4.2))
ax.hist(_maxdist4, bins=18, color=_GREY, alpha=.6,
        label=f'Best-of-four monthly\npermutation ($B={len(_maxdist4):,}$)')
ax.axvline(_obs_target, color=_RUST, lw=2.2,
           label='Observed small–mid caps\n(ex dot-com)')
ax.axvline(_maxdist4.mean(), color=_NAVY, lw=1.2, ls='--',
           label=f'Null mean ({_maxdist4.mean():+.4f})')
ax.set_xlabel(r'$\max_{c\in\mathcal{F}_4}\;\Delta\mathrm{SR}$ (dynamic $-$ static)')
ax.set_ylabel('Frequency')
ax.set_title('Family-wise max-$T$ test across the four reported cells', pad=12)
ax.legend(loc='upper right', fontsize=8.3,
          framealpha=.96, facecolor='white', edgecolor='#dddddd',
          handlelength=2.5, labelspacing=.75)
ax.text(.02, .96, fr'$p_{{\mathrm{{max\text{{-}}T_4}}}}={_p_target4:.3f}$',
        transform=ax.transAxes, va='top', ha='left', fontsize=9.5,
        bbox=dict(boxstyle='round,pad=.4', facecolor='white', edgecolor='#dddddd'))
fig.subplots_adjust(top=.90); fig.tight_layout(); save_fig(fig, 9, 'maxt_multiplicity'); plt.show()

_labels = {'L_all':'Big caps, full sample','L_exdc':'Big caps, ex dot-com',
'S_all':'Small--mid caps, full sample','S_exdc':'Small--mid caps, ex dot-com',
'L_d1':'Big caps, 1995--2004','L_d2':'Big caps, 2005--2014','L_d3':'Big caps, 2015--2025',
'S_d1':'Small--mid caps, 1995--2004','S_d2':'Small--mid caps, 2005--2014',
'S_d3':'Small--mid caps, 2015--2025','L_rho_hi':'Big caps, high dispersion state',
'L_rho_lo':'Big caps, low dispersion state'}
_net4 = _maxt4[_maxt4['basis'].eq('net')].set_index('cell').loc[timing_manifest['cells4']]
_rows4 = [f'{_labels[c]} & {r.observed_delta_sharpe:+.4f} & {r.p_marginal:.3f} & {r.p_maxT4:.3f} ' + r'\\'
          for c, r in _net4.iterrows()]
_tex4 = '\n'.join([r'\begin{tabular}{lrrr}', r'\toprule',
r'Cell & Net $\Delta\mathrm{SR}$ & Marginal $p$ & max-$T_4$ $p$ \\',
r'\midrule', *_rows4, r'\bottomrule', r'\end{tabular}']) + '\n'
P07_TABLE = TAB / 'P_07_maxt4.tex'; _tmp = P07_TABLE.with_suffix('.tex.tmp')
_tmp.write_text(_tex4); _tmp.replace(P07_TABLE)

_net12 = _maxt12[_maxt12['basis'].eq('net')].set_index('cell').loc[timing_manifest['cells12']]
_rows12 = [f'{_labels[c]} & {r.observed_delta_sharpe:+.4f} & {r.p_marginal:.3f} & {r.p_maxT12:.3f} ' + r'\\'
           for c, r in _net12.iterrows()]
_tex12 = '\n'.join([r'\begin{tabular}{lrrr}', r'\toprule',
r'Cell & Net $\Delta\mathrm{SR}$ & Marginal $p$ & max-$T_{12}$ $p$ \\',
r'\midrule', *_rows12, r'\bottomrule', r'\end{tabular}']) + '\n'
P08_TABLE = TAB / 'P_08_maxt12.tex'; _tmp = P08_TABLE.with_suffix('.tex.tmp')
_tmp.write_text(_tex12); _tmp.replace(P08_TABLE)

print('=' * 76)
print('  Monthly dependence-preserving timing inference')
print('=' * 76)
display(_net4[['observed_delta_sharpe', 'p_marginal', 'p_maxT4']])
print(f'TARGET S_exdc | p_marginal={_target_net["p_marginal"]:.4f} '
      f'| p_maxT4={_target_net["p_maxT4"]:.4f} | p_maxT12={_target_net["p_maxT12"]:.4f}')
print('saved  images/R_P_09_maxT_multiplicity.pdf')
print(f'saved  tables/{P07_TABLE.name}'); print(f'saved  tables/{P08_TABLE.name}')
print(f'saved  audit/{MAXT_DRAWS_OUT.name}')
print(f'saved  audit/{MAXT_4_OUT.name}'); print(f'saved  audit/{MAXT_12_OUT.name}')


## 6.2.9 Robustness  [FIGS P_10--P_12 + TABLE P_09]

The preceding results are conditional on a deliberately transparent reference specification. This section evaluates whether the measured timing contribution depends on a narrow region of the ambiguity-radius continuum, on a particular construction choice, or on one historical episode. The analysis is organised into three complementary diagnostics: the useful operating window of the baseline radius, controlled variations of the signal and portfolio construction, and temporal stability across non-overlapping subperiods.

Every specification is evaluated with the same point-in-time memberships, realised holding factors, delisting convention, risky-weight drift, turnover definition, and historical transaction-cost schedule as the reference portfolios. Net-of-cost $\Delta\mathrm{SR}$ is the primary statistic; the gross counterpart is retained as an implementation audit. A failed optimisation is always a hard failure and is never replaced by equal weight. Each expensive specification is signed by its complete data-and-method contract and checkpointed independently.

### 6.2.9.1 The useful operating window of the radius  [FIG P_10]

The value of timing need not be uniform across the baseline ambiguity radius $\bar\varepsilon$. For each point on a pre-declared logarithmic grid, the static strategy holds $\bar\varepsilon$ fixed while the dynamic strategy applies the same point-in-time modulation used in the reference specification. Both paths are re-solved and passed through the exact accounting engine. Figure P_10 reports the resulting net timing gain together with the mean static weight concentration; the gross timing gain is shown as a secondary numerical audit.

At very large radii, both strategies approach the diversified limit and their difference should vanish. At very small radii, the allocation is more concentrated and the effect of changing the radius can be unstable. An interior region with a positive net gain therefore identifies where the signal has economically relevant portfolio consequences. The Esfahani--Kuhn radius is included explicitly on the grid and marked ex ante; the sweep is diagnostic and does not select a replacement operating point.


In [ ]:
# FIG P_10 — useful radius window under the exact PIT accounting engine.
USEFUL_BASELINES = np.unique(np.r_[np.round(np.geomspace(0.10, 2.00, 10), 6), EPS_EK])
_useful_contract = {
    'engine': 'portfolio-useful-radius-window-v1.0.0',
    'base_engine_digest': ENGINE_DIGEST, 'baselines': USEFUL_BASELINES.tolist(),
    'dynamic_modulation': 'sqrt_rho_over_expanding_mean',
    'primary_basis': 'net', 'gross_audit': True,
    'turnover': 'sum_abs_risky_assets_cash_excluded',
    'initial_pretrade_state': '100pct_cash', 'solver': str(SOLVER),
    'W_EST': W_EST, 'eps_floor': EPS_FLOOR, 'eps_cap': EPS_CAP,
}
USEFUL_DIGEST = hashlib.sha256(json.dumps(_useful_contract, sort_keys=True).encode()).hexdigest()
USEFUL_CACHE = CACHE / f'useful_radius_monthly_{USEFUL_DIGEST[:16]}.parquet'
USEFUL_SOLVER_CACHE = CACHE / f'useful_radius_solver_audit_{USEFUL_DIGEST[:16]}.parquet'
USEFUL_MONTHLY_OUT = DATA_OUT / 'P_09_useful_radius_monthly.parquet'
USEFUL_SUMMARY_OUT = DATA_OUT / 'P_09_useful_radius_summary.parquet'
USEFUL_SOLVER_OUT = DATA_OUT / 'P_09_useful_radius_solver_audit.parquet'

def _run_useful_radius_path(base_radius, strategy):
    assert strategy in {'static', 'dynamic'}
    previous_risky = {}
    monthly_rows, solver_rows = [], []
    for ordinal, entry in enumerate(realised_ledger, start=1):
        fm = pd.Timestamp(entry['formation_month'])
        modulation = float(signal_by_fm.loc[fm, 'rho_modulation']) if strategy == 'dynamic' else 1.0
        epsilon = float(np.clip(base_radius * modulation, EPS_FLOOR, EPS_CAP))
        weights, status = w_dro(entry['est'], epsilon)
        assert weights is not None, f'{fm:%Y-%m} base={base_radius:g} {strategy}: {status}'
        assert len(weights) == 100 and np.isfinite(weights).all()
        assert (weights >= -1e-12).all() and abs(weights.sum() - 1.0) < 1e-10
        target = dict(zip(map(int, entry['assets']), map(float, weights)))
        turnover = float(sum(abs(target.get(asset, 0.0) - previous_risky.get(asset, 0.0))
                             for asset in set(target) | set(previous_risky)))
        holding_month = holding_groups[fm]
        assert set(holding_month.index.astype(int)) == set(target)
        terminal = {asset: target[asset] * float(holding_month.loc[asset, 'hold_factor'])
                    for asset in target}
        gross_factor = float(sum(terminal.values()))
        assert np.isfinite(gross_factor) and gross_factor > 0.0
        weighted_return = float(sum(target[asset] * (float(holding_month.loc[asset, 'hold_factor']) - 1.0)
                                    for asset in target))
        gross_return = gross_factor - 1.0
        assert abs(gross_return - weighted_return) < 1e-12
        previous_risky = {asset: value / gross_factor for asset, value in terminal.items()
                          if not bool(holding_month.loc[asset, 'delisted']) and value > 0.0}
        end_cash = float(sum(value for asset, value in terminal.items()
                             if bool(holding_month.loc[asset, 'delisted'])) / gross_factor)
        assert abs(sum(previous_risky.values()) + end_cash - 1.0) < 1e-10
        transaction_cost = cost_bps(entry['formation_date']) / 1e4 * turnover
        monthly_rows.append({
            'base_radius': float(base_radius), 'strategy': strategy,
            'formation_month': fm, 'return_date': entry['holding_month'] + pd.offsets.MonthEnd(0),
            'epsilon': epsilon, 'gross_return': gross_return,
            'net_return': gross_return - transaction_cost, 'turnover': turnover,
            'transaction_cost': transaction_cost, 'end_cash_weight': end_cash,
            'weight_l2': float(np.linalg.norm(weights)),
            'n_delisted': int(holding_month['delisted'].sum()), 'engine_digest': USEFUL_DIGEST,
        })
        solver_rows.append({
            'base_radius': float(base_radius), 'strategy': strategy, 'formation_month': fm,
            'epsilon': epsilon, 'solver_status': status, 'weight_sum': float(weights.sum()),
            'weight_min': float(weights.min()), 'weight_max': float(weights.max()),
            'weight_l2': float(np.linalg.norm(weights)), 'engine_digest': USEFUL_DIGEST,
        })
        if ordinal % 60 == 0:
            print(f'    {strategy} {ordinal}/{len(realised_ledger)}')
    return pd.DataFrame(monthly_rows), pd.DataFrame(solver_rows)

def _load_compatible_useful(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_parquet(path)
    if frame.empty or 'engine_digest' not in frame or not frame['engine_digest'].eq(USEFUL_DIGEST).all():
        raise RuntimeError(f'incompatible useful-radius cache: {path}')
    return frame

useful_monthly = _load_compatible_useful(USEFUL_CACHE)
useful_solver_audit = _load_compatible_useful(USEFUL_SOLVER_CACHE)
def _completed_useful_bases(monthly_frame, solver_frame):
    expected = 2 * len(realised_ledger)
    monthly_ok = set(monthly_frame.groupby('base_radius').size().loc[lambda x: x == expected].index) if not monthly_frame.empty else set()
    solver_ok = set(solver_frame.groupby('base_radius').size().loc[lambda x: x == expected].index) if not solver_frame.empty else set()
    return monthly_ok & solver_ok

completed_bases = _completed_useful_bases(useful_monthly, useful_solver_audit)
for base_radius in USEFUL_BASELINES:
    if float(base_radius) in completed_bases:
        print(f'loaded useful-radius checkpoint base={base_radius:g}')
        continue
    print(f'computing useful-radius checkpoint base={base_radius:g}')
    monthly_parts, solver_parts = [], []
    for strategy in ['static', 'dynamic']:
        monthly_part, solver_part = _run_useful_radius_path(float(base_radius), strategy)
        monthly_parts.append(monthly_part); solver_parts.append(solver_part)
    useful_monthly = pd.concat([useful_monthly, *monthly_parts], ignore_index=True)
    useful_solver_audit = pd.concat([useful_solver_audit, *solver_parts], ignore_index=True)
    useful_monthly = (useful_monthly.drop_duplicates(['base_radius', 'strategy', 'formation_month'], keep='last')
                      .sort_values(['base_radius', 'strategy', 'formation_month']).reset_index(drop=True))
    useful_solver_audit = (useful_solver_audit.drop_duplicates(['base_radius', 'strategy', 'formation_month'], keep='last')
                           .sort_values(['base_radius', 'strategy', 'formation_month']).reset_index(drop=True))
    _atomic_df(useful_monthly, USEFUL_CACHE); _atomic_df(useful_solver_audit, USEFUL_SOLVER_CACHE)

expected_rows = len(USEFUL_BASELINES) * 2 * len(realised_ledger)
assert len(useful_monthly) == expected_rows and len(useful_solver_audit) == expected_rows
assert useful_monthly.groupby(['base_radius', 'strategy']).size().eq(len(realised_ledger)).all()
assert useful_solver_audit['solver_status'].isin(['optimal', 'optimal_inaccurate']).all()
assert np.isfinite(useful_monthly[['gross_return', 'net_return', 'turnover', 'weight_l2']]).all().all()
_atomic_df(useful_monthly, USEFUL_MONTHLY_OUT)
_atomic_df(useful_solver_audit, USEFUL_SOLVER_OUT)

def _useful_sharpe(values):
    values = pd.Series(values)
    assert len(values) == len(realised_ledger) and values.std(ddof=1) > 0
    return float(values.mean() / values.std(ddof=1) * np.sqrt(12.0))

summary_rows = []
for base_radius, frame in useful_monthly.groupby('base_radius', sort=True):
    static = frame[frame['strategy'].eq('static')].sort_values('return_date')
    dynamic = frame[frame['strategy'].eq('dynamic')].sort_values('return_date')
    assert np.array_equal(static['return_date'].to_numpy(), dynamic['return_date'].to_numpy())
    summary_rows.append({
        'base_radius': float(base_radius),
        'delta_sharpe_net': _useful_sharpe(dynamic['net_return']) - _useful_sharpe(static['net_return']),
        'delta_sharpe_gross': _useful_sharpe(dynamic['gross_return']) - _useful_sharpe(static['gross_return']),
        'static_weight_l2': float(static['weight_l2'].mean()),
        'dynamic_weight_l2': float(dynamic['weight_l2'].mean()),
        'static_turnover': float(static['turnover'].mean()),
        'dynamic_turnover': float(dynamic['turnover'].mean()),
        'dynamic_clip_fraction': float((dynamic['epsilon'] >= EPS_CAP - 1e-12).mean()),
        'engine_digest': USEFUL_DIGEST,
    })
useful_summary = pd.DataFrame(summary_rows).sort_values('base_radius').reset_index(drop=True)
assert len(useful_summary) == len(USEFUL_BASELINES) and useful_summary.notna().all().all()
# Non-regression guard: the exact EK grid point must reproduce the locked reference pair.
_ek_row = useful_summary[np.isclose(useful_summary['base_radius'], EPS_EK, rtol=0.0, atol=1e-12)]
assert len(_ek_row) == 1
_main_big = portfolio_monthly.sort_values(['strategy', 'return_date'])
_main_static = _main_big[_main_big['strategy'].eq('static')]
_main_dynamic = _main_big[_main_big['strategy'].eq('dynamic')]
_main_delta_net = _useful_sharpe(_main_dynamic['net_return']) - _useful_sharpe(_main_static['net_return'])
_main_delta_gross = _useful_sharpe(_main_dynamic['gross_return']) - _useful_sharpe(_main_static['gross_return'])
assert abs(float(_ek_row.iloc[0]['delta_sharpe_net']) - _main_delta_net) < 5e-7
assert abs(float(_ek_row.iloc[0]['delta_sharpe_gross']) - _main_delta_gross) < 5e-7
_atomic_df(useful_summary, USEFUL_SUMMARY_OUT)

fig, ax = plt.subplots(figsize=(9.0, 4.6))
ax.plot(useful_summary['base_radius'], useful_summary['delta_sharpe_net'], 'o-',
        color=_RUST, lw=1.8, ms=5, label=r'Net $\Delta\mathrm{SR}$')
ax.plot(useful_summary['base_radius'], useful_summary['delta_sharpe_gross'], 'o--',
        color=_RUST, lw=1.0, ms=3.5, alpha=0.45, label=r'Gross $\Delta\mathrm{SR}$ (audit)')
ax.axhline(0.0, color=_GREY, lw=0.8)
ax.axvline(EPS_EK, color=_GREEN, lw=1.2, ls='--', label=rf'$\varepsilon_0={EPS_EK:.2f}$')
ax.set_xscale('log')
from matplotlib.ticker import FixedLocator, FixedFormatter
_useful_ticks = [0.1, 0.2, 0.3, 0.5, 1.0, 2.0]
ax.xaxis.set_major_locator(FixedLocator(_useful_ticks))
ax.xaxis.set_major_formatter(FixedFormatter([f'{tick:g}' for tick in _useful_ticks]))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.set_xlabel(r'Baseline ambiguity radius $\bar\varepsilon$ (log scale)')
ax.set_ylabel(r'Timing gain $\Delta\mathrm{SR}$')
ax.set_title('Timing gain across the baseline-radius continuum')
ax2 = ax.twinx()
ax2.plot(useful_summary['base_radius'], useful_summary['static_weight_l2'], 's--',
         color=_NAVY, lw=1.2, ms=4, alpha=0.70, label=r'Mean static $\|w\|_2$')
ax2.axhline(0.1, color=_NAVY, lw=0.8, ls=':', alpha=0.60)
ax2.annotate('EW limit', (USEFUL_BASELINES.min() * 1.15, 0.1),
             xytext=(0, 5), textcoords='offset points', ha='left', va='bottom',
             color=_NAVY, fontsize=8)
ax2.set_ylabel(r'Mean static concentration $\|w\|_2$', color=_NAVY)
ax2.tick_params(axis='y', colors=_NAVY); ax2.grid(False)
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color(_NAVY); ax2.spines['right'].set_linewidth(0.8)
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles1 + handles2, labels1 + labels2, loc='best', fontsize=8.8)
fig.tight_layout(); save_fig(fig, 10, 'useful_window'); plt.show()

print('=' * 76)
print('  Useful radius window — exact accounting, net primary')
print('=' * 76)
display(useful_summary)
print('saved  images/R_P_10_useful_window.pdf')
print(f'saved  audit/{USEFUL_MONTHLY_OUT.name}')
print(f'saved  audit/{USEFUL_SUMMARY_OUT.name}')
print(f'saved  audit/{USEFUL_SOLVER_OUT.name}')


### 6.2.9.2 Robustness to construction choices  [FIG P_11 + TABLE P_09]

The reference specification is varied one lever at a time while all remaining elements of the portfolio engine are held fixed. Four pre-declared dimensions are considered: the signal-estimation window $W_\rho\in\{36,48,60\}$ months, the point-in-time modulation map, the rebalancing interval, and the Esfahani--Kuhn confidence parameter $\beta\in\{0.05,0.10,0.20\}$. Figure P_11 reports the net-of-cost timing gain for every setting in four aligned panels; the reference setting is identified consistently rather than selected from the displayed alternatives. Table P_09 retains the corresponding gross result, performance metrics, turnover, clipping frequency, and solver audit.

All signal transformations are strictly point in time. The identity map divides current $\sqrt{\rho}$ by its expanding mean. The exponential map standardises current $\sqrt{\rho}$ using only observations available through the formation month and exponentiates the resulting expanding $z$-score; its initialisation rule is fixed before evaluation. The rank map uses the expanding percentile rank of the current observation among the history available at that date, followed by the same expanding-mean normalisation. No full-sample mean, volatility, or rank enters a portfolio decision. The 48- and 60-month signals are loaded only from the validated signal pipeline and are never recomputed inside the portfolio notebook.

Rebalancing-frequency variants alter trading dates, not the information set. At quarterly or annual frequency, the portfolio is solved only on scheduled formation months. Between those dates, risky positions drift with realised returns, delisting proceeds remain in cash, and no asset is inserted merely because it enters a later monthly universe. At the next scheduled rebalance, the current point-in-time universe and estimation matrix determine the new target. This buy-and-hold convention avoids the implicit monthly reconstitution and vector-length fallback present in the exploratory reference implementation. Extending the holding calendar beyond monthly memberships exposes 24 non-delisting daily return gaps in asset-months actually held. They are recorded in an explicit allowlist and assigned a zero return for that isolated day; no other missing holding return is tolerated.

The comparison is descriptive robustness analysis rather than another model-selection exercise. Positivity across settings would indicate that the timing result is not tied to a single discretionary choice; dispersion across settings remains economically informative and is reported without selecting the best-performing configuration.


In [ ]:
# FIG P_11 + TABLE P_09 — one-at-a-time PIT robustness, exact buy-and-hold accounting.
ROBUST_SIGNAL_PATHS = {36: SIG/'V_uni.parquet', 48: SIG/'V_uni_W48.parquet', 60: SIG/'V_uni_W60.parquet'}
for _p in ROBUST_SIGNAL_PATHS.values(): assert _p.exists(), f'missing validated signal: {_p}'
_fm = pd.DatetimeIndex([e['formation_month'] for e in realised_ledger])
def _sqrt_signal(path):
    s = pd.read_parquet(path).set_index('date')['rho'].sort_index(); s.index = pd.to_datetime(s.index)+pd.offsets.MonthEnd(0)
    out = np.array([math.sqrt(float(s.loc[d+pd.offsets.MonthEnd(0)])) for d in _fm])
    assert np.isfinite(out).all() and (out>0).all(); return out
def _mod_identity(x): return x/pd.Series(x).expanding().mean().to_numpy()
def _mod_expanding_z(x, init=12):
    z=np.zeros(len(x)); sx=pd.Series(x)
    mu=sx.expanding().mean(); sd=sx.expanding(min_periods=init).std(ddof=1)
    ok=sd.notna() & sd.gt(0); z[ok.to_numpy()]=((sx[ok]-mu[ok])/sd[ok]).to_numpy()
    g=np.exp(z); return g/pd.Series(g).expanding().mean().to_numpy()
def _mod_expanding_rank(x):
    g=np.array([sps.rankdata(x[:i+1],method='average')[-1]/(i+1) for i in range(len(x))])
    return g/pd.Series(g).expanding().mean().to_numpy()
_sqrt_by_W={w:_sqrt_signal(p) for w,p in ROBUST_SIGNAL_PATHS.items()}
MODS={'w36_id':_mod_identity(_sqrt_by_W[36]),'w48_id':_mod_identity(_sqrt_by_W[48]),
      'w60_id':_mod_identity(_sqrt_by_W[60]),'w36_expz':_mod_expanding_z(_sqrt_by_W[36]),
      'w36_rank':_mod_expanding_rank(_sqrt_by_W[36])}
assert all(np.isfinite(v).all() and (v>0).all() for v in MODS.values())

# Monthly factors only for asset-months genuinely held under the three schedules.
ROBUST_ALLOWED_NULL_RETURNS={(11042,d) for d in ['2002-06-26','2002-06-27','2002-06-28']}
ROBUST_ALLOWED_NULL_RETURNS|={(76614,'2015-06-09')}
_required_pairs=set()
for _r in [1,3,12]:
    for _i,_entry in enumerate(realised_ledger):
        _source=realised_ledger[_i-_i%_r]; _H=_entry['holding_month']+pd.offsets.MonthEnd(0)
        _required_pairs|={(int(a),_H) for a in _source['assets']}
_all_hold=raw[['PERMNO','date','DlyRet','is_delisting_event']].copy(); _all_hold['PERMNO']=_all_hold['PERMNO'].astype(int)
_all_hold['month_end']=_all_hold['date'].dt.to_period('M').dt.to_timestamp('M')
_all_hold=_all_hold[[(a,m) in _required_pairs for a,m in zip(_all_hold.PERMNO,_all_hold.month_end)]].copy()
_null=_all_hold['DlyRet'].isna(); _pairs=set(zip(_all_hold.loc[_null,'PERMNO'],_all_hold.loc[_null,'date'].dt.strftime('%Y-%m-%d')))
assert _pairs==ROBUST_ALLOWED_NULL_RETURNS,f'robustness held-day null mismatch: {_pairs^ROBUST_ALLOWED_NULL_RETURNS}'
assert len(_pairs)==len(ROBUST_ALLOWED_NULL_RETURNS) and not _all_hold.loc[_null,'is_delisting_event'].any()
_all_hold.loc[_null,'DlyRet']=0.0
_all_factors=(_all_hold.groupby(['PERMNO','month_end']).agg(hold_factor=('DlyRet',lambda x:float((1+x).prod())),
    delisted=('is_delisting_event','any')).reset_index().set_index(['PERMNO','month_end']))

def _eps_beta(beta): return float(np.clip(math.sqrt(2*math.log(1/beta)/W_EST),EPS_FLOOR,EPS_CAP))
PATH_SPECS={}; CONFIGS=[]
def _path(dynamic,mod='w36_id',beta=.10,rebal=1):
    key=f'{"dyn" if dynamic else "stat"}_{mod if dynamic else "constant"}_b{beta:.2f}_r{rebal}'
    PATH_SPECS[key]={'dynamic':dynamic,'mod':mod,'beta':beta,'rebal':rebal}; return key
def _cfg(lever,value,mod='w36_id',beta=.10,rebal=1):
    CONFIGS.append({'lever':lever,'value':value,'dynamic_path':_path(True,mod,beta,rebal),
                    'static_path':_path(False,mod,beta,rebal)})
for w in [36,48,60]: _cfg(r'$W_\rho$',str(w),f'w{w}_id')
for value,mod in [('identity','w36_id'),(r'$\exp(z_\tau)$','w36_expz'),('rank','w36_rank')]: _cfg('Form',value,mod)
for value,r in [('monthly',1),('quarterly',3),('annual',12)]: _cfg('Rebalancing',value,rebal=r)
for b in [.05,.10,.20]: _cfg(r'$\beta$',f'{b:.2f}',beta=b)
_robust_contract={'engine':'portfolio-robustness-v1.0.0','base_engine_digest':ENGINE_DIGEST,
    'signals':{w:_file_signature(p) for w,p in ROBUST_SIGNAL_PATHS.items()},'specs':PATH_SPECS,
    'transforms':'PIT expanding; expz neutral first 12','accounting':'exact scheduled rebalance'}
ROBUST_DIGEST=hashlib.sha256(json.dumps(_robust_contract,sort_keys=True).encode()).hexdigest()
ROBUST_CACHE=CACHE/f'robustness_paths_{ROBUST_DIGEST[:16]}.parquet'; ROBUST_SOLVER_CACHE=CACHE/f'robustness_solver_{ROBUST_DIGEST[:16]}.parquet'
ROBUST_MONTHLY_OUT=DATA_OUT/'P_09_robustness_monthly.parquet'; ROBUST_METRICS_OUT=DATA_OUT/'P_09_robustness_metrics.parquet'

def _run_robust_path(path_id,spec):
    pre_risky={}; pre_cash=1.0; rows=[]; audits=[]; modulation=MODS[spec['mod']] if spec['dynamic'] else np.ones(len(_fm))
    for i,e in enumerate(realised_ledger):
        fm=pd.Timestamp(e['formation_month']); rebalance=(i%spec['rebal']==0)
        if rebalance:
            eps=float(np.clip(_eps_beta(spec['beta'])*modulation[i],EPS_FLOOR,EPS_CAP)); weights,status=w_dro(e['est'],eps)
            assert weights is not None,f'{path_id} {fm:%Y-%m}: {status}'
            target=dict(zip(map(int,e['assets']),map(float,weights))); target_cash=0.0
            turnover=float(sum(abs(target.get(a,0)-pre_risky.get(a,0)) for a in set(target)|set(pre_risky)))
            audits.append({'path_id':path_id,'formation_month':fm,'epsilon':eps,'solver_status':status,
                           'weight_sum':weights.sum(),'weight_l2':np.linalg.norm(weights),'engine_digest':ROBUST_DIGEST})
        else:
            target=pre_risky.copy(); target_cash=pre_cash; turnover=0.0; eps=np.nan
        H=e['holding_month']+pd.offsets.MonthEnd(0); terminal={}; delisted={}
        for a,w in target.items():
            assert (a,H) in _all_factors.index,f'missing buy-hold leg {(a,H)}'
            leg=_all_factors.loc[(a,H)]; terminal[a]=w*float(leg['hold_factor']); delisted[a]=bool(leg['delisted'])
        gross_factor=float(target_cash+sum(terminal.values())); assert gross_factor>0
        gross_return=gross_factor-1; pre_risky={a:v/gross_factor for a,v in terminal.items() if not delisted[a] and v>0}
        pre_cash=float((target_cash+sum(v for a,v in terminal.items() if delisted[a]))/gross_factor)
        assert abs(sum(pre_risky.values())+pre_cash-1)<1e-10
        tc=cost_bps(e['formation_date'])/1e4*turnover
        rows.append({'path_id':path_id,'formation_month':fm,'return_date':H,'rebalanced':rebalance,'epsilon':eps,
            'gross_return':gross_return,'net_return':gross_return-tc,'turnover':turnover,'transaction_cost':tc,
            'end_cash_weight':pre_cash,'engine_digest':ROBUST_DIGEST})
    return pd.DataFrame(rows),pd.DataFrame(audits)

robust_paths=pd.read_parquet(ROBUST_CACHE) if ROBUST_CACHE.exists() else pd.DataFrame()
robust_solver=pd.read_parquet(ROBUST_SOLVER_CACHE) if ROBUST_SOLVER_CACHE.exists() else pd.DataFrame()
for frame in [robust_paths,robust_solver]:
    if not frame.empty: assert frame['engine_digest'].eq(ROBUST_DIGEST).all(),'incompatible robustness cache'
done=set(robust_paths.groupby('path_id').size().loc[lambda x:x==len(realised_ledger)].index) if not robust_paths.empty else set()
for path_id,spec in PATH_SPECS.items():
    if path_id in done: print('loaded',path_id); continue
    print('computing',path_id); part,audit=_run_robust_path(path_id,spec)
    robust_paths=pd.concat([robust_paths,part],ignore_index=True).drop_duplicates(['path_id','formation_month'],keep='last')
    robust_solver=pd.concat([robust_solver,audit],ignore_index=True).drop_duplicates(['path_id','formation_month'],keep='last')
    _atomic_df(robust_paths,ROBUST_CACHE); _atomic_df(robust_solver,ROBUST_SOLVER_CACHE)
assert robust_paths.groupby('path_id').size().eq(372).all() and set(robust_paths.path_id)==set(PATH_SPECS)
assert robust_solver.solver_status.isin(['optimal','optimal_inaccurate']).all()
_atomic_df(robust_paths,ROBUST_MONTHLY_OUT); _atomic_df(robust_solver,DATA_OUT/'P_09_robustness_solver_audit.parquet')

metric_rows=[]
for cfg in CONFIGS:
    d=robust_paths[robust_paths.path_id.eq(cfg['dynamic_path'])].sort_values('return_date'); s=robust_paths[robust_paths.path_id.eq(cfg['static_path'])].sort_values('return_date')
    assert np.array_equal(d.return_date.to_numpy(),s.return_date.to_numpy())
    _idx=pd.DatetimeIndex(d.return_date); dn=pd.Series(d.net_return.to_numpy(),index=_idx); sn=pd.Series(s.net_return.to_numpy(),index=_idx)
    dg=pd.Series(d.gross_return.to_numpy(),index=_idx); sg=pd.Series(s.gross_return.to_numpy(),index=_idx)
    mdn=portfolio_metrics(dn,d.turnover); msn=portfolio_metrics(sn,s.turnover)
    mdg=portfolio_metrics(dg,d.turnover); msg=portfolio_metrics(sg,s.turnover)
    metric_rows.append({**cfg,'delta_sharpe_net':mdn['Sharpe']-msn['Sharpe'],'delta_sharpe_gross':mdg['Sharpe']-msg['Sharpe'],
        'dynamic_sharpe_net':mdn['Sharpe'],'annual_return_net':mdn['annual_return'],'volatility_net':mdn['volatility'],
        'MaxDD_net':mdn['MaxDD'],'turnover':mdn['turnover'],'clip_fraction':float(d.loc[d.rebalanced,'epsilon'].ge(EPS_CAP-1e-12).mean()),
        'engine_digest':ROBUST_DIGEST})
robust_metrics=pd.DataFrame(metric_rows)
_ref=robust_metrics[(robust_metrics.lever==r'$W_\rho$') & (robust_metrics.value=='36')]
assert len(_ref)==1 and abs(float(_ref.iloc[0].delta_sharpe_net)-delta_sharpe_net)<5e-7
assert abs(float(_ref.iloc[0].delta_sharpe_gross)-delta_sharpe_gross)<5e-7
_atomic_df(robust_metrics,ROBUST_METRICS_OUT)

fig,axes=plt.subplots(1,4,figsize=(11.5,3.5),sharey=True); _order=[r'$W_\rho$','Form','Rebalancing',r'$\beta$']
for ax,lever in zip(axes,_order):
    q=robust_metrics[robust_metrics.lever.eq(lever)]; x=np.arange(len(q)); vals=q.delta_sharpe_net.to_numpy()
    ax.plot(x,vals,color=_GREY,lw=.9,alpha=.98); ax.scatter(x,vals,s=58,color=[_GREEN if v>0 else _RUST for v in vals],zorder=3,edgecolor='white',lw=.7)
    ax.axhline(0, color=_RUST, lw=1, ls='--', alpha=.65)
    ax.set_xticks(x, q.value, fontsize=8)
    ax.margins(x=0.10)
    for xi,v in zip(x,vals): ax.annotate(f'{v:+.4f}',(xi,v),xytext=(0,8),textcoords='offset points',ha='center',fontsize=7.5)
    ax.set_title(lever)
    ax.grid(axis='x', alpha=0)
    ax.grid(axis='y', color='#b8b8b8', alpha=0.55, linewidth=0.35)
axes[0].set_ylim(top=0.075)
axes[0].set_ylabel(r'Net $\Delta\mathrm{SR}$')
fig.suptitle('Robustness of the timing gain to construction choices', y=1.03)
fig.tight_layout(); save_fig(fig,11,'robustness_levers'); plt.show()

_rows=[f'{r.lever} & {r.value} & {r.delta_sharpe_net:+.4f} & {r.delta_sharpe_gross:+.4f} & {r.dynamic_sharpe_net:.3f} & {r.turnover:.4f} & {100*r.clip_fraction:.1f} ' + r'\\' for r in robust_metrics.itertuples()]
_tex='\n'.join([r'\begin{tabular}{llrrrrr}',r'\toprule',r'Lever & Value & $\Delta\mathrm{SR}_{\mathrm{net}}$ & $\Delta\mathrm{SR}_{\mathrm{gross}}$ & $\mathrm{SR}_{\mathrm{dyn,net}}$ & Turnover & Clipped (\%) \\',r'\midrule',*_rows,r'\bottomrule',r'\end{tabular}'])+'\n'
ROBUST_TABLE=TAB/'P_09_robustness_metrics.tex'; _tmp=ROBUST_TABLE.with_suffix('.tex.tmp'); _tmp.write_text(_tex); _tmp.replace(ROBUST_TABLE)
display(robust_metrics); print('saved  images/R_P_11_robustness_levers.pdf'); print(f'saved  tables/{ROBUST_TABLE.name}')


### 6.2.9.3 Volatility-timing benchmarks  [TABLE P_10]

In [ ]:
# TABLE P_10 — implied- and realized-volatility timing benchmarks under the exact PIT engine.
VIX_PATH = DATA / 'processed' / 'vix_1990.parquet'
for _required in (VIX_PATH, SMALL_PANEL):
    assert _required.exists(), f'missing volatility-benchmark input: {_required}'
for _name in ('portfolio_monthly', 'small_monthly', 'small_ledger', 'small_holding'):
    assert _name in globals(), f'run Section 6.2.8 before this cell: missing {_name}'

RV_WINDOW_MONTHS = 36
VOL_STRATEGIES = ('iv', 'rv36')
VOL_UNIVERSE_ORDER = ('big_caps', 'small_caps_p20_p50')

# Realised ledgers only: formation M produces the observed holding return in M+1.
_small_realised_months = set(pd.to_datetime(small_holding['formation_month']))
small_realised_ledger = [
    e for e in small_ledger
    if pd.Timestamp(e['formation_month']) in _small_realised_months
]
assert len(realised_ledger) == len(small_realised_ledger) == 372
assert [pd.Timestamp(e['formation_month']) for e in realised_ledger] == [
    pd.Timestamp(e['formation_month']) for e in small_realised_ledger
]
_benchmark_formation_months = pd.DatetimeIndex(
    [pd.Timestamp(e['formation_month']) for e in realised_ledger]
)
_benchmark_signal_dates = _benchmark_formation_months + pd.offsets.MonthEnd(0)


def _positive_expanding_modulation(values):
    """Positive PIT level divided by its expanding mean, including the current formation month."""
    series = pd.Series(np.asarray(values, dtype=float))
    assert len(series) == 372 and np.isfinite(series).all() and series.gt(0).all()
    expanding_mean = series.expanding(min_periods=1).mean()
    modulation = series / expanding_mean
    assert np.isfinite(modulation).all() and modulation.gt(0).all()
    return expanding_mean.to_numpy(float), modulation.to_numpy(float)


# IV benchmark: mean daily VIX observed during formation month M, known at the close of M.
_vix = pd.read_parquet(VIX_PATH).copy()
assert {'date', 'vix'}.issubset(_vix.columns)
_vix['date'] = pd.to_datetime(_vix['date'])
_vix_monthly = _vix.set_index('date')['vix'].sort_index().resample('ME').mean()
_iv_levels = np.asarray([float(_vix_monthly.loc[d]) for d in _benchmark_signal_dates])
_iv_expanding_mean, _iv_modulation = _positive_expanding_modulation(_iv_levels)


def _rv36_from_daily_pivot(daily_pivot, ledger_u, universe):
    """Annualised RV of the EW return for the exact 100-member, 36-month PIT formation window."""
    levels, rows = [], []
    for e in ledger_u:
        fm = pd.Timestamp(e['formation_month'])
        # Include every trading day in the first of the 36 formation months.
        start = (fm.to_period('M') - (RV_WINDOW_MONTHS - 1)).start_time
        stop = pd.Timestamp(e['formation_date'])
        assets = list(map(int, e['assets']))
        window = daily_pivot.loc[
            (daily_pivot.index >= start) & (daily_pivot.index <= stop),
            assets,
        ]
        assert window.shape[1] == 100
        assert 700 <= window.shape[0] <= 800, (
            f'{universe} {fm:%Y-%m}: unexpected daily count {window.shape[0]}'
        )
        assert window.notna().all().all(), (
            f'{universe} {fm:%Y-%m}: RV36 window contains missing returns'
        )
        ew_daily = window.mean(axis=1)
        value = float(ew_daily.std(ddof=1) * math.sqrt(252.0))
        assert np.isfinite(value) and value > 0.0
        levels.append(value)
        rows.append({
            'universe': universe,
            'formation_month': fm,
            'signal': 'rv36',
            'signal_level': value,
            'n_daily_observations': int(window.shape[0]),
            'n_assets': int(window.shape[1]),
        })
    return np.asarray(levels, dtype=float), pd.DataFrame(rows)


# Big-cap RV reuses the daily pivot already loaded by the reference engine.
_big_rv_levels, _big_rv_audit = _rv36_from_daily_pivot(
    pivot_d, realised_ledger, 'big_caps'
)

# Small-cap RV is built once from the clean PIT panel, selecting only required columns.
_small_daily = (
    pl.scan_parquet(SMALL_PANEL)
    .select(['DlyCalDt', 'PERMNO', 'DlyRet'])
    .collect(engine='streaming')
    .to_pandas()
)
_small_daily['date'] = pd.to_datetime(_small_daily['DlyCalDt'])
_small_daily['PERMNO'] = _small_daily['PERMNO'].astype(int)
_small_pivot_d = (
    _small_daily.drop_duplicates(['date', 'PERMNO'])
    .pivot(index='date', columns='PERMNO', values='DlyRet')
    .sort_index()
)
_small_pivot_d.columns = _small_pivot_d.columns.astype(int)
_small_rv_levels, _small_rv_audit = _rv36_from_daily_pivot(
    _small_pivot_d, small_realised_ledger, 'small_caps_p20_p50'
)
del _small_daily, _small_pivot_d

_rv_levels_by_universe = {
    'big_caps': _big_rv_levels,
    'small_caps_p20_p50': _small_rv_levels,
}
_ledger_by_universe = {
    'big_caps': realised_ledger,
    'small_caps_p20_p50': small_realised_ledger,
}
_holding_by_universe = {
    'big_caps': holding,
    'small_caps_p20_p50': small_holding,
}

# Signed PIT modulation panel. IV is common; RV36 is universe-specific.
_vol_signal_rows = []
for _universe in VOL_UNIVERSE_ORDER:
    _rv_mean, _rv_mod = _positive_expanding_modulation(
        _rv_levels_by_universe[_universe]
    )
    for _i, _fm in enumerate(_benchmark_formation_months):
        for _signal, _level, _mean, _mod in [
            ('iv', _iv_levels[_i], _iv_expanding_mean[_i], _iv_modulation[_i]),
            (
                'rv36',
                _rv_levels_by_universe[_universe][_i],
                _rv_mean[_i],
                _rv_mod[_i],
            ),
        ]:
            _vol_signal_rows.append({
                'universe': _universe,
                'formation_month': _fm,
                'signal_date': _benchmark_signal_dates[_i],
                'strategy': _signal,
                'signal_level': float(_level),
                'expanding_mean': float(_mean),
                'modulation': float(_mod),
                'epsilon': float(np.clip(EPS_EK * _mod, EPS_FLOOR, EPS_CAP)),
            })
volatility_signal_panel = pd.DataFrame(_vol_signal_rows)
assert len(volatility_signal_panel) == 2 * 2 * 372
assert volatility_signal_panel.groupby(['universe', 'strategy']).size().eq(372).all()
assert np.isfinite(
    volatility_signal_panel[['signal_level', 'expanding_mean', 'modulation', 'epsilon']]
).all().all()
_vol_signal_key = volatility_signal_panel.set_index(
    ['universe', 'strategy', 'formation_month']
)

_vol_contract = {
    'engine': 'portfolio-volatility-benchmarks-v1.0.0',
    'base_engine_digest': ENGINE_DIGEST,
    'small_panel_signature': _file_signature(SMALL_PANEL),
    'vix_signature': _file_signature(VIX_PATH),
    'strategies': {
        'iv': 'formation-month mean daily VIX / PIT expanding mean',
        'rv36': 'annualised EW daily RV over exact 100-member 36m PIT window / PIT expanding mean',
    },
    'base_radius': EPS_EK,
    'epsilon_floor': EPS_FLOOR,
    'epsilon_cap': EPS_CAP,
    'accounting': 'monthly rebalance; exact held factors; delisting cash; risky-only turnover',
    'primary_basis': 'net',
}
VOL_DIGEST = hashlib.sha256(
    json.dumps(_vol_contract, sort_keys=True).encode()
).hexdigest()
VOL_CACHE = CACHE / f'volatility_benchmark_paths_{VOL_DIGEST[:16]}.parquet'
VOL_SOLVER_CACHE = CACHE / f'volatility_benchmark_solver_{VOL_DIGEST[:16]}.parquet'
VOL_MONTHLY_OUT = DATA_OUT / 'P_10_volatility_benchmark_monthly.parquet'
VOL_SIGNAL_OUT = DATA_OUT / 'P_10_volatility_benchmark_signals.parquet'
VOL_METRICS_OUT = DATA_OUT / 'P_10_volatility_benchmark_metrics.parquet'
VOL_INFERENCE_OUT = DATA_OUT / 'P_10_volatility_benchmark_inference.parquet'


def _run_volatility_benchmark_path(universe, strategy):
    """Solve and account one monthly volatility-timing path with no fallback."""
    ledger_u = _ledger_by_universe[universe]
    holding_u = _holding_by_universe[universe]
    holding_groups = {
        pd.Timestamp(fm): grp.set_index('PERMNO')
        for fm, grp in holding_u.groupby('formation_month', sort=True)
    }
    pre_risky, pre_cash = {}, 1.0
    rows, audits = [], []
    for _ordinal, e in enumerate(ledger_u, start=1):
        fm = pd.Timestamp(e['formation_month'])
        timing = _vol_signal_key.loc[(universe, strategy, fm)]
        epsilon = float(timing['epsilon'])
        weights, status = w_dro(e['est'], epsilon)
        assert weights is not None, (
            f'{universe} {strategy} {fm:%Y-%m}: solver failed ({status})'
        )
        assert len(weights) == 100 and np.isfinite(weights).all()
        assert (weights >= -1e-12).all() and abs(weights.sum() - 1.0) < 1e-10
        target = dict(zip(map(int, e['assets']), map(float, weights)))
        turnover = float(sum(
            abs(target.get(asset, 0.0) - pre_risky.get(asset, 0.0))
            for asset in set(target) | set(pre_risky)
        ))

        h = holding_groups[fm]
        assert set(h.index.astype(int)) == set(target)
        terminal = {
            asset: target[asset] * float(h.loc[asset, 'hold_factor'])
            for asset in target
        }
        gross_factor = float(sum(terminal.values()))
        assert gross_factor > 0.0
        gross_return = gross_factor - 1.0
        weighted_return = float(sum(
            target[asset] * (float(h.loc[asset, 'hold_factor']) - 1.0)
            for asset in target
        ))
        assert abs(gross_return - weighted_return) < 1e-12

        pre_risky = {
            asset: value / gross_factor
            for asset, value in terminal.items()
            if not bool(h.loc[asset, 'delisted']) and value > 0.0
        }
        pre_cash = float(sum(
            value for asset, value in terminal.items()
            if bool(h.loc[asset, 'delisted'])
        ) / gross_factor)
        assert abs(sum(pre_risky.values()) + pre_cash - 1.0) < 1e-10
        transaction_cost = cost_bps(e['formation_date']) / 1e4 * turnover

        rows.append({
            'universe': universe,
            'strategy': strategy,
            'formation_month': fm,
            'return_date': pd.Timestamp(e['holding_month']) + pd.offsets.MonthEnd(0),
            'signal_level': float(timing['signal_level']),
            'modulation': float(timing['modulation']),
            'epsilon': epsilon,
            'gross_return': gross_return,
            'net_return': gross_return - transaction_cost,
            'turnover': turnover,
            'transaction_cost': transaction_cost,
            'end_cash_weight': pre_cash,
            'n_delisted': int(h['delisted'].sum()),
            'weight_l2': float(np.linalg.norm(weights)),
            'engine_digest': VOL_DIGEST,
        })
        audits.append({
            'universe': universe,
            'strategy': strategy,
            'formation_month': fm,
            'epsilon': epsilon,
            'solver_status': status,
            'weight_sum': float(weights.sum()),
            'weight_min': float(weights.min()),
            'weight_max': float(weights.max()),
            'weight_l2': float(np.linalg.norm(weights)),
            'engine_digest': VOL_DIGEST,
        })
        if _ordinal % 60 == 0 or _ordinal == len(ledger_u):
            print(
                f'  {universe} {strategy}: '
                f'{_ordinal}/{len(ledger_u)} formations'
            )
    return pd.DataFrame(rows), pd.DataFrame(audits)


def _load_volatility_cache(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_parquet(path)
    assert not frame.empty and 'engine_digest' in frame
    assert frame['engine_digest'].eq(VOL_DIGEST).all(), (
        f'incompatible volatility-benchmark cache: {path}'
    )
    return frame


volatility_paths = _load_volatility_cache(VOL_CACHE)
volatility_solver_audit = _load_volatility_cache(VOL_SOLVER_CACHE)
_monthly_done = (
    set(
        volatility_paths.groupby(['universe', 'strategy']).size()
        .loc[lambda x: x.eq(372)].index
    )
    if not volatility_paths.empty else set()
)
_solver_done = (
    set(
        volatility_solver_audit.groupby(['universe', 'strategy']).size()
        .loc[lambda x: x.eq(372)].index
    )
    if not volatility_solver_audit.empty else set()
)
_completed_vol_paths = _monthly_done & _solver_done

for _universe in VOL_UNIVERSE_ORDER:
    for _strategy in VOL_STRATEGIES:
        _path_key = (_universe, _strategy)
        if _path_key in _completed_vol_paths:
            print(f'loaded volatility checkpoint: {_universe} {_strategy}')
            continue
        print(f'computing volatility checkpoint: {_universe} {_strategy}')
        _part, _audit = _run_volatility_benchmark_path(_universe, _strategy)
        volatility_paths = pd.concat(
            [volatility_paths, _part], ignore_index=True
        ).drop_duplicates(
            ['universe', 'strategy', 'formation_month'], keep='last'
        )
        volatility_solver_audit = pd.concat(
            [volatility_solver_audit, _audit], ignore_index=True
        ).drop_duplicates(
            ['universe', 'strategy', 'formation_month'], keep='last'
        )
        _atomic_df(
            volatility_paths.sort_values(
                ['universe', 'strategy', 'formation_month']
            ).reset_index(drop=True),
            VOL_CACHE,
        )
        _atomic_df(
            volatility_solver_audit.sort_values(
                ['universe', 'strategy', 'formation_month']
            ).reset_index(drop=True),
            VOL_SOLVER_CACHE,
        )

assert len(volatility_paths) == 2 * 2 * 372
assert volatility_paths.groupby(['universe', 'strategy']).size().eq(372).all()
assert volatility_solver_audit.groupby(['universe', 'strategy']).size().eq(372).all()
assert volatility_solver_audit['solver_status'].isin(
    ['optimal', 'optimal_inaccurate']
).all()
assert np.isfinite(
    volatility_paths[['gross_return', 'net_return', 'turnover', 'epsilon']]
).all().all()

# Append the already locked static and rho paths; they are not re-solved.
_reference = universe_monthly[
    [
        'universe',
        'formation_month',
        'return_date',
        'strategy',
        'gross_return',
        'net_return',
        'turnover',
        'transaction_cost',
    ]
].copy()

_reference['strategy'] = _reference['strategy'].replace(
    {'dynamic': 'rho'}
)
_reference['engine_digest'] = VOL_DIGEST
_reference['strategy'] = _reference['strategy'].replace({'dynamic': 'rho'})
_reference['engine_digest'] = VOL_DIGEST
volatility_benchmark_monthly = pd.concat(
    [
        _reference,
        volatility_paths[
            ['universe', 'formation_month', 'return_date', 'strategy',
             'gross_return', 'net_return', 'turnover', 'transaction_cost',
             'engine_digest']
        ],
    ],
    ignore_index=True,
).sort_values(
    ['universe', 'strategy', 'return_date']
).reset_index(drop=True)
assert set(volatility_benchmark_monthly['strategy']) == {
    'static', 'rho', 'iv', 'rv36'
}
assert volatility_benchmark_monthly.groupby(
    ['universe', 'strategy']
).size().eq(372).all()
assert volatility_benchmark_monthly.groupby(
    ['universe', 'strategy']
)['return_date'].nunique().eq(372).all()

# Performance table on gross and net returns.
_vol_metric_rows = []
for _universe in VOL_UNIVERSE_ORDER:
    _uf = volatility_benchmark_monthly[
        volatility_benchmark_monthly['universe'].eq(_universe)
    ]
    for _basis, _return_col in [('net', 'net_return'), ('gross', 'gross_return')]:
        _static = _uf[_uf['strategy'].eq('static')].sort_values('return_date')
        _rho_path = _uf[_uf['strategy'].eq('rho')].sort_values('return_date')
        _static_metrics = portfolio_metrics(
            _static[_return_col], _static['turnover']
        )
        _rho_metrics = portfolio_metrics(
            _rho_path[_return_col], _rho_path['turnover']
        )
        for _strategy in ('static', 'rho', 'iv', 'rv36'):
            _path = _uf[_uf['strategy'].eq(_strategy)].sort_values('return_date')
            assert np.array_equal(
                _static['return_date'].to_numpy(),
                _path['return_date'].to_numpy(),
            )
            _metrics = portfolio_metrics(
                _path[_return_col], _path['turnover']
            )
            _vol_metric_rows.append({
                'universe': _universe,
                'basis': _basis,
                'strategy': _strategy,
                'sharpe': _metrics['Sharpe'],
                'delta_sharpe_vs_static': (
                    _metrics['Sharpe'] - _static_metrics['Sharpe']
                ),
                'rho_minus_strategy_sharpe': (
                    _rho_metrics['Sharpe'] - _metrics['Sharpe']
                ),
                'annual_return': _metrics['annual_return'],
                'volatility': _metrics['volatility'],
                'MaxDD': _metrics['MaxDD'],
                'turnover': _metrics['turnover'],
                'engine_digest': VOL_DIGEST,
            })
volatility_benchmark_metrics = pd.DataFrame(_vol_metric_rows)
assert len(volatility_benchmark_metrics) == 2 * 2 * 4
_big_rho_net = volatility_benchmark_metrics[
    volatility_benchmark_metrics['universe'].eq('big_caps')
    & volatility_benchmark_metrics['basis'].eq('net')
    & volatility_benchmark_metrics['strategy'].eq('rho')
]
assert len(_big_rho_net) == 1
assert abs(float(_big_rho_net.iloc[0]['sharpe']) - M_dyn_net['Sharpe']) < 1e-12

# Exploratory paired stationary bootstrap: rho minus each volatility benchmark.
VOL_BOOTSTRAP_DRAWS = 10_000
VOL_BOOTSTRAP_RESTART = 0.10
VOL_BOOTSTRAP_SEED = 20250727
_vol_boot_contract = {
    'volatility_digest': VOL_DIGEST,
    'draws': VOL_BOOTSTRAP_DRAWS,
    'restart_probability': VOL_BOOTSTRAP_RESTART,
    'seed': VOL_BOOTSTRAP_SEED,
    'statistic': 'paired annualised Sharpe(rho)-Sharpe(volatility benchmark), net',
}
VOL_BOOTSTRAP_DIGEST = hashlib.sha256(
    json.dumps(_vol_boot_contract, sort_keys=True).encode()
).hexdigest()
VOL_BOOTSTRAP_CACHE = (
    CACHE / f'volatility_pair_bootstrap_{VOL_BOOTSTRAP_DIGEST[:16]}.parquet'
)


def _vol_annualised_sharpe(values):
    values = np.asarray(values, dtype=float)
    sd = float(np.std(values, ddof=1))
    assert np.isfinite(values).all() and sd > 0.0
    return float(np.mean(values) / sd * math.sqrt(12.0))


if VOL_BOOTSTRAP_CACHE.exists():
    volatility_bootstrap = pd.read_parquet(VOL_BOOTSTRAP_CACHE)
    _vol_boot_ok = (
        len(volatility_bootstrap) == VOL_BOOTSTRAP_DRAWS * 2 * 2
        and volatility_bootstrap['bootstrap_digest'].eq(
            VOL_BOOTSTRAP_DIGEST
        ).all()
    )
else:
    _vol_boot_ok = False

if not _vol_boot_ok:
    _vol_arrays = {}
    for _universe in VOL_UNIVERSE_ORDER:
        _uf = volatility_benchmark_monthly[
            volatility_benchmark_monthly['universe'].eq(_universe)
        ]
        for _strategy in ('rho', 'iv', 'rv36'):
            _path = _uf[_uf['strategy'].eq(_strategy)].sort_values('return_date')
            _vol_arrays[(_universe, _strategy)] = _path['net_return'].to_numpy(float)
    _rng = np.random.default_rng(VOL_BOOTSTRAP_SEED)
    _vol_boot_rows = []
    for _draw in range(VOL_BOOTSTRAP_DRAWS):
        _index = np.empty(372, dtype=int)
        _index[0] = _rng.integers(372)
        for _t in range(1, 372):
            _index[_t] = (
                _rng.integers(372)
                if _rng.random() < VOL_BOOTSTRAP_RESTART
                else (_index[_t - 1] + 1) % 372
            )
        for _universe in VOL_UNIVERSE_ORDER:
            _rho_draw = _vol_annualised_sharpe(
                _vol_arrays[(_universe, 'rho')][_index]
            )
            for _benchmark in VOL_STRATEGIES:
                _benchmark_draw = _vol_annualised_sharpe(
                    _vol_arrays[(_universe, _benchmark)][_index]
                )
                _vol_boot_rows.append({
                    'draw': _draw,
                    'universe': _universe,
                    'benchmark': _benchmark,
                    'rho_minus_benchmark_sharpe': _rho_draw - _benchmark_draw,
                    'bootstrap_digest': VOL_BOOTSTRAP_DIGEST,
                })
    volatility_bootstrap = pd.DataFrame(_vol_boot_rows)
    _atomic_df(volatility_bootstrap, VOL_BOOTSTRAP_CACHE)
    print('volatility-benchmark bootstrap computed and cached')
else:
    print('volatility-benchmark bootstrap loaded from signed cache')

_vol_inference_rows = []
for _universe in VOL_UNIVERSE_ORDER:
    for _benchmark in VOL_STRATEGIES:
        _draws = volatility_bootstrap[
            volatility_bootstrap['universe'].eq(_universe)
            & volatility_bootstrap['benchmark'].eq(_benchmark)
        ]['rho_minus_benchmark_sharpe'].to_numpy(float)
        assert len(_draws) == VOL_BOOTSTRAP_DRAWS
        _observed = float(volatility_benchmark_metrics[
            volatility_benchmark_metrics['universe'].eq(_universe)
            & volatility_benchmark_metrics['basis'].eq('net')
            & volatility_benchmark_metrics['strategy'].eq(_benchmark)
        ]['rho_minus_strategy_sharpe'].iloc[0])
        _low, _high = np.quantile(_draws, [0.025, 0.975])
        _left = (np.sum(_draws <= 0.0) + 1.0) / (len(_draws) + 1.0)
        _right = (np.sum(_draws >= 0.0) + 1.0) / (len(_draws) + 1.0)
        _vol_inference_rows.append({
            'universe': _universe,
            'benchmark': _benchmark,
            'rho_minus_benchmark_sharpe': _observed,
            'ci_low': float(_low),
            'ci_high': float(_high),
            'p_value_unadjusted': float(min(1.0, 2.0 * min(_left, _right))),
            'draws': VOL_BOOTSTRAP_DRAWS,
            'bootstrap_digest': VOL_BOOTSTRAP_DIGEST,
            'engine_digest': VOL_DIGEST,
        })
volatility_benchmark_inference = pd.DataFrame(_vol_inference_rows)
assert len(volatility_benchmark_inference) == 4

# Atomic audit exports.
_atomic_df(
    volatility_signal_panel.sort_values(
        ['universe', 'strategy', 'formation_month']
    ).reset_index(drop=True),
    VOL_SIGNAL_OUT,
)
_atomic_df(volatility_benchmark_monthly, VOL_MONTHLY_OUT)
_atomic_df(volatility_benchmark_metrics, VOL_METRICS_OUT)
_atomic_df(volatility_benchmark_inference, VOL_INFERENCE_OUT)
_atomic_df(
    volatility_solver_audit.sort_values(
        ['universe', 'strategy', 'formation_month']
    ).reset_index(drop=True),
    DATA_OUT / 'P_10_volatility_benchmark_solver_audit.parquet',
)

# Compact article table: net results and paired exploratory inference;
# gross results and turnover remain in the audit parquet.
_universe_labels = {
    'big_caps': 'Big caps',
    'small_caps_p20_p50': r'NYSE small--mid',
}
_strategy_labels = {
    'rho': r'$P^{(\rho)}$',
    'iv': r'$P^{(\mathrm{IV})}$',
    'rv36': r'$P^{(\mathrm{RV}_{36})}$',
}
_table_rows = []
for _universe in VOL_UNIVERSE_ORDER:
    for _strategy in ('rho', 'iv', 'rv36'):
        _metric = volatility_benchmark_metrics[
            volatility_benchmark_metrics['universe'].eq(_universe)
            & volatility_benchmark_metrics['basis'].eq('net')
            & volatility_benchmark_metrics['strategy'].eq(_strategy)
        ].iloc[0]
        if _strategy == 'rho':
            _rho_difference = '--'
            _ci = '--'
            _p_value = '--'
        else:
            _inf = volatility_benchmark_inference[
                volatility_benchmark_inference['universe'].eq(_universe)
                & volatility_benchmark_inference['benchmark'].eq(_strategy)
            ].iloc[0]
            _rho_difference = f"{_inf['rho_minus_benchmark_sharpe']:+.4f}"
            _ci = f"[{_inf['ci_low']:+.4f}, {_inf['ci_high']:+.4f}]"
            _p_value = f"{_inf['p_value_unadjusted']:.3f}"
        _table_rows.append(
            f"{_universe_labels[_universe]} & {_strategy_labels[_strategy]} & "
            f"{_metric['sharpe']:.4f} & {_metric['delta_sharpe_vs_static']:+.4f} & "
            f"{_rho_difference} & {_ci} & {_p_value} "
            + r"\\"
        )
_vol_tex = '\n'.join([
    r'\begin{tabular}{llrrrcr}',
    r'\toprule',
    r'Universe & Strategy & $\mathrm{SR}_{\mathrm{net}}$ & '
    r'$\Delta\mathrm{SR}_{\mathrm{stat}}$ & '
    r'$\mathrm{SR}_{\rho}-\mathrm{SR}_{j}$ & 95\% CI & $p_{\mathrm{boot}}$ \\',
    r'\midrule',
    *_table_rows,
    r'\bottomrule',
    r'\end{tabular}',
]) + '\n'
VOL_TABLE = TAB / 'P_10_volatility_benchmark.tex'
_tmp = VOL_TABLE.with_suffix('.tex.tmp')
_tmp.write_text(_vol_tex)
_tmp.replace(VOL_TABLE)

print('=' * 84)
print('  Volatility-timing benchmarks — net primary, gross retained in audit')
print('  IV: formation-month mean VIX | RV36: exact universe/window realized volatility')
print('  Bootstrap intervals are exploratory and unadjusted for multiplicity.')
print('=' * 84)
display(
    volatility_benchmark_metrics[
        volatility_benchmark_metrics['basis'].eq('net')
    ].sort_values(['universe', 'strategy'])
)
display(volatility_benchmark_inference)
print(f'saved  tables/{VOL_TABLE.name}')
print(f'saved  audit/{VOL_SIGNAL_OUT.name}')
print(f'saved  audit/{VOL_MONTHLY_OUT.name}')
print(f'saved  audit/{VOL_METRICS_OUT.name}')
print(f'saved  audit/{VOL_INFERENCE_OUT.name}')


### 6.2.9.4 Temporal stability across subperiods  [FIG P_12]

The final robustness diagnostic asks whether the timing contribution is concentrated in one historical episode. The evaluation sample is partitioned into three non-overlapping intervals fixed independently of the realised portfolio results: 1995--2004, 2005--2014, and 2015--2025. For each interval, the static-versus-dynamic $\Delta\mathrm{SR}$ is computed separately for the big-cap and NYSE small--mid-cap universes. Figure P_12 reports the net-of-cost estimates; gross estimates are retained in the machine-readable audit.

The subperiods contain 120, 120, and 132 monthly holding returns, respectively. They are descriptive stability diagnostics rather than three new standalone significance claims. The same six observed cells enter the pre-declared 12-cell max-$T$ family in Section 6.2.8, so the graphical decomposition and the extended multiplicity correction share exactly the same dates, return basis, and portfolio paths. No portfolio is re-estimated specifically for a decade: only the realised return evaluation window changes.


In [ ]:
# FIG P_12 — temporal stability from the locked two-universe monthly audit; no solve here.
TEMPORAL_INPUT=DATA_OUT/'P_06_universe_monthly.parquet'; TEMPORAL_OUT=DATA_OUT/'P_09_temporal_stability.parquet'
assert TEMPORAL_INPUT.exists(),'run Section 6.2.8 through Figure P_08 first'
_temporal=pd.read_parquet(TEMPORAL_INPUT).copy(); _temporal['return_date']=pd.to_datetime(_temporal['return_date'])
assert set(_temporal.universe)=={'big_caps','small_caps_p20_p50'} and set(_temporal.strategy)=={'static','dynamic'}
assert _temporal.groupby(['universe','strategy']).size().eq(372).all()
TEMPORAL_PERIODS=[('1995--2004',pd.Timestamp('1995-01-01'),pd.Timestamp('2004-12-31'),120),
                  ('2005--2014',pd.Timestamp('2005-01-01'),pd.Timestamp('2014-12-31'),120),
                  ('2015--2025',pd.Timestamp('2015-01-01'),pd.Timestamp('2025-12-31'),132)]
def _temporal_sharpe(v):
    v=pd.Series(v).dropna(); assert len(v)>1 and v.std(ddof=1)>0; return float(v.mean()/v.std(ddof=1)*np.sqrt(12))
_rows=[]
for universe,frame in _temporal.groupby('universe',sort=False):
    for period,start,end,n in TEMPORAL_PERIODS:
        sub=frame[frame.return_date.between(start,end)]
        for basis,column in [('net','net_return'),('gross','gross_return')]:
            s=sub[sub.strategy.eq('static')].sort_values('return_date'); d=sub[sub.strategy.eq('dynamic')].sort_values('return_date')
            assert len(s)==len(d)==n and np.array_equal(s.return_date.to_numpy(),d.return_date.to_numpy())
            ss,ds=_temporal_sharpe(s[column]),_temporal_sharpe(d[column])
            _rows.append({'universe':universe,'period':period,'basis':basis,'date_start':start,'date_end':end,
                          'n_months':n,'static_sharpe':ss,'dynamic_sharpe':ds,'delta_sharpe':ds-ss})
temporal_stability=pd.DataFrame(_rows); assert len(temporal_stability)==12 and temporal_stability.delta_sharpe.notna().all()
_atomic_df(temporal_stability,TEMPORAL_OUT)
# Exact cross-contract guard against the six decade cells sent to the VPS.
_manifest_path=CACHE/'timing_test_manifest_88e42643b59e.json'
if _manifest_path.exists():
    _obs=json.loads(_manifest_path.read_text())['observed_cells_net']; _pc={'1995--2004':'d1','2005--2014':'d2','2015--2025':'d3'}; _uc={'big_caps':'L','small_caps_p20_p50':'S'}
    for r in temporal_stability[temporal_stability.basis.eq('net')].itertuples():
        assert np.isclose(
    float(r.delta_sharpe),
    float(_obs[f'{_uc[r.universe]}_{_pc[r.period]}']),
    rtol=0.0,
    atol=2e-5,
)
net_temporal=temporal_stability[temporal_stability.basis.eq('net')]; _periods=[p[0] for p in TEMPORAL_PERIODS]
_large=net_temporal[net_temporal.universe.eq('big_caps')].set_index('period').loc[_periods,'delta_sharpe'].to_numpy()
_small=net_temporal[net_temporal.universe.eq('small_caps_p20_p50')].set_index('period').loc[_periods,'delta_sharpe'].to_numpy()
fig,ax=plt.subplots(figsize=(8.4,4.2)); x=np.arange(3); width=.36
b1=ax.bar(x-width/2,_large,width,color=_NAVY,label='Big caps (top 100 above NYSE p90)')
b2=ax.bar(x+width/2,_small,width,color=_RUST,label='NYSE small–mid caps (p20–p50)'); ax.axhline(0,color=_GREY,lw=.8)
for bars in (b1,b2):
    for bar in bars:
        v=bar.get_height(); ax.annotate(f'{v:+.4f}',(bar.get_x()+bar.get_width()/2,v),xytext=(0,4 if v>=0 else -4),
            textcoords='offset points',ha='center',va='bottom' if v>=0 else 'top',fontsize=8.5,color=_GREY)
_v=np.r_[_large,_small]; _pad=max(.001,.15*np.max(np.abs(_v)))
ax.set_ylim(min(0,float(_v.min()))-(_pad if _v.min()<0 else 0),max(0,float(_v.max()))+_pad)
ax.set_xticks(x,[period.replace('--','–') for period in _periods]); ax.set_ylabel(r'Net $\Delta\mathrm{SR}$ (dynamic $-$ static)')
ax.set_title('Temporal stability of the timing gain across universes'); ax.legend(loc='best',fontsize=9.2)
fig.tight_layout(); save_fig(fig,12,'decades_universe'); plt.show()
print('saved  images/R_P_12_decades_universe.pdf'); print(f'saved  audit/{TEMPORAL_OUT.name}')
display(temporal_stability.sort_values(['basis','period','universe']))


# Appendix E Additional portfolio-robustness evidence

## E.1 Full metrics across construction choices  [TABLE P_07]

Table P_07 expands the compact robustness comparison in Section 6.2.9.2. For each one-at-a-time configuration it reports the dynamic portfolio's net performance and risk, the corresponding gross and net timing gains, implementation turnover, and the fraction of scheduled decisions at which the ambiguity radius reaches its safety cap. Repeated reference rows are retained deliberately so that each lever can be interpreted independently.


In [ ]:
# TABLE P_07 — full robustness metrics for landscape presentation.
assert "robust_metrics" in globals() and len(robust_metrics) == 12, (
    "run Section 6.2.9.2 first"
)

_rows = []
_previous_lever = None

for row in robust_metrics.itertuples(index=False):
    lever = str(row.lever)

    if _previous_lever is not None and lever != _previous_lever:
        _rows.append(r"\addlinespace[4pt]")

    displayed_lever = lever if lever != _previous_lever else ""

    _rows.append(
        f"{displayed_lever} & {row.value} "
        f"& {row.dynamic_sharpe_net:.3f} "
        f"& {100 * row.annual_return_net:.2f} "
        f"& {100 * row.volatility_net:.2f} "
        f"& {100 * row.MaxDD_net:.2f} "
        f"& {row.turnover:.4f} "
        f"& {100 * row.clip_fraction:.1f} "
        f"& {row.delta_sharpe_gross:+.4f} "
        f"& {row.delta_sharpe_net:+.4f} "
        + r"\\"
    )

    _previous_lever = lever

_tex_lines = [
    r"\begin{tabular}{@{}llrrrrrrrr@{}}",
    r"\toprule",
    (
        r" & "
        r" & \multicolumn{4}{c}{Dynamic portfolio, net}"
        r" & \multicolumn{2}{c}{Implementation}"
        r" & \multicolumn{2}{c}{Timing gain} \\"
    ),
    r"\cmidrule(lr){3-6}\cmidrule(lr){7-8}\cmidrule(lr){9-10}",
    (
        r"Lever & Value"
        r" & $\mathrm{SR}$"
        r" & Return (\%)"
        r" & Volatility (\%)"
        r" & MaxDD (\%)"
        r" & Turnover"
        r" & Clipped (\%)"
        r" & Gross $\Delta\mathrm{SR}$"
        r" & Net $\Delta\mathrm{SR}$ \\"
    ),
    r"\midrule",
    *_rows,
    r"\bottomrule",
    r"\end{tabular}",
]

_tex = "\n".join(_tex_lines) + "\n"

APPENDIX_E_TABLE = TAB / "P_07_robustness_metrics.tex"
_tmp = APPENDIX_E_TABLE.with_suffix(".tex.tmp")
_tmp.write_text(_tex, encoding="utf-8")
_tmp.replace(APPENDIX_E_TABLE)

# Clean notebook preview.
_preview = (
    robust_metrics[
        [
            "lever",
            "value",
            "dynamic_sharpe_net",
            "annual_return_net",
            "volatility_net",
            "MaxDD_net",
            "turnover",
            "clip_fraction",
            "delta_sharpe_gross",
            "delta_sharpe_net",
        ]
    ]
    .rename(
        columns={
            "lever": "Lever",
            "value": "Value",
            "dynamic_sharpe_net": "Net SR",
            "annual_return_net": "Net return (%)",
            "volatility_net": "Net volatility (%)",
            "MaxDD_net": "MaxDD (%)",
            "turnover": "Turnover",
            "clip_fraction": "Clipped (%)",
            "delta_sharpe_gross": "Gross ΔSR",
            "delta_sharpe_net": "Net ΔSR",
        }
    )
    .copy()
)

for column in ["Net return (%)", "Net volatility (%)", "MaxDD (%)", "Clipped (%)"]:
    _preview[column] *= 100

display(
    _preview.style
    .hide(axis="index")
    .format(
        {
            "Net SR": "{:.3f}",
            "Net return (%)": "{:.2f}",
            "Net volatility (%)": "{:.2f}",
            "MaxDD (%)": "{:.2f}",
            "Turnover": "{:.4f}",
            "Clipped (%)": "{:.1f}",
            "Gross ΔSR": "{:+.4f}",
            "Net ΔSR": "{:+.4f}",
        }
    )
)

print(f"saved  tables/{APPENDIX_E_TABLE.name}")
print("layout: one row per configuration, intended for a landscape page")

## E.2 Functional-form diagnostics  [FIGS P_13--P_14]

Figure P_13 displays the ambiguity-radius paths generated by the three point-in-time modulation maps over the complete evaluation sample. The static operational radius and the safety cap provide fixed references, while shaded bands identify the stress windows used elsewhere in the analysis. Figure P_14 then centers each point-in-time modulation series on the five dated stress peaks. For visual comparability only, each already-constructed PIT series is standardized over the displayed sample; this ex post scaling is never used by the portfolio engine.


In [ ]:
# FIGS P_13--P_14 — descriptive views of the exact PIT modulation forms used in P_11.
assert all(k in MODS for k in ['w36_id','w36_expz','w36_rank'])
_form_mod={'identity':MODS['w36_id'],'exp(z)':MODS['w36_expz'],'rank':MODS['w36_rank']}
_form_color={'identity':_NAVY,'exp(z)':_RUST,'rank':_GREEN}
_form_label={'identity':'identity','exp(z)':r'$\exp(z_{\tau})$','rank':'expanding rank'}
_form_dates=pd.DatetimeIndex([e['holding_month']+pd.offsets.MonthEnd(0) for e in realised_ledger])
_form_eps={k:pd.Series(np.clip(EPS_EK*v,EPS_FLOOR,EPS_CAP),index=_form_dates) for k,v in _form_mod.items()}
fig,ax=plt.subplots(figsize=(11,4.4))
for _,peak in CRISES:
    p=pd.Timestamp(peak)+pd.offsets.MonthEnd(0); ax.axvspan(p-pd.DateOffset(months=3),p+pd.DateOffset(months=3),color=_GREY,alpha=.10,lw=0)
for form in ['identity','exp(z)','rank']: ax.plot(_form_eps[form].index,_form_eps[form],color=_form_color[form],lw=1.3,label=_form_label[form])
ax.axhline(EPS_EK,color=_GREY,lw=.9,ls=':',label=rf'static $\varepsilon_0={EPS_EK:.2f}$')
ax.axhline(EPS_CAP,color='#555555',lw=.9,ls='--',alpha=.7,label=rf'safety cap $\varepsilon_{{\max}}={EPS_CAP:g}$')
ax.set_xlabel('Date'); ax.set_ylabel(r'Ambiguity radius $\varepsilon_{\tau}$')
ax.set_title('Point-in-time ambiguity radius by modulation form')
ax.legend(loc='upper right',bbox_to_anchor=(1.0,0.88),fontsize=8.5)
ax.margins(x=.01); fig.tight_layout(); save_fig(fig,13,'radius_forms_fullsample'); plt.show()

_zforms={k:pd.Series((v-np.mean(v))/np.std(v,ddof=1),index=_form_dates) for k,v in _form_mod.items()}
H=18; fig,axes=plt.subplots(2,3,figsize=(11,6))
for ax,(name,peak) in zip(axes.flat,CRISES):
    p=pd.Timestamp(peak)+pd.offsets.MonthEnd(0)
    for form in ['identity','exp(z)','rank']:
        s=_zforms[form].loc[p-pd.DateOffset(months=H):p+pd.DateOffset(months=H)]
        months=(s.index.year-p.year)*12+(s.index.month-p.month); ax.plot(months,s.values,color=_form_color[form],lw=1.4)
    ax.axhline(0,color=_GREY,lw=.6,ls=':'); ax.axvline(0,color=_RUST,lw=1,ls='--')
    ax.set_title(f'{name} ({p:%Y-%m})',fontsize=11); ax.set_xlim(-H,H); ax.set_xticks([-12,0,12]); ax.margins(y=.10)
for ax in axes[1,:]: ax.set_xlabel('Months from peak')
for ax in axes[:,0]: ax.set_ylabel(r'Standardized PIT modulation')
axes.flat[-1].axis('off')
from matplotlib.lines import Line2D
_handles=[Line2D([0],[0],color=_form_color[k],lw=1.6,label=_form_label[k]) for k in ['identity','exp(z)','rank']]
_handles.append(Line2D([0],[0],color=_RUST,lw=1,ls='--',label='stress peak'))
axes.flat[-1].legend(handles=_handles,loc='center',fontsize=10,frameon=False)
fig.suptitle('Point-in-time modulation forms around stress episodes',y=.99)
fig.tight_layout(); save_fig(fig,14,'crises_signal_forms'); plt.show()
print('saved  images/R_P_13_radius_forms_fullsample.pdf')
print('saved  images/R_P_14_crises_signal_forms.pdf')
